# FIT5196 Assessment 1 - Solution Notebook

**Group:** Group005  
**Members:** Replace with names and student IDs

## 0. Configuration and reproducibility

### Scope — Tasks 1, 2, 3 and 4

This notebook covers **specification Tasks 1, 2, 3 and 4**:

| Rubric criterion | Marks | Covered here |
|---|---|---|
| A1. Structured parsing and source profiling | 0.9 | sections `T1-P1` … `T1-P9` |
| A2. Source-to-target mapping | 0.6 | section `T1-M1` |
| B1–B3. Core relational transformation | 3.0 | sections `T2-R1` … `T2-O1` |
| C1–C2. Reconciliation/integrity evidence used here | 1.5 | `T2-R1` and validation register |
| E1–E2. Validation/reproducibility evidence | 2.0 | validation register (`VAL-*`) |

Tasks 5 and 6 are **not** implemented here. Section headings carry stable evidence IDs
(`T1-P1` … `T1-M1`) so the report and the source-to-target mapping cite them without copying
the evidence.

**Method.** Following the unit's applied sessions, the JSON export is read with `json` and
flattened with `pandas.json_normalize` (Week 3), the XML export is read with
`BeautifulSoup(..., "lxml-xml")` (Week 2), and every profiling question is then answered with
**pandas** on those DataFrames.

**Where regular expressions are and are not used.** The specification draws a hard line:
*"Do not parse either structured document as plain text with regular expressions"* (Task 1),
while regex is the assessed tool for *"bounded patterns such as tags, markers, URLs,
whitespace and business-reference extraction"* applied **after** a structured parser has
returned the value (Task 3).

This notebook keeps that line visible:

| Stage | Tool | Regex? |
|---|---|---|
| Reading the JSON document | `json.load` | no |
| Reading the XML document | `pandas.read_xml` (lxml), `BeautifulSoup("lxml-xml")` for the tree walk | no |
| Flattening to DataFrames | `pandas.json_normalize`, `pandas.read_xml` | no |
| Grain, keys, duplicates, overlap, foreign keys | pandas | no |
| Currency / whitespace / digit-shape handling | `str.split`, `str.replace(regex=False)`, `str.translate` | **no** |
| `T1-P8` narrative markers, tags and business references | `re` / `pandas.str`, verbose `(?x)` | yes — as Task 3 requires |

`T1-P8` is the **only** section that uses a regular expression, and it operates on values the
structured parsers already returned. A check at the end of the notebook verifies this.

No canonical row count, expected value or duplicate-id list is hard-coded anywhere.

### 0.1 Environment and dependencies

---
#### Imports

In [1]:
import glob
import hashlib
import html
import json
import os
import re
import sys
import unicodedata
from collections import Counter

import pandas as pd
from bs4 import BeautifulSoup as bsoup
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

print("python :", sys.version.split()[0])
print("pandas :", pd.__version__)

python : 3.11.5
pandas : 2.0.3


---
#### Configuration cell

All paths are relative to the notebook directory and can be overridden with environment
variables, so the workflow runs from a fresh kernel with **Restart and Run All** without
editing code.

In [2]:
GROUP_ID = "Group005"
INPUT_DIR = os.environ.get("FIT5196_INPUT_DIR", "raw_input")
OUTPUT_DIR = os.environ.get("FIT5196_OUTPUT_DIR", "outputs")
REFERENCE_DIR = os.environ.get("FIT5196_REFERENCE_DIR", "reference")
TEMPLATE_DIR = os.environ.get("FIT5196_TEMPLATE_DIR", "templates")
PROFILE_DIR = os.environ.get("FIT5196_PROFILE_DIR", "profiling")

JSON_PATH = os.path.join(INPUT_DIR, f"{GROUP_ID}_commerce.json")
XML_PATH = os.path.join(INPUT_DIR, f"{GROUP_ID}_operations.xml")
DICTIONARY_PATH = os.path.join(REFERENCE_DIR, "public_data_dictionary.csv")
MANIFEST_PATH = os.path.join(REFERENCE_DIR, "A1_manifest.json")
MAPPING_PATH = f"{GROUP_ID}_source_to_target_mapping.csv"

for directory in (OUTPUT_DIR, PROFILE_DIR):
    os.makedirs(directory, exist_ok=True)

MISSING = "NaN"           # published literal missing-string sentinel
MONEY_TOLERANCE = 0.01    # published comparison tolerance for currency fields
NUMERIC_TOLERANCE = 0.01  # comparison tolerance for non-currency numeric fields

def save_profile(frame, name):
    '''Write a Task 1 profiling artefact and echo where it landed.'''
    path = os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_{name}.csv")
    frame.to_csv(path, index=False)
    print(f"[saved] {path}  ({len(frame)} rows)")
    return path

for label, path in [("json", JSON_PATH), ("xml", XML_PATH), ("dictionary", DICTIONARY_PATH)]:
    print(f"{label:11s} {path}  exists={os.path.exists(path)}")

json        raw_input/Group005_commerce.json  exists=True
xml         raw_input/Group005_operations.xml  exists=True
dictionary  reference/public_data_dictionary.csv  exists=True


## 1. Parse and profile the two sources

### 1.1 JSON structure and profile

---
#### `T1-P1` — Structured parsing and file integrity

The JSON export is read with the standard-library `json` parser and the XML export with
`BeautifulSoup` using the `lxml-xml` parser, exactly as in the Week 2 and Week 3 applied
sessions. No regular expression touches document structure.

File integrity is checked against the published `A1_manifest.json` SHA-256 digests, so a
truncated or substituted input is detected before any conclusion is drawn.

In [3]:
def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

with open(MANIFEST_PATH) as fh:
    manifest = json.load(fh)
expected = {os.path.basename(entry["path"]): entry for entry in manifest["files"]}

integrity = pd.DataFrame([
    {
        "file": os.path.basename(path),
        "bytes_observed": os.path.getsize(path),
        "bytes_manifest": expected.get(os.path.basename(path), {}).get("bytes"),
        "sha256_matches_manifest": sha256_of(path) == expected.get(os.path.basename(path), {}).get("sha256"),
    }
    for path in (JSON_PATH, XML_PATH)
])
print("manifest group alias:", manifest.get("group_alias"), "| notebook GROUP_ID:", GROUP_ID)
integrity

manifest group alias: Group005 | notebook GROUP_ID: Group005


,file,bytes_observed,bytes_manifest,sha256_matches_manifest
0,Group005_commerce.json,14139534,14139534,True
1,Group005_operations.xml,17811170,17811170,True


In [4]:
# --- JSON: json.load, then pandas (Week 3 applied session) ------------------
with open(JSON_PATH, encoding="utf-8") as fh:
    commerce = json.load(fh)

print("JSON top level  :", type(commerce).__name__)
for key, value in commerce.items():
    print(f"   {key:20s} {type(value).__name__:6s} {len(value) if hasattr(value, '__len__') else ''}")
print("JSON exportMetadata :", commerce["exportMetadata"])

JSON top level  : dict
   customerProfiles     list   500
   exportMetadata       dict   3
   orders               list   2818
   productReviews       list   3946
JSON exportMetadata : {'groupAlias': 'Group005', 'period': 2018, 'sourceSystem': 'CommercePlatform'}


### 1.2 XML structure and profile

In [5]:
# --- XML: BeautifulSoup with the lxml-xml parser (Week 2 applied session) ---
with open(XML_PATH, encoding="utf-8") as fh:
    operations = bsoup(fh, "lxml-xml")

root = operations.find("OperationsExport")
print("XML root        :", root.name, dict(root.attrs))
for child in root.find_all(recursive=False):
    print(f"   {child.name:22s} {len(child.find_all(recursive=False)):5d} direct children")
print("XML Export_Metadata :",
      {c.name: c.text for c in root.find("Export_Metadata").find_all(recursive=False)})

XML root        : OperationsExport {'groupAlias': 'Group005', 'sourceSystem': 'OperationsERP', 'period': '2018'}
   Export_Metadata            2 direct children
   Orders                  2818 direct children
   ProductCatalogue        1000 direct children
   ProductReviews          3946 direct children
   WarehouseDirectory         3 direct children
XML Export_Metadata : {'System': 'OperationsERP', 'Contract_Version': 'A1-DATA-2.0'}


### 1.3 Source comparison and assumptions

**Observation.** Both exports declare the same group alias (`Group005`) and the same reporting
period (2018) but a different source system (`CommercePlatform` vs `OperationsERP`). The two
systems describe overlapping business events with different nesting, element names and value
conventions — which is what the rest of Task 1 quantifies.

---
#### `T1-P2` — Flattening to DataFrames, and the structural profile

Every source collection is flattened into a **pandas DataFrame**: JSON with
`pandas.json_normalize` (including `record_path` for the nested cart array), XML with
`pandas.read_xml` and an XPath per collection. Both are real structured parsers — `read_xml`
runs `lxml` underneath — so no document is scraped as text.

The eleven frames below are the single tabular representation used from `T1-P3` onwards, and
the hand-off point into Task 2. Each one is displayed so the actual parsed values are visible,
not just their shape.

Source typing is deliberately **not** coerced here: JSON frames keep native JSON types
(`bool`/`int`/`float`/`str`), XML frames are entirely `object`/`str` because XML has no value
types. That difference is itself the finding profiled in `T1-P4`.

In [6]:
# --- JSON -> DataFrame with pandas.json_normalize --------------------------
orders_flat = pd.json_normalize(commerce["orders"])          # header.* and delivery.* columns

json_headers = (orders_flat.filter(like="header.")
                .rename(columns=lambda c: c.split(".", 1)[1]))
json_deliveries = (orders_flat.filter(like="delivery.")
                   .rename(columns=lambda c: c.split(".", 1)[1]))
json_items = pd.json_normalize(commerce["orders"], record_path="shoppingCart")
json_customers = pd.json_normalize(commerce["customerProfiles"])
json_reviews = pd.json_normalize(commerce["productReviews"])

for name, frame in [("json_customers", json_customers), ("json_headers", json_headers),
                    ("json_items", json_items), ("json_deliveries", json_deliveries),
                    ("json_reviews", json_reviews)]:
    print(f"  {name:16s} {frame.shape}")

  json_customers   (500, 20)
  json_headers     (2818, 22)
  json_items       (8947, 6)
  json_deliveries  (2818, 20)
  json_reviews     (3946, 15)


In [7]:
# Parse XML with pandas/lxml; dtype=str preserves source formats and leading zeros.
XML_COLLECTIONS = {
    "xml_headers":    ".//Order/Header",
    "xml_items":      ".//Order/Shopping_Cart/Item",
    "xml_deliveries": ".//Order/Delivery",
    "xml_products":   ".//ProductCatalogue/Product",
    "xml_reviews":    ".//ProductReviews/Review",
    "xml_warehouses": ".//WarehouseDirectory/Warehouse",
}

xml_frames = {name: pd.read_xml(XML_PATH, xpath=xpath, dtype=str)
              for name, xpath in XML_COLLECTIONS.items()}

xml_headers = xml_frames["xml_headers"]
xml_items = xml_frames["xml_items"]
xml_deliveries = xml_frames["xml_deliveries"]
xml_products = xml_frames["xml_products"]
xml_reviews = xml_frames["xml_reviews"]
xml_warehouses = xml_frames["xml_warehouses"]

for name, frame in xml_frames.items():
    print(f"  {name:16s} {frame.shape}")

  xml_headers      (2818, 22)
  xml_items        (8803, 6)
  xml_deliveries   (2818, 20)
  xml_products     (1000, 21)
  xml_reviews      (3946, 15)
  xml_warehouses   (3, 3)


In [8]:
# Show the actual parsed data, not just its shape. Every downstream conclusion in
# this notebook is drawn from these eleven frames, so they are displayed here.
RAW_FRAMES = {
    ("JSON", "$.customerProfiles[*]"):                              json_customers,
    ("JSON", "$.orders[*].header"):                                 json_headers,
    ("JSON", "$.orders[*].shoppingCart[*]"):                        json_items,
    ("JSON", "$.orders[*].delivery"):                               json_deliveries,
    ("JSON", "$.productReviews[*]"):                                json_reviews,
    ("XML",  "/OperationsExport/Orders/Order/Header"):              xml_headers,
    ("XML",  "/OperationsExport/Orders/Order/Shopping_Cart/Item"):  xml_items,
    ("XML",  "/OperationsExport/Orders/Order/Delivery"):            xml_deliveries,
    ("XML",  "/OperationsExport/ProductCatalogue/Product"):         xml_products,
    ("XML",  "/OperationsExport/ProductReviews/Review"):            xml_reviews,
    ("XML",  "/OperationsExport/WarehouseDirectory/Warehouse"):     xml_warehouses,
}

for (source, path), frame in RAW_FRAMES.items():
    print(f"\n{'=' * 100}\n{source}  {path}      {frame.shape[0]:,} rows x {frame.shape[1]} columns\n{'=' * 100}")
    print(frame.head())


JSON  $.customerProfiles[*]      500 rows x 20 columns
  accountStatus acquisitionSource ageBand contactFrequencyPreference customerID customerSegment  emailDomain homeCountry homePostcode homeState homeSuburb householdSizeBand  \
0        Active             Store   25-34                  Quarterly   CUS00001           Value  example.net   Australia         3182       VIC   St Kilda                5+   
1        Active       Paid Search     55+             Essential only   CUS00002  Small Business    mail.test   Australia         3070       VIC  Northcote                5+   
2        Active            Social   35-44                    Monthly   CUS00003           Value  example.net   Australia         3070       VIC  Northcote                5+   
3        Review          Referral     55+                  Quarterly   CUS00004  Small Business  example.net   Australia         3121       VIC   Richmond               3-4   
4        Active          Referral   25-34                     We

In [9]:
raw_frame_profile = pd.DataFrame([
    {
        "source": source,
        "structural_path": path,
        "rows": len(frame),
        "columns": frame.shape[1],
        "dtypes": ", ".join(f"{kind}:{count}" for kind, count
                            in frame.dtypes.astype(str).value_counts().items()),
        "cells_missing": int(frame.isna().sum().sum()),
    }
    for (source, path), frame in RAW_FRAMES.items()
])
save_profile(raw_frame_profile, "raw_frame_profile")
raw_frame_profile

[saved] profiling/Group005_T1_raw_frame_profile.csv  (11 rows)


,source,structural_path,rows,columns,dtypes,cells_missing
0,JSON,$.customerProfiles[*],500,20,"object:17, float64:1, bool:1, int64:1",0
1,JSON,$.orders[*].header,2818,22,"object:14, float64:6, int64:1, bool:1",0
2,JSON,$.orders[*].shoppingCart[*],8947,6,"object:3, float64:2, int64:1",0
3,JSON,$.orders[*].delivery,2818,20,"object:11, int64:4, float64:3, bool:2",0
4,JSON,$.productReviews[*],3946,15,"object:12, int64:2, bool:1",0
5,XML,/OperationsExport/Orders/Order/Header,2818,22,object:22,1792
6,XML,/OperationsExport/Orders/Order/Shopping_Cart/Item,8803,6,object:6,0
7,XML,/OperationsExport/Orders/Order/Delivery,2818,20,object:20,0
8,XML,/OperationsExport/ProductCatalogue/Product,1000,21,object:21,0
9,XML,/OperationsExport/ProductReviews/Review,3946,15,object:15,0


In [10]:
# --- structural path profile (on the parse tree, where nesting still exists) ---
def profile_json_paths(node, path="$", acc=None):
    '''Recursively enumerate JSON structural paths with counts and node kind.'''
    acc = {"count": Counter(), "kind": {}, "types": {}} if acc is None else acc
    acc["count"][path] += 1
    acc["types"].setdefault(path, Counter())
    if isinstance(node, dict):
        acc["kind"][path] = "container"
        acc["types"][path]["object"] += 1
        for key, value in node.items():
            profile_json_paths(value, f"{path}.{key}", acc)
    elif isinstance(node, list):
        acc["kind"][path] = "container"
        acc["types"][path]["array"] += 1
        for element in node:
            profile_json_paths(element, f"{path}[*]", acc)
    else:
        acc["kind"].setdefault(path, "leaf")
        acc["types"][path][type(node).__name__] += 1
    return acc

json_acc = profile_json_paths(commerce)
json_paths = pd.DataFrame([
    {"source": "JSON", "structural_path": path, "occurrences": count,
     "node_kind": json_acc["kind"][path],
     "node_types": ", ".join(f"{k}:{v}" for k, v in json_acc["types"][path].most_common())}
    for path, count in json_acc["count"].items()
])

def profile_xml_paths(element, prefix="", acc=None):
    acc = {"count": Counter(), "kind": {}} if acc is None else acc
    path = f"{prefix}/{element.name}"
    acc["count"][path] += 1
    children = element.find_all(recursive=False)
    acc["kind"][path] = "container" if children else "leaf"
    for child in children:
        profile_xml_paths(child, path, acc)
    return acc

xml_acc = profile_xml_paths(root)
xml_paths = pd.DataFrame([
    {"source": "XML", "structural_path": path, "occurrences": count,
     "node_kind": xml_acc["kind"][path],
     "node_types": "element container" if xml_acc["kind"][path] == "container" else "text leaf"}
    for path, count in xml_acc["count"].items()
])

structure_profile = (pd.concat([json_paths, xml_paths], ignore_index=True)
                     .sort_values(["source", "structural_path"], ignore_index=True))
save_profile(structure_profile, "source_structure_profile")
print(f"JSON distinct paths: {len(json_paths)} | XML distinct paths: {len(xml_paths)}")

containers = structure_profile.query("node_kind == 'container'").reset_index(drop=True)
print(f"containers: JSON {(containers['source'] == 'JSON').sum()}, "
      f"XML {(containers['source'] == 'XML').sum()}")
containers

[saved] profiling/Group005_T1_source_structure_profile.csv  (201 rows)
JSON distinct paths: 98 | XML distinct paths: 103
containers: JSON 12, XML 14


,source,structural_path,occurrences,node_kind,node_types
0,JSON,$,1,container,object:1
1,JSON,$.customerProfiles,1,container,array:1
2,JSON,$.customerProfiles[*],500,container,object:500
3,JSON,$.exportMetadata,1,container,object:1
4,JSON,$.orders,1,container,array:1
5,JSON,$.orders[*],2818,container,object:2818
6,JSON,$.orders[*].delivery,2818,container,object:2818
7,JSON,$.orders[*].header,2818,container,object:2818
8,JSON,$.orders[*].shoppingCart,2818,container,array:2818
9,JSON,$.orders[*].shoppingCart[*],8947,container,object:8947


**Material structural findings**

* **JSON** nests three collections under one object: `customerProfiles[*]`, `orders[*]` (each
  with a `header` object, a `delivery` object and a **repeated `shoppingCart[*]` array**) and
  `productReviews[*]`. The order grain is therefore *not* the record grain for cart lines —
  the array is flattened with `json_normalize(..., record_path="shoppingCart")` to reach the
  `order_items` grain.
* **XML** nests four collections under `OperationsExport`: `Orders/Order` (with `Header`, a
  **repeated `Shopping_Cart/Item`** element and `Delivery`), `ProductCatalogue/Product`,
  `ProductReviews/Review` and `WarehouseDirectory/Warehouse`.
* **Entity coverage differs by source.** Customers exist only in JSON; the product catalogue
  exists only in XML. Orders, order items, deliveries and reviews exist in both. This drives
  every `source_format` value in the mapping.
* **`WarehouseDirectory` and the two metadata blocks map to no target table.** They are read
  and used as cross-checks but carry no mapping row.

---
#### `T1-P3` — Grain, candidate primary keys and candidate foreign keys

Each collection is tested against its candidate business key with pandas: row count, distinct
key count, and therefore whether the key is unique *within that source*.

In [11]:
COLLECTIONS = [
    # (source, structural path, frame, candidate key, target table, stated grain)
    ("JSON", "$.orders[*].header",                                json_headers,    "orderID",      "orders",          "one row per canonical order"),
    ("XML",  "/OperationsExport/Orders/Order/Header",             xml_headers,     "Order_ID",     "orders",          "one row per canonical order"),
    ("JSON", "$.orders[*].shoppingCart[*]",                       json_items,      "orderItemID",  "order_items",     "one row per order item"),
    ("XML",  "/OperationsExport/Orders/Order/Shopping_Cart/Item", xml_items,       "Order_Item_ID","order_items",     "one row per order item"),
    ("JSON", "$.customerProfiles[*]",                             json_customers,  "customerID",   "customers",       "one row per customer"),
    ("JSON", "$.orders[*].delivery",                              json_deliveries, "deliveryID",   "deliveries",      "one row per completed order delivery"),
    ("XML",  "/OperationsExport/Orders/Order/Delivery",           xml_deliveries,  "Delivery_ID",  "deliveries",      "one row per completed order delivery"),
    ("XML",  "/OperationsExport/ProductCatalogue/Product",        xml_products,    "Product_ID",   "products",        "one row per product"),
    ("JSON", "$.productReviews[*]",                               json_reviews,    "reviewID",     "product_reviews", "one row per canonical product review"),
    ("XML",  "/OperationsExport/ProductReviews/Review",           xml_reviews,     "Review_ID",    "product_reviews", "one row per canonical product review"),
]

grain_profile = pd.DataFrame([
    {
        "source": source,
        "structural_path": path,
        "target_table": table,
        "target_grain": grain,
        "records": len(frame),
        "distinct_keys": frame[key].nunique(dropna=False),
        "repeated_key_rows": len(frame) - frame[key].nunique(dropna=False),
        "key_unique_within_source": bool(frame[key].is_unique),
        "key_null_or_blank": int(frame[key].isna().sum() + (frame[key] == "").sum()),
    }
    for source, path, frame, key, table, grain in COLLECTIONS
])
save_profile(grain_profile, "source_grain_and_keys")
grain_profile

[saved] profiling/Group005_T1_source_grain_and_keys.csv  (10 rows)


,source,structural_path,target_table,target_grain,records,distinct_keys,repeated_key_rows,key_unique_within_source,key_null_or_blank
0,JSON,$.orders[*].header,orders,one row per canonical order,2818,2750,68,False,0
1,XML,/OperationsExport/Orders/Order/Header,orders,one row per canonical order,2818,2750,68,False,0
2,JSON,$.orders[*].shoppingCart[*],order_items,one row per order item,8947,8723,224,False,0
3,XML,/OperationsExport/Orders/Order/Shopping_Cart/Item,order_items,one row per order item,8803,8618,185,False,0
4,JSON,$.customerProfiles[*],customers,one row per customer,500,500,0,True,0
5,JSON,$.orders[*].delivery,deliveries,one row per completed order delivery,2818,2750,68,False,0
6,XML,/OperationsExport/Orders/Order/Delivery,deliveries,one row per completed order delivery,2818,2750,68,False,0
7,XML,/OperationsExport/ProductCatalogue/Product,products,one row per product,1000,1000,0,True,0
8,JSON,$.productReviews[*],product_reviews,one row per canonical product review,3946,3850,96,False,0
9,XML,/OperationsExport/ProductReviews/Review,product_reviews,one row per canonical product review,3946,3850,96,False,0


In [12]:
# Candidate foreign keys observed in the raw sources (before any reconciliation).
fk_candidates = pd.DataFrame([
    ("orders.customer_id",            "customers.customer_id",     "JSON header.customerID | XML Header/Customer_ID"),
    ("order_items.order_id",          "orders.order_id",           "item element repeats the parent order id in both sources"),
    ("order_items.product_id",        "products.product_id",       "XML ProductCatalogue is the only product source"),
    ("deliveries.order_id",           "orders.order_id",           "delivery block repeats the parent order id in both sources"),
    ("product_reviews.order_id",      "orders.order_id",           "review carries the order id directly"),
    ("product_reviews.order_item_id", "order_items.order_item_id", "review is tied to a single purchased line"),
    ("product_reviews.product_id",    "products.product_id",       "review carries the product id directly"),
    ("product_reviews.customer_id",   "customers.customer_id",     "review carries the customer id directly"),
], columns=["child_field", "parent_field", "source_evidence"])
fk_candidates

,child_field,parent_field,source_evidence
0,orders.customer_id,customers.customer_id,JSON header.customerID | XML Header/Customer_ID
1,order_items.order_id,orders.order_id,item element repeats the parent order id in both sources
2,order_items.product_id,products.product_id,XML ProductCatalogue is the only product source
3,deliveries.order_id,orders.order_id,delivery block repeats the parent order id in both sources
4,product_reviews.order_id,orders.order_id,review carries the order id directly
5,product_reviews.order_item_id,order_items.order_item_id,review is tied to a single purchased line
6,product_reviews.product_id,products.product_id,review carries the product id directly
7,product_reviews.customer_id,customers.customer_id,review carries the customer id directly


**Material grain findings**

* Every candidate key is fully populated — no null or blank identifier anywhere.
* **None of the four shared transactional collections is unique on its own key.** Orders,
  order items, deliveries and reviews each repeat a subset of their own records verbatim in
  *both* exports, so within-source duplication must be resolved before the cross-source
  overlap is even considered (`T1-P6`). The two single-source collections are the exception:
  `customerProfiles` (500/500) and `ProductCatalogue` (1,000/1,000) are already unique.
* `shoppingCart` / `Shopping_Cart/Item` is the only one-to-many child of an order. Flattening
  it produces the `order_items` grain; the parent order grain is untouched, so order-level
  money is never multiplied by the number of lines.
* Every order in **both** exports has `Order_Status = Completed` and exactly one delivery
  block, so `deliveries` is 1:1 with `orders` in this package rather than a strict subset.
  The grain is satisfied without filtering here, so Task 1 profiles the collection unfiltered.
  The explicit completed-order filter is applied later in this notebook, in `T2-B3`, which builds
  `completed_order_ids` from `order_status == "Completed"` and restricts the canonical deliveries
  to it; `VAL-GRAIN-01` then checks that result against the submitted table.

---
#### `T1-P4` — Value-format conventions

The two systems publish the same business values in different notations. Each convention is
probed with pandas over the **flattened DataFrame columns**, so the label and the evidence
cannot drift apart: every field named in `target_fields_probed` is actually probed.

In [13]:
# Digit masking uses str.translate, not a regular expression: nothing in the parsing
# or normalisation path needs regex, and keeping it out makes the boundary unambiguous.
DIGIT_MASK = str.maketrans("0123456789", "9999999999")

def pattern_counts(values, top=4):
    '''Digit-masked shape summary of a Series, e.g. 'AUD 9,999.99'.'''
    shapes = Counter()
    for value in values:
        if value is None or value == "" or (isinstance(value, float) and pd.isna(value)):
            shapes["<missing>"] += 1   # one bucket for all three absence encodings
        else:
            shapes[str(value).translate(DIGIT_MASK)[:28]] += 1
    return " | ".join(f"{shape} ({count})" for shape, count in shapes.most_common(top))

def columns_of(*specs):
    '''Stack one or more DataFrame columns into a single Series for profiling.'''
    return pd.concat([frame[column].astype(object) for frame, column in specs],
                     ignore_index=True)

conventions = [
    ("date",
     "customers.signup_date | deliveries.dispatch_date | deliveries.promised_date | "
     "deliveries.delivered_date | products.launch_date",
     columns_of((json_customers, "signupDate"), (json_deliveries, "dispatchDate"),
                (json_deliveries, "promisedDate"), (json_deliveries, "deliveredDate")),
     columns_of((xml_deliveries, "Dispatch_Date"), (xml_deliveries, "Promised_Date"),
                (xml_deliveries, "Delivered_Date"), (xml_products, "Launch_Date")),
     "YYYY-MM-DD"),
    ("timestamp",
     "orders.order_timestamp | product_reviews.review_timestamp",
     columns_of((json_headers, "orderTimestamp"), (json_reviews, "reviewTimestamp")),
     columns_of((xml_headers, "Order_Timestamp"), (xml_reviews, "Review_Timestamp")),
     "YYYY-MM-DD HH:MM:SS"),
    ("boolean",
     "orders.expedited_delivery | deliveries.on_time_in_full | deliveries.signature_required | "
     "customers.marketing_consent (JSON only) | products.recyclable_packaging, products.active_flag "
     "(XML only) | product_reviews.verified_purchase",
     columns_of((json_headers, "expeditedDelivery"), (json_deliveries, "onTimeInFull"),
                (json_deliveries, "signatureRequired"), (json_customers, "marketingConsent"),
                (json_reviews, "verifiedPurchase")),
     columns_of((xml_headers, "Expedited_Delivery"), (xml_deliveries, "On_Time_In_Full"),
                (xml_deliveries, "Signature_Required"), (xml_products, "Recyclable_Packaging"),
                (xml_products, "Active_Flag"), (xml_reviews, "Verified_Purchase")),
     "True / False"),
    ("currency",
     "orders.order_price | orders.delivery_charges | orders.tax_amount | orders.order_total | "
     "order_items.unit_price | order_items.line_revenue | deliveries.delivery_cost | "
     "products.unit_price, products.unit_cost (XML only)",
     columns_of((json_headers, "orderPrice"), (json_headers, "deliveryCharges"),
                (json_headers, "taxAmount"), (json_headers, "orderTotal"),
                (json_items, "unitPrice"), (json_items, "lineRevenue"),
                (json_deliveries, "deliveryCost")),
     columns_of((xml_headers, "Order_Price"), (xml_headers, "Delivery_Charges"),
                (xml_headers, "Tax_Amount"), (xml_headers, "Order_Total"),
                (xml_items, "Unit_Price"), (xml_items, "Line_Revenue"),
                (xml_deliveries, "Delivery_Cost"), (xml_products, "Unit_Price"),
                (xml_products, "Unit_Cost")),
     "float, no label, no thousands separator"),
    ("percentage", "orders.coupon_discount",
     columns_of((json_headers, "couponDiscount")),
     columns_of((xml_headers, "Coupon_Discount")),
     "numeric percentage points (15 not 0.15)"),
    ("missing string", "orders.coupon_code",
     columns_of((json_headers, "couponCode")),
     columns_of((xml_headers, "Coupon_Code")),
     f"literal {MISSING!r}"),
    ("identifier",
     "orders.order_id | order_items.order_item_id | deliveries.delivery_id | "
     "product_reviews.review_id | customers.customer_id (JSON only) | products.product_id (XML only)",
     columns_of((json_headers, "orderID"), (json_items, "orderItemID"),
                (json_deliveries, "deliveryID"), (json_reviews, "reviewID"),
                (json_customers, "customerID")),
     columns_of((xml_headers, "Order_ID"), (xml_items, "Order_Item_ID"),
                (xml_deliveries, "Delivery_ID"), (xml_reviews, "Review_ID"),
                (xml_products, "Product_ID")),
     "verbatim string, leading zeros and case preserved"),
    ("geo / measure",
     "orders.customer_lat | orders.customer_long | deliveries.shipping_distance_km | "
     "deliveries.estimated_carbon_kg | products.weight_kg (XML only)",
     columns_of((json_headers, "customerLat"), (json_headers, "customerLong"),
                (json_deliveries, "shippingDistanceKm"), (json_deliveries, "estimatedCarbonKg")),
     columns_of((xml_headers, "Customer_Lat"), (xml_headers, "Customer_Long"),
                (xml_deliveries, "Shipping_Distance_Km"),
                (xml_deliveries, "Estimated_Carbon_Kg"), (xml_products, "Weight_Kg")),
     "float, published precision preserved"),
]

format_profile = pd.DataFrame([
    {"convention": name, "target_fields_probed": fields,
     "json_values_probed": len(js), "json_representation": pattern_counts(js),
     "xml_values_probed": len(xs), "xml_representation": pattern_counts(xs),
     "target_rule": rule}
    for name, fields, js, xs, rule in conventions
])
save_profile(format_profile, "format_conventions")
format_profile

[saved] profiling/Group005_T1_format_conventions.csv  (8 rows)


,convention,target_fields_probed,json_values_probed,json_representation,xml_values_probed,xml_representation,target_rule
0,date,customers.signup_date | deliveries.dispatch_date | deliveries.promised_date | deliveri...,8954,9999-99-99 (8954),9454,99/99/9999 (9454),YYYY-MM-DD
1,timestamp,orders.order_timestamp | product_reviews.review_timestamp,6764,9999-99-99 99:99:99 (6764),6764,99/99/9999 99:99:99 (6764),YYYY-MM-DD HH:MM:SS
2,boolean,orders.expedited_delivery | deliveries.on_time_in_full | deliveries.signature_required...,12900,True (9233) | False (3667),14400,Y (10538) | N (3862),True / False
3,currency,orders.order_price | orders.delivery_charges | orders.tax_amount | orders.order_total ...,31984,999.99 (12837) | 9999.99 (8961) | 99.99 (5105) | 9.99 (1776),33696,"AUD 999.99 (15336) | AUD 9,999.99 (10691) | AUD 99.99 (5718) | AUD 9.99 (1865)","float, no label, no thousands separator"
4,percentage,orders.coupon_discount,2818,99 (1682) | 9 (1136),2818,99% (1649) | 9% (1169),numeric percentage points (15 not 0.15)
5,missing string,orders.coupon_code,2818,<missing> (1741) | B9SAVE-99 (1077),2818,<missing> (1792) | B9SAVE-99 (1026),literal 'NaN'
6,identifier,orders.order_id | order_items.order_item_id | deliveries.delivery_id | product_reviews...,19029,HITM9999999 (8947) | HREV999999 (3946) | HORD999999 (2818) | HDEL999999 (2818),19385,HITM9999999 (8803) | HREV999999 (3946) | HORD999999 (2818) | HDEL999999 (2818),"verbatim string, leading zeros and case preserved"
7,geo / measure,orders.customer_lat | orders.customer_long | deliveries.shipping_distance_km | deliver...,11272,9.999 (2822) | 999.999999 (2527) | -99.999999 (2509) | 9.9999 (2497),12272,9.999 (3746) | 999.999999 (2516) | 9.9999 (2491) | -99.999999 (2482),"float, published precision preserved"


In [14]:
# Day-first vs month-first is not assumed: the XML day component is checked with pandas .str
xml_date_text = pd.concat([xml_deliveries["Dispatch_Date"], xml_products["Launch_Date"]],
                          ignore_index=True)
print(f"XML date first component max  = {xml_date_text.str.slice(0, 2).astype(int).max()}  -> day  (exceeds 12)")
print(f"XML date second component max = {xml_date_text.str.slice(3, 5).astype(int).max()} -> month")

# Preserve the observed source difference: JSON uses an empty string, while an
# empty XML <Coupon_Code/> is parsed as a missing value.
print(f"\nJSON coupon codes absent (empty string \"\")   : {int((json_headers['couponCode'] == '').sum())}")
print(f"JSON coupon codes read as NaN                : {int(json_headers['couponCode'].isna().sum())}")
print(f"XML  coupon codes absent (<Coupon_Code/>)    : {int(xml_headers['Coupon_Code'].isna().sum())}")
print(f"XML  coupon codes that are an empty string   : {int((xml_headers['Coupon_Code'] == '').sum())}")

XML date first component max  = 31  -> day  (exceeds 12)
XML date second component max = 12 -> month

JSON coupon codes absent (empty string "")   : 1741
JSON coupon codes read as NaN                : 0
XML  coupon codes absent (<Coupon_Code/>)    : 1792
XML  coupon codes that are an empty string   : 0


**Material format findings**

| Convention | JSON export | XML export | Target rule |
|---|---|---|---|
| Date | already `YYYY-MM-DD` | `DD/MM/YYYY` | `YYYY-MM-DD` |
| Timestamp | already `YYYY-MM-DD HH:MM:SS` | `DD/MM/YYYY HH:MM:SS` | `YYYY-MM-DD HH:MM:SS` |
| Boolean | native JSON `true`/`false` | `Y` / `N` | `True` / `False` |
| Money | native float | `"AUD 1,234.56"` | float, label and separators removed |
| Percentage | native integer `15` | `"15%"` | numeric percentage **points** |
| Missing string | `""` (empty string) | empty element `<Coupon_Code/>` | literal `NaN` |

* Day-first is **confirmed, not assumed**: the XML first date component reaches 31.
* `coupon_discount` is percentage points; treating `15%` as `0.15` would break `order_total`
  on every discounted order.
* `deliveries.delay_reason` publishes the literal token `none`. That is a **category**, not a
  missing value, and must not be converted to the `NaN` sentinel.

##### Allowed categorical values

The permitted value set of every categorical target field is **measured**, not asserted. The
mapping renders its "Observed domain {...}" phrases from this table at build time, so a domain
claim cannot drift from the data. This is also the evidence a Task 4 check on allowed
categorical values will assert against.

In [15]:
def distinct_values(json_spec, xml_spec):
    '''Distinct non-missing values of a target field across whichever sources carry it.'''
    values = set()
    for spec in (json_spec, xml_spec):
        if spec is None:
            continue
        frame, column = spec
        series = frame[column].dropna().astype(str)
        values |= set(series[series != ""])
    return values

def ordered_domain(values):
    '''Numeric order where the labels are numeric, alphabetical otherwise.'''
    try:
        return [str(v) for v in sorted(values, key=float)]
    except ValueError:
        return sorted(values)

CATEGORICAL_TARGETS = [
    ("orders", "sales_channel",       (json_headers, "salesChannel"),     (xml_headers, "Sales_Channel")),
    ("orders", "payment_method",      (json_headers, "paymentMethod"),    (xml_headers, "Payment_Method")),
    ("orders", "currency",            (json_headers, "currency"),         (xml_headers, "Currency")),
    ("orders", "nearest_warehouse",   (json_headers, "nearestWarehouse"), (xml_headers, "Nearest_Warehouse")),
    ("orders", "order_status",        (json_headers, "orderStatus"),      (xml_headers, "Order_Status")),
    ("orders", "season",              (json_headers, "season"),           (xml_headers, "Season")),
    ("orders", "device_type",         (json_headers, "deviceType"),       (xml_headers, "Device_Type")),
    ("orders", "referral_source",     (json_headers, "referralSource"),   (xml_headers, "Referral_Source")),
    ("order_items", "quantity",       (json_items, "quantity"),           (xml_items, "Quantity")),
    ("customers", "loyalty_tier",     (json_customers, "loyaltyTier"),           None),
    ("customers", "customer_segment", (json_customers, "customerSegment"),       None),
    ("customers", "age_band",         (json_customers, "ageBand"),               None),
    ("customers", "preferred_channel",(json_customers, "preferredChannel"),      None),
    ("customers", "home_state",       (json_customers, "homeState"),             None),
    ("customers", "home_country",     (json_customers, "homeCountry"),           None),
    ("customers", "acquisition_source", (json_customers, "acquisitionSource"),   None),
    ("customers", "account_status",   (json_customers, "accountStatus"),         None),
    ("customers", "preferred_device", (json_customers, "preferredDevice"),       None),
    ("customers", "email_domain",     (json_customers, "emailDomain"),           None),
    ("customers", "household_size_band", (json_customers, "householdSizeBand"),  None),
    ("customers", "contact_frequency_preference", (json_customers, "contactFrequencyPreference"), None),
    ("deliveries", "carrier",         (json_deliveries, "carrier"),       (xml_deliveries, "Carrier")),
    ("deliveries", "service_level",   (json_deliveries, "serviceLevel"),  (xml_deliveries, "Service_Level")),
    ("deliveries", "delivery_status", (json_deliveries, "deliveryStatus"),(xml_deliveries, "Delivery_Status")),
    ("deliveries", "delay_reason",    (json_deliveries, "delayReason"),   (xml_deliveries, "Delay_Reason")),
    ("deliveries", "promised_days",   (json_deliveries, "promisedDays"),  (xml_deliveries, "Promised_Days")),
    ("deliveries", "delivery_window", (json_deliveries, "deliveryWindow"),(xml_deliveries, "Delivery_Window")),
    ("deliveries", "delivery_note_clean", (json_deliveries, "deliveryNoteClean"), (xml_deliveries, "Delivery_Note_Clean")),
    ("products", "brand",             None, (xml_products, "Brand")),
    ("products", "category",          None, (xml_products, "Category")),
    ("products", "subcategory",       None, (xml_products, "Subcategory")),
    ("products", "colour",            None, (xml_products, "Colour")),
    ("products", "supplier_country",  None, (xml_products, "Supplier_Country")),
    ("products", "warranty_months",   None, (xml_products, "Warranty_Months")),
    ("products", "tax_category",      None, (xml_products, "Tax_Category")),
    ("products", "package_type",      None, (xml_products, "Package_Type")),
    ("product_reviews", "language_code",       (json_reviews, "languageCode"),       (xml_reviews, "Language_Code")),
    ("product_reviews", "rating",              (json_reviews, "rating"),             (xml_reviews, "Rating")),
    ("product_reviews", "delivery_experience", (json_reviews, "deliveryExperience"), (xml_reviews, "Delivery_Experience")),
    ("product_reviews", "value_experience",    (json_reviews, "valueExperience"),    (xml_reviews, "Value_Experience")),
    ("product_reviews", "writing_style",       (json_reviews, "writingStyle"),       (xml_reviews, "Writing_Style")),
]

CATEGORICAL_DOMAINS = {(table, field): ordered_domain(distinct_values(js, xs))
                       for table, field, js, xs in CATEGORICAL_TARGETS}

def DOM(table, field):
    '''Render a measured domain phrase for the source-to-target mapping.'''
    values = CATEGORICAL_DOMAINS[(table, field)]
    if len(values) == 1:
        return f" Single observed value {values[0]}."
    return f" Observed domain {{{', '.join(values)}}}."

categorical_profile = pd.DataFrame([
    {"target_table": table, "target_field": field,
     "distinct_values": len(CATEGORICAL_DOMAINS[(table, field)]),
     "observed_domain": ", ".join(CATEGORICAL_DOMAINS[(table, field)])}
    for table, field, _, _ in CATEGORICAL_TARGETS
])
save_profile(categorical_profile, "categorical_domain_profile")
categorical_profile

[saved] profiling/Group005_T1_categorical_domain_profile.csv  (41 rows)


,target_table,target_field,distinct_values,observed_domain
0,orders,sales_channel,3,"Mobile, Store, Web"
1,orders,payment_method,4,"Bank Transfer, Card, Gift Card, PayPal"
2,orders,currency,1,AUD
3,orders,nearest_warehouse,3,"Bakers, Nickolson, Thompson"
4,orders,order_status,1,Completed
5,orders,season,4,"Autumn, Spring, Summer, Winter"
6,orders,device_type,3,"Desktop, Mobile, Tablet"
7,orders,referral_source,5,"Organic, Paid Search, Referral, Social, Store"
8,order_items,quantity,3,"1, 2, 3"
9,customers,loyalty_tier,4,"Bronze, Gold, Platinum, Silver"


##### The published order arithmetic, and how the source rounds

All six published steps are checked on the raw parsed values, before any transformation.

One result deserves its own note. `Series.round(2)` (numpy) and Python's built-in `round`
**disagree** on values landing exactly on a half-cent, because numpy scales by 100 and rounds
the binary result while Python uses a correctly-rounded decimal algorithm. The published
`order_total` figures follow Python's `round`, so that is the rounding used here and in Task 2
(`ASM-14`).

In [16]:
def money_series(series):
    '''Currency column -> float. Handles native floats and 'AUD 1,234.56' text.

    No regex: the currency label is split off as a whitespace token and the
    thousands separators are removed with a literal replace.
    '''
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return (series.str.split().str[-1]                       # drop the 'AUD' label
                  .str.replace(",", "", regex=False)          # drop thousands separators
                  .astype(float))

def percent_series(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return series.str.rstrip("%").astype(float)

def round2(series):
    '''Round to cents the way the source did: Python's round, not numpy's.'''
    return series.map(lambda value: round(float(value), 2))

# Evidence for ASM-14: how far the two rounding rules disagree on this package.
_price = money_series(json_headers["orderPrice"])
_disc = percent_series(json_headers["couponDiscount"])
_charges = money_series(json_headers["deliveryCharges"])
_published = money_series(json_headers["orderTotal"])
_raw = _price * (1 - _disc / 100) + _charges
rounding_evidence = pd.DataFrame([
    {"rounding_rule": "pandas/numpy Series.round(2)",
     "orders_matching_published_total": int((_raw.round(2).sub(_published).abs() <= MONEY_TOLERANCE).sum())},
    {"rounding_rule": "Python round(value, 2)",
     "orders_matching_published_total": int((round2(_raw).sub(_published).abs() <= MONEY_TOLERANCE).sum())},
]).assign(orders_probed=len(json_headers))
rounding_evidence

,rounding_rule,orders_matching_published_total,orders_probed
0,pandas/numpy Series.round(2),2794,2818
1,"Python round(value, 2)",2818,2818


In [17]:
# Validate steps 1-2 after de-duplication; the final column shows the inflation
# caused by retaining repeated source records.
def check_steps_1_and_2(label, items, headers, item_cols, header_cols):
    item_key, order_col, qty_col, price_col, line_col = item_cols
    order_key, order_price_col = header_cols

    lines = pd.DataFrame({
        "order_item_id": items[item_key],
        "order_id": items[order_col],
        "derived": round2(items[qty_col].astype(float) * money_series(items[price_col])),
        "published": round2(money_series(items[line_col])),
    })
    distinct_lines = lines.drop_duplicates(subset="order_item_id")
    step1_ok = int((distinct_lines["derived"].sub(distinct_lines["published"]).abs()
                    <= MONEY_TOLERANCE).sum())

    published_price = (pd.DataFrame({"order_id": headers[order_key],
                                     "order_price": round2(money_series(headers[order_price_col]))})
                       .drop_duplicates(subset="order_id").set_index("order_id")["order_price"])
    dedup = round2(distinct_lines.groupby("order_id")["derived"].sum())
    with_repeats = round2(lines.groupby("order_id")["derived"].sum())

    return {
        "source": label,
        "distinct_items": len(distinct_lines),
        "step 1: line_revenue == round(qty*unit_price, 2)": f"{step1_ok}/{len(distinct_lines)}",
        "distinct_orders": len(published_price),
        "step 2: order_price == sum(line_revenue), de-duplicated":
            f"{int((dedup.reindex(published_price.index).sub(published_price).abs() <= MONEY_TOLERANCE).sum())}/{len(published_price)}",
        "step 2 if within-source repeats were kept":
            f"{int((with_repeats.reindex(published_price.index).sub(published_price).abs() <= MONEY_TOLERANCE).sum())}/{len(published_price)}",
    }

steps_12 = pd.DataFrame([
    check_steps_1_and_2("JSON", json_items, json_headers,
                        ("orderItemID", "orderID", "quantity", "unitPrice", "lineRevenue"),
                        ("orderID", "orderPrice")),
    check_steps_1_and_2("XML", xml_items, xml_headers,
                        ("Order_Item_ID", "Order_ID", "Quantity", "Unit_Price", "Line_Revenue"),
                        ("Order_ID", "Order_Price")),
])

# Steps 3 to 6, on the order header rows as published.
arithmetic_rows = []
for label, headers, cols in [
    ("JSON", json_headers, ("orderPrice", "couponDiscount", "deliveryCharges", "taxAmount", "orderTotal")),
    ("XML", xml_headers, ("Order_Price", "Coupon_Discount", "Delivery_Charges", "Tax_Amount", "Order_Total")),
]:
    price = money_series(headers[cols[0]])
    discount = percent_series(headers[cols[1]])
    charges = money_series(headers[cols[2]])
    tax = round2(money_series(headers[cols[3]]))
    total = round2(money_series(headers[cols[4]]))
    arithmetic_rows.append({
        "source": label,
        "order_rows_probed": len(headers),
        "step 3: tax == round(order_price/11, 2)":
            f"{int((round2(price / 11).sub(tax).abs() <= MONEY_TOLERANCE).sum())}/{len(headers)}",
        "steps 4-6: total == round(price*(1-disc/100)+charges, 2)":
            f"{int((round2(price * (1 - discount / 100) + charges).sub(total).abs() <= MONEY_TOLERANCE).sum())}/{len(headers)}",
    })
arithmetic_steps = pd.DataFrame(arithmetic_rows)
save_profile(steps_12.merge(arithmetic_steps, on="source"), "order_arithmetic_checks")
print(steps_12)
arithmetic_steps

[saved] profiling/Group005_T1_order_arithmetic_checks.csv  (2 rows)
  source  distinct_items step 1: line_revenue == round(qty*unit_price, 2)  distinct_orders step 2: order_price == sum(line_revenue), de-duplicated step 2 if within-source repeats were kept
0   JSON            8723                                        8723/8723             2750                                               2750/2750                                 2682/2750
1    XML            8618                                        8618/8618             2750                                               2750/2750                                 2682/2750


,source,order_rows_probed,"step 3: tax == round(order_price/11, 2)","steps 4-6: total == round(price*(1-disc/100)+charges, 2)"
0,JSON,2818,2818/2818,2818/2818
1,XML,2818,2818/2818,2818/2818


---
#### `T1-P5` — Field coverage: which target fields come from one source or both

In [18]:
dictionary = pd.read_csv(DICTIONARY_PATH, keep_default_na=False)
print(f"required target fields: {len(dictionary)}")
dictionary.groupby(["output_table", "grain"], sort=False).size().rename("fields").reset_index()

required target fields: 111


,output_table,grain,fields
0,orders,one row per order,23
1,order_items,one row per order item,6
2,customers,one row per customer,20
3,deliveries,one row per completed order,20
4,products,one row per product,21
5,product_reviews,one row per canonical review,21


In [19]:
# Source elements that are read but deliberately carry no target field.
unmapped = pd.DataFrame([
    ("JSON", "$.exportMetadata", "group alias, period, source system",
     "Used to confirm the package identity in T1-P1; not a business entity in the dictionary."),
    ("XML", "/OperationsExport/Export_Metadata", "system name, contract version",
     "Used to confirm the package identity in T1-P1; not a business entity in the dictionary."),
    ("XML", "/OperationsExport/WarehouseDirectory/Warehouse", "warehouse name, latitude, longitude",
     "No warehouse table is required. Used only to cross-check the orders.nearest_warehouse vocabulary."),
], columns=["source", "structural_path", "content", "treatment"])

warehouse_names = set(xml_warehouses["Name"])
order_warehouses = set(json_headers["nearestWarehouse"]) | set(xml_headers["Nearest_Warehouse"])
print("WarehouseDirectory names      :", sorted(warehouse_names))
print("nearest_warehouse values used :", sorted(order_warehouses))
print("vocabulary matches            :", warehouse_names == order_warehouses)
unmapped

WarehouseDirectory names      : ['Bakers', 'Nickolson', 'Thompson']
nearest_warehouse values used : ['Bakers', 'Nickolson', 'Thompson']
vocabulary matches            : True


,source,structural_path,content,treatment
0,JSON,$.exportMetadata,"group alias, period, source system",Used to confirm the package identity in T1-P1; not a business entity in the dictionary.
1,XML,/OperationsExport/Export_Metadata,"system name, contract version",Used to confirm the package identity in T1-P1; not a business entity in the dictionary.
2,XML,/OperationsExport/WarehouseDirectory/Warehouse,"warehouse name, latitude, longitude",No warehouse table is required. Used only to cross-check the orders.nearest_warehouse ...


---
#### `T1-P6` — Duplicate records within a source and overlapping records across sources

Overlap cannot be judged on raw values, because the same order is written as `AUD 5,287.78` /
`14/10/2018 09:28:00` / `Y` in one system and `5287.78` / `2018-10-14 09:28:00` / `true` in
the other. Comparable columns are therefore **normalised first**, then records are compared
field-by-field on the table primary key.

The normalisers below are profiling-scope; Task 2 reuses the same conventions.

In [20]:
# --- vectorised, source-aware normalisers ----------------------------------
def is_absent(value):
    '''One definition of "missing" for all three source encodings.

    JSON writes an empty string, read_xml renders an empty element as NaN, and a
    genuinely absent key is None. Collapsing them here is what makes ASM-04 true and
    stops the same absence being read as a cross-source conflict.
    '''
    return value is None or value == "" or (isinstance(value, float) and pd.isna(value))

def norm_id(series):
    return series.astype(object).map(lambda v: None if is_absent(v) else str(v).strip())

def norm_text(series):
    def clean(value):
        if is_absent(value):
            return None
        # str.split() collapses any run of whitespace and trims - no regex needed here.
        return " ".join(html.unescape(unicodedata.normalize("NFC", str(value))).split())
    return series.map(clean)

def norm_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    return series.map({"Y": True, "N": False})

def norm_int(series):
    return series.astype(float).astype(int)

def norm_float(series, places=6):
    return series.astype(float).round(places)

def norm_date(series, source):
    fmt = "%Y-%m-%d" if source == "JSON" else "%d/%m/%Y"
    return pd.to_datetime(series, format=fmt).dt.strftime("%Y-%m-%d")

def norm_datetime(series, source):
    fmt = "%Y-%m-%d %H:%M:%S" if source == "JSON" else "%d/%m/%Y %H:%M:%S"
    return pd.to_datetime(series, format=fmt).dt.strftime("%Y-%m-%d %H:%M:%S")

# Column name per source, so one normalisation spec serves both exports.
COLUMN_MAP = {
    "orders": {
        "JSON": dict(order_id="orderID", source_system_record_id="sourceSystemRecordID",
                     customer_id="customerID", order_timestamp="orderTimestamp",
                     sales_channel="salesChannel", payment_method="paymentMethod", currency="currency",
                     nearest_warehouse="nearestWarehouse", order_status="orderStatus",
                     order_price="orderPrice", delivery_charges="deliveryCharges",
                     coupon_code="couponCode", coupon_discount="couponDiscount",
                     tax_amount="taxAmount", order_total="orderTotal", season="season",
                     expedited_delivery="expeditedDelivery", customer_lat="customerLat",
                     customer_long="customerLong", device_type="deviceType",
                     referral_source="referralSource", customer_note="customerNote"),
        "XML": dict(order_id="Order_ID", source_system_record_id="Source_System_Record_ID",
                    customer_id="Customer_ID", order_timestamp="Order_Timestamp",
                    sales_channel="Sales_Channel", payment_method="Payment_Method", currency="Currency",
                    nearest_warehouse="Nearest_Warehouse", order_status="Order_Status",
                    order_price="Order_Price", delivery_charges="Delivery_Charges",
                    coupon_code="Coupon_Code", coupon_discount="Coupon_Discount",
                    tax_amount="Tax_Amount", order_total="Order_Total", season="Season",
                    expedited_delivery="Expedited_Delivery", customer_lat="Customer_Lat",
                    customer_long="Customer_Long", device_type="Device_Type",
                    referral_source="Referral_Source", customer_note="Customer_Note"),
    },
    "order_items": {
        "JSON": dict(order_item_id="orderItemID", order_id="orderID", product_id="productID",
                     quantity="quantity", unit_price="unitPrice", line_revenue="lineRevenue"),
        "XML": dict(order_item_id="Order_Item_ID", order_id="Order_ID", product_id="Product_ID",
                    quantity="Quantity", unit_price="Unit_Price", line_revenue="Line_Revenue"),
    },
    "deliveries": {
        "JSON": dict(delivery_id="deliveryID", order_id="orderID", dispatch_date="dispatchDate",
                     promised_date="promisedDate", delivered_date="deliveredDate", carrier="carrier",
                     service_level="serviceLevel", delivery_status="deliveryStatus",
                     delay_days="delayDays", on_time_in_full="onTimeInFull",
                     fulfilment_hours="fulfilmentHours", delivery_cost="deliveryCost",
                     delay_reason="delayReason", promised_days="promisedDays",
                     tracking_event_count="trackingEventCount", delivery_window="deliveryWindow",
                     shipping_distance_km="shippingDistanceKm", signature_required="signatureRequired",
                     estimated_carbon_kg="estimatedCarbonKg", delivery_note_clean="deliveryNoteClean"),
        "XML": dict(delivery_id="Delivery_ID", order_id="Order_ID", dispatch_date="Dispatch_Date",
                    promised_date="Promised_Date", delivered_date="Delivered_Date", carrier="Carrier",
                    service_level="Service_Level", delivery_status="Delivery_Status",
                    delay_days="Delay_Days", on_time_in_full="On_Time_In_Full",
                    fulfilment_hours="Fulfilment_Hours", delivery_cost="Delivery_Cost",
                    delay_reason="Delay_Reason", promised_days="Promised_Days",
                    tracking_event_count="Tracking_Event_Count", delivery_window="Delivery_Window",
                    shipping_distance_km="Shipping_Distance_Km", signature_required="Signature_Required",
                    estimated_carbon_kg="Estimated_Carbon_Kg", delivery_note_clean="Delivery_Note_Clean"),
    },
    "product_reviews": {
        "JSON": dict(review_id="reviewID", order_id="orderID", order_item_id="orderItemID",
                     product_id="productID", customer_id="customerID",
                     review_timestamp="reviewTimestamp", language_code="languageCode", rating="rating",
                     review_title="reviewTitle", verified_purchase="verifiedPurchase",
                     helpful_votes="helpfulVotes", delivery_experience="deliveryExperience",
                     value_experience="valueExperience", writing_style="writingStyle",
                     review_text="reviewText"),
        "XML": dict(review_id="Review_ID", order_id="Order_ID", order_item_id="Order_Item_ID",
                    product_id="Product_ID", customer_id="Customer_ID",
                    review_timestamp="Review_Timestamp", language_code="Language_Code", rating="Rating",
                    review_title="Review_Title", verified_purchase="Verified_Purchase",
                    helpful_votes="Helpful_Votes", delivery_experience="Delivery_Experience",
                    value_experience="Value_Experience", writing_style="Writing_Style",
                    review_text="Review_Text"),
    },
}

# How each normalised column is produced, independent of source spelling.
FIELD_RULES = {
    "orders": {
        "order_id": norm_id, "source_system_record_id": norm_id, "customer_id": norm_id,
        "order_timestamp": "datetime", "sales_channel": norm_id, "payment_method": norm_id,
        "currency": norm_id, "nearest_warehouse": norm_id, "order_status": norm_id,
        "order_price": "money", "delivery_charges": "money", "coupon_code": norm_id,
        "coupon_discount": "percent", "tax_amount": "money", "order_total": "money",
        "season": norm_id, "expedited_delivery": norm_bool, "customer_lat": "float6",
        "customer_long": "float6", "device_type": norm_id, "referral_source": norm_id,
        "customer_note": norm_text,
    },
    "order_items": {
        "order_item_id": norm_id, "order_id": norm_id, "product_id": norm_id,
        "quantity": norm_int, "unit_price": "money", "line_revenue": "money",
    },
    "deliveries": {
        "delivery_id": norm_id, "order_id": norm_id, "dispatch_date": "date",
        "promised_date": "date", "delivered_date": "date", "carrier": norm_id,
        "service_level": norm_id, "delivery_status": norm_id, "delay_days": norm_int,
        "on_time_in_full": norm_bool, "fulfilment_hours": norm_int, "delivery_cost": "money",
        "delay_reason": norm_id, "promised_days": norm_int, "tracking_event_count": norm_int,
        "delivery_window": norm_id, "shipping_distance_km": "float4",
        "signature_required": norm_bool, "estimated_carbon_kg": "float3",
        "delivery_note_clean": norm_text,
    },
    "product_reviews": {
        "review_id": norm_id, "order_id": norm_id, "order_item_id": norm_id, "product_id": norm_id,
        "customer_id": norm_id, "review_timestamp": "datetime", "language_code": norm_id,
        "rating": norm_int, "review_title": norm_text, "verified_purchase": norm_bool,
        "helpful_votes": norm_int, "delivery_experience": norm_id, "value_experience": norm_id,
        "writing_style": norm_id, "review_text": norm_text,
    },
}

def normalise(frame, table, source):
    '''Return a DataFrame with target column names and comparable, source-neutral values.'''
    columns, rules = COLUMN_MAP[table][source], FIELD_RULES[table]
    out = {}
    for target, rule in rules.items():
        series = frame[columns[target]]
        if rule == "money":       out[target] = round2(money_series(series))
        elif rule == "percent":   out[target] = percent_series(series)
        elif rule == "date":      out[target] = norm_date(series, source)
        elif rule == "datetime":  out[target] = norm_datetime(series, source)
        elif rule == "float6":    out[target] = norm_float(series, 6)
        elif rule == "float4":    out[target] = norm_float(series, 4)
        elif rule == "float3":    out[target] = norm_float(series, 3)
        else:                     out[target] = rule(series)
    return pd.DataFrame(out)

NORMALISED = {
    ("orders", "JSON"): normalise(json_headers, "orders", "JSON"),
    ("orders", "XML"): normalise(xml_headers, "orders", "XML"),
    ("order_items", "JSON"): normalise(json_items, "order_items", "JSON"),
    ("order_items", "XML"): normalise(xml_items, "order_items", "XML"),
    ("deliveries", "JSON"): normalise(json_deliveries, "deliveries", "JSON"),
    ("deliveries", "XML"): normalise(xml_deliveries, "deliveries", "XML"),
    ("product_reviews", "JSON"): normalise(json_reviews, "product_reviews", "JSON"),
    ("product_reviews", "XML"): normalise(xml_reviews, "product_reviews", "XML"),
    # Single-source tables need no cross-source comparison, only their own key check.
    ("customers", "JSON"): json_customers.rename(columns={"customerID": "customer_id"}),
    ("products", "XML"): xml_products.rename(columns={"Product_ID": "product_id"}),
}

PRIMARY_KEYS = {"orders": "order_id", "order_items": "order_item_id", "deliveries": "delivery_id",
                "product_reviews": "review_id", "customers": "customer_id", "products": "product_id"}

for (table, source), frame in NORMALISED.items():
    print(f"  {table:16s} {source:4s} {frame.shape[0]:6d} rows x {frame.shape[1]:2d} normalised columns")

  orders           JSON   2818 rows x 22 normalised columns
  orders           XML    2818 rows x 22 normalised columns
  order_items      JSON   8947 rows x  6 normalised columns
  order_items      XML    8803 rows x  6 normalised columns
  deliveries       JSON   2818 rows x 20 normalised columns
  deliveries       XML    2818 rows x 20 normalised columns
  product_reviews  JSON   3946 rows x 15 normalised columns
  product_reviews  XML    3946 rows x 15 normalised columns
  customers        JSON    500 rows x 20 normalised columns
  products         XML    1000 rows x 21 normalised columns


In [21]:
conflict_log = []   # every field-level disagreement is recorded, never silently resolved

def within_source_repeats(frame, key, scope):
    '''Split repeated key rows into identical repeats and genuine conflicts.'''
    if frame.empty:
        return 0, 0
    comparable = frame.astype(str)                      # NaN-safe row comparison
    repeat_rows = len(frame) - frame[key].nunique()
    identical_rows = len(frame) - len(comparable.drop_duplicates())
    conflicting = repeat_rows - identical_rows
    if conflicting:
        duplicated_keys = frame[key][frame[key].duplicated(keep=False)]
        for k, group in comparable[frame[key].isin(duplicated_keys)].groupby(key):
            first = group.iloc[0]
            for _, other in group.iloc[1:].iterrows():
                differing = [c for c in group.columns if first[c] != other[c]]
                if differing:
                    conflict_log.append({"scope": scope, "key_name": key, "key": k,
                                         "fields": ", ".join(differing)})
    return repeat_rows, conflicting

def cross_source_conflicts(json_frame, xml_frame, key, scope):
    '''Compare the two exports field-by-field on their shared keys.'''
    if json_frame.empty or xml_frame.empty:
        return 0, 0
    left = json_frame.drop_duplicates(subset=key).set_index(key).astype(str)
    right = xml_frame.drop_duplicates(subset=key).set_index(key).astype(str)
    shared = left.index.intersection(right.index)
    if shared.empty:
        return 0, 0
    left, right = left.loc[shared], right.loc[shared, left.columns]
    differs = left.ne(right)                            # vectorised cell-wise comparison
    rows_with_conflict = differs.any(axis=1)
    for k in differs.index[rows_with_conflict]:
        conflict_log.append({"scope": scope, "key_name": key, "key": k,
                             "fields": ", ".join(differs.columns[differs.loc[k]])})
    return len(shared), int(rows_with_conflict.sum())

overlap_rows = []
for table, key in PRIMARY_KEYS.items():
    json_frame = NORMALISED.get((table, "JSON"), pd.DataFrame())
    xml_frame = NORMALISED.get((table, "XML"), pd.DataFrame())

    json_repeats, json_conflicting = within_source_repeats(json_frame, key, f"{table}: within JSON")
    xml_repeats, xml_conflicting = within_source_repeats(xml_frame, key, f"{table}: within XML")
    shared, cross_conflicting = cross_source_conflicts(json_frame, xml_frame, key,
                                                       f"{table}: JSON vs XML")
    json_keys = set(json_frame[key]) if not json_frame.empty else set()
    xml_keys = set(xml_frame[key]) if not xml_frame.empty else set()

    overlap_rows.append({
        "target_table": table, "primary_key": key,
        "json_records": len(json_frame), "json_distinct_keys": len(json_keys),
        "xml_records": len(xml_frame), "xml_distinct_keys": len(xml_keys),
        "within_json_repeat_pairs": json_repeats, "within_json_conflicting": json_conflicting,
        "within_xml_repeat_pairs": xml_repeats, "within_xml_conflicting": xml_conflicting,
        "keys_in_both_sources": shared, "cross_source_conflicting": cross_conflicting,
        "canonical_keys_expected": len(json_keys | xml_keys),
    })

overlap_profile = pd.DataFrame(overlap_rows)
save_profile(overlap_profile, "duplicate_and_overlap_profile")
overlap_profile

[saved] profiling/Group005_T1_duplicate_and_overlap_profile.csv  (6 rows)


,target_table,primary_key,json_records,json_distinct_keys,xml_records,xml_distinct_keys,within_json_repeat_pairs,within_json_conflicting,within_xml_repeat_pairs,within_xml_conflicting,keys_in_both_sources,cross_source_conflicting,canonical_keys_expected
0,orders,order_id,2818,2750,2818,2750,68,0,68,0,500,0,5000
1,order_items,order_item_id,8947,8723,8803,8618,224,0,185,0,1602,0,15739
2,deliveries,delivery_id,2818,2750,2818,2750,68,0,68,0,500,0,5000
3,product_reviews,review_id,3946,3850,3946,3850,96,0,96,0,700,0,7000
4,customers,customer_id,500,500,0,0,0,0,0,0,0,0,500
5,products,product_id,0,0,1000,1000,0,0,0,0,0,0,1000


In [22]:
conflicts = pd.DataFrame(conflict_log, columns=["scope", "key_name", "key", "fields"])
save_profile(conflicts, "field_level_conflicts")
print(f"field-level conflicts recorded: {len(conflicts)}")
print(conflicts.head(20) if len(conflicts) else "No field-level conflict detected after normalisation.")

[saved] profiling/Group005_T1_field_level_conflicts.csv  (0 rows)
field-level conflicts recorded: 0
No field-level conflict detected after normalisation.


**Material reconciliation findings**

* **Both duplication types are present and must be handled in that order.** Each export
  repeats a subset of its own records verbatim (within-source duplication), *and* the two
  exports independently publish a shared subset of orders, items, deliveries and reviews
  (cross-source overlap). Concatenating first and de-duplicating once on the primary key
  handles both without double counting.
* **After the published normalisation, every repeat agrees on every compared field** — zero
  field-level conflicts within a source and zero across sources. A correctly normalised
  duplicate therefore needs no arbitrary JSON-over-XML precedence rule, and **none is applied**.
* The conflict log is written to `profiling/Group005_T1_field_level_conflicts.csv`. It is
  produced by the same code path that would record a conflict, so the empty file is evidence
  rather than an absence of checking.
* `customers` and `products` are single-source and already unique on their keys, so they need
  no reconciliation at all.

---
#### `T1-P7` — Source-level referential integrity

Foreign keys are checked against the **union** of both sources, because a child row in one
export may legitimately reference a parent that only the other export publishes.

In [23]:
def canonical_keys(table):
    keys = set()
    for source in ("JSON", "XML"):
        frame = NORMALISED.get((table, source))
        if frame is not None and not frame.empty:
            keys |= set(frame[PRIMARY_KEYS[table]])
    return keys

def referenced_values(table, column):
    values = set()
    for source in ("JSON", "XML"):
        frame = NORMALISED.get((table, source))
        if frame is not None and not frame.empty:
            values |= set(frame[column].dropna())
    return values

CANONICAL = {table: canonical_keys(table) for table in PRIMARY_KEYS}

fk_checks = [
    ("orders.customer_id",            "customers.customer_id",     ("orders", "customer_id"),            "customers"),
    ("order_items.order_id",          "orders.order_id",           ("order_items", "order_id"),          "orders"),
    ("order_items.product_id",        "products.product_id",       ("order_items", "product_id"),        "products"),
    ("deliveries.order_id",           "orders.order_id",           ("deliveries", "order_id"),           "orders"),
    ("product_reviews.order_id",      "orders.order_id",           ("product_reviews", "order_id"),      "orders"),
    ("product_reviews.order_item_id", "order_items.order_item_id", ("product_reviews", "order_item_id"), "order_items"),
    ("product_reviews.product_id",    "products.product_id",       ("product_reviews", "product_id"),    "products"),
    ("product_reviews.customer_id",   "customers.customer_id",     ("product_reviews", "customer_id"),   "customers"),
]

fk_profile = pd.DataFrame([
    {
        "child_field": child, "parent_field": parent,
        "distinct_values_referenced": len(child_values),
        "distinct_parent_keys": len(parent_keys),
        "orphans_against_union": len(child_values - parent_keys),
        "parent_coverage_pct": round(100 * len(child_values & parent_keys) / len(parent_keys), 2),
        "status": "PASS" if not (child_values - parent_keys) else "FAIL",
    }
    for child, parent, (child_table, child_column), parent_table in fk_checks
    for child_values in [referenced_values(child_table, child_column)]
    for parent_keys in [CANONICAL[parent_table]]
])
save_profile(fk_profile, "source_referential_integrity")
fk_profile

[saved] profiling/Group005_T1_source_referential_integrity.csv  (8 rows)


,child_field,parent_field,distinct_values_referenced,distinct_parent_keys,orphans_against_union,parent_coverage_pct,status
0,orders.customer_id,customers.customer_id,500,500,0,100.00,PASS
1,order_items.order_id,orders.order_id,5000,5000,0,100.00,PASS
2,order_items.product_id,products.product_id,1000,1000,0,100.00,PASS
3,deliveries.order_id,orders.order_id,5000,5000,0,100.00,PASS
4,product_reviews.order_id,orders.order_id,4017,5000,0,80.34,PASS
5,product_reviews.order_item_id,order_items.order_item_id,7000,15739,0,44.48,PASS
6,product_reviews.product_id,products.product_id,1000,1000,0,100.00,PASS
7,product_reviews.customer_id,customers.customer_id,500,500,0,100.00,PASS


**Material integrity findings**

* All eight required relationships resolve against the union of the two exports, with **zero
  orphans**. Neither source is self-sufficient — the XML order lines reference products the
  JSON export never mentions, and both exports' orders reference customers only the JSON
  export publishes — so reconciliation must complete *before* integrity can be asserted.
* Every customer and every catalogue product is actually referenced by transactional data.

---
#### `T1-P9` — Assumptions register

Assumptions that had to be settled before transformation can begin. Each is stated with the
evidence that supports it and the consequence if it is wrong.

In [24]:
assumptions = pd.DataFrame([
    ("ASM-01", "XML dates are day-first DD/MM/YYYY.",
     "T1-P4: the first component reaches 31, which cannot be a month.",
     "Month/day inversion on every XML date, breaking temporal ordering checks."),
    ("ASM-02", "coupon_discount is percentage points, not a fraction.",
     "T1-P4: XML publishes '15%'; applying 15/100 reproduces order_total on every order in both sources.",
     "order_total would be wrong on every discounted order."),
    ("ASM-03", "Product prices are GST-inclusive; tax_amount is the included component order_price/11 taken before the discount.",
     "T1-P4: round(order_price/11, 2) reproduces the published tax_amount on all orders; products.tax_category is GST_STANDARD.",
     "GST would be added again and order_total overstated."),
    ("ASM-04", "Absence of a coupon code is the same fact in both sources.",
     "T1-P4: JSON writes an empty string, XML writes an empty <Coupon_Code/> element; both mean 'no coupon'.",
     "Two encodings of the same absence would be read as a cross-source conflict."),
    ("ASM-05", "A missing prescribed string is written as the literal three characters NaN, not an empty cell or a pandas null.",
     "Specification s.4 Task 2; applies to coupon_code, promo_code and the derived narrative fields.",
     "Missing values would be unreadable to the marking comparison."),
    ("ASM-06", "deliveries.delay_reason value 'none' is a category, not a missing value.",
     "T1-P4: it co-occurs with delay_days = 0 and on_time_in_full = True on every such row.",
     "A valid category would be destroyed and on-time deliveries would look unexplained."),
    ("ASM-07", "No source precedence rule is applied when reconciling.",
     "T1-P6: zero field-level conflicts within or across sources after normalisation.",
     "An arbitrary precedence rule would hide a genuine conflict if one ever appeared."),
    ("ASM-08", "deliveries is 1:1 with orders in this package because every order is Completed.",
     "T1-P3: Order_Status is 'Completed' and Delivery_Status is 'Delivered' on all rows in both exports.",
     "No filter is needed for this package, but T2-B3 applies one regardless, because a package containing open orders would otherwise silently break the deliveries grain."),
    ("ASM-09", "product_reviews.language_code comes from the structured source attribute only.",
     "T1-P8 / specification s.4 Task 3: language must not be inferred from the Latin-analysis field.",
     "Inferred language would misclassify every accented-Latin and mixed-script review."),
    ("ASM-10", "Review timestamps after 2018-12-31 are valid, not errors.",
     "T1-P4: reviews run to 2019-02-20 while orders stop at 2018-12-31; a review follows its delivery.",
     "Valid late reviews would be dropped as out-of-period."),
    ("ASM-11", "products.unit_price is the current catalogue price and does not overwrite order_items.unit_price.",
     "T1-P3: order lines are historical 2018 events; catalogue prices are a separate current-state attribute.",
     "Historical line revenue and order arithmetic would be retrospectively rewritten."),
    ("ASM-12", "Inactive products (active_flag = N) are retained.",
     "T1-P4: 40 of 1,000 products are inactive yet still appear on 2018 order lines.",
     "Order items would be orphaned from their product parent."),
    ("ASM-13", "When review_body_clean is the literal NaN sentinel, review_length_chars = 0, review_word_count = 0 and contains_non_latin_script = False.",
     "Specification s.4 Task 3 requires the sentinel not to be counted as an ordinary review, and the dictionary types these three fields numeric/numeric/boolean and non-nullable, so the string NaN cannot be written into them. T1-P8: no value in this package cleans to empty, so the rule is defined for the contract and for private tests rather than exercised here.",
     "Length 3 and word count 1 would treat the sentinel as a one-word review and bias every text-length statistic; writing the string NaN would break the required numeric/boolean types."),
    ("ASM-14", "Monetary rounding uses Python's round(value, 2), not pandas/numpy Series.round(2).",
     "T1-P4: the two rules disagree on half-cent values because numpy scales by 100 and rounds the binary result. Python's round reproduces the published order_total on every order in both exports; numpy's does not.",
     "Roughly 1% of orders would fail the published order_total contract at tolerance 0.01, and Task 2 would emit values the marking comparison rejects."),
    ("ASM-15", "The Latin analysis keeps Latin-script letters with their combining marks, keeps digits and narrow punctuation, and drops non-Latin letters together with the wide and fullwidth punctuation that belongs to the same script.",
     "Specification s.4 Task 3 retains 'applicable' digits and punctuation. Wide and fullwidth punctuation (U+3001, U+FF0C, U+3002) is applicable to the script being removed, not to the Latin analysis: keeping it leaves a residue such as 'candle quest 768 , , . .' on a Chinese review. Narrow Latin typography (<<, >>, em dash) is ambiguous or narrow width and survives. NFC composes European diacritics; a combining mark is kept only after a retained Latin base.",
     "Keeping wide punctuation would leave meaningless residue from the removed script; dropping narrow punctuation would discard characters the contract keeps."),
    ("ASM-16", "coupon_discount is written as an integer while every observed discount is a whole percentage point.",
     "The dictionary compares coupon_discount 'exact after published normalisation' rather than on a tolerance, and both exports publish whole points ('15' / '15%'). The cast is conditional on the parsed column being integral, so a fractional discount in another package keeps the float representation.",
     "'15.0' would be the only float stand-in for an integer among the six outputs and could fail an exact string comparison."),
], columns=["assumption_id", "assumption", "evidence", "consequence_if_wrong"])
save_profile(assumptions, "assumptions_register")
assumptions

[saved] profiling/Group005_T1_assumptions_register.csv  (16 rows)


,assumption_id,assumption,evidence,consequence_if_wrong
0,ASM-01,XML dates are day-first DD/MM/YYYY.,"T1-P4: the first component reaches 31, which cannot be a month.","Month/day inversion on every XML date, breaking temporal ordering checks."
1,ASM-02,"coupon_discount is percentage points, not a fraction.",T1-P4: XML publishes '15%'; applying 15/100 reproduces order_total on every order in b...,order_total would be wrong on every discounted order.
2,ASM-03,Product prices are GST-inclusive; tax_amount is the included component order_price/11 ...,"T1-P4: round(order_price/11, 2) reproduces the published tax_amount on all orders; pro...",GST would be added again and order_total overstated.
3,ASM-04,Absence of a coupon code is the same fact in both sources.,"T1-P4: JSON writes an empty string, XML writes an empty <Coupon_Code/> element; both m...",Two encodings of the same absence would be read as a cross-source conflict.
4,ASM-05,"A missing prescribed string is written as the literal three characters NaN, not an emp...","Specification s.4 Task 2; applies to coupon_code, promo_code and the derived narrative...",Missing values would be unreadable to the marking comparison.
5,ASM-06,"deliveries.delay_reason value 'none' is a category, not a missing value.",T1-P4: it co-occurs with delay_days = 0 and on_time_in_full = True on every such row.,A valid category would be destroyed and on-time deliveries would look unexplained.
6,ASM-07,No source precedence rule is applied when reconciling.,T1-P6: zero field-level conflicts within or across sources after normalisation.,An arbitrary precedence rule would hide a genuine conflict if one ever appeared.
7,ASM-08,deliveries is 1:1 with orders in this package because every order is Completed.,T1-P3: Order_Status is 'Completed' and Delivery_Status is 'Delivered' on all rows in b...,"No filter is needed for this package, but T2-B3 applies one regardless, because a pack..."
8,ASM-09,product_reviews.language_code comes from the structured source attribute only.,T1-P8 / specification s.4 Task 3: language must not be inferred from the Latin-analysi...,Inferred language would misclassify every accented-Latin and mixed-script review.
9,ASM-10,"Review timestamps after 2018-12-31 are valid, not errors.",T1-P4: reviews run to 2019-02-20 while orders stop at 2018-12-31; a review follows its...,Valid late reviews would be dropped as out-of-period.


## 2. Source-to-target mapping

---
#### `T1-M1` — Source-to-target mapping

The mapping is a **data artefact, not code**. It lives in
`Group005_source_to_target_mapping.csv` — one of the required submission files — so any group
member can edit it in a spreadsheet without touching Python.

This notebook therefore **audits** the mapping rather than generating it. Six checks run
against the published contract and against the measurements made earlier in this notebook, so
a hand edit that breaks the contract, invents a path, or contradicts the measured data fails
loudly instead of reaching the marker.

| Column | Content |
|---|---|
| `mapping_id` | stable `MAP-###` identifier (Appendix A submission checklist) |
| `source_format` | `JSON`, `XML`, `both` or `derived` |
| `json_source_path` | JSONPath-style structural path, or `N/A` |
| `xml_source_path` | absolute XPath, or `N/A` |
| `transformation_or_derivation` | the normalisation or derivation applied |
| `overlap_or_conflict_rule` | how duplication/overlap and disagreement are handled |
| `notebook_evidence` | stable Task 1 section IDs supporting the row |

Path columns hold **only** structural paths, several inputs separated by `|`. Where a derived
field is built from another *target* field, that dependency is stated in
`transformation_or_derivation`, not smuggled into the path.

In [25]:
# T1-M1: audit the hand-maintained mapping CSV against the contract and the measurements.
mapping = pd.read_csv(MAPPING_PATH, keep_default_na=False)
print(f"loaded {MAPPING_PATH}: {len(mapping)} rows x {mapping.shape[1]} columns")

# --- 1. completeness and field order, against the published data dictionary ----
required = list(zip(dictionary.output_table, dictionary.field_name))
supplied = list(zip(mapping.output_table, mapping.target_field))
assert supplied == required, (
    f"mapping does not match the dictionary. missing={sorted(set(required)-set(supplied))} "
    f"extra={sorted(set(supplied)-set(required))} order_ok={supplied == required}")

# --- 2. stable per-table ids, exactly as the supplied template numbers them -----
# Rebuild MAP-<table>-<nn> IDs to detect reordered or renumbered mapping rows.
expected_ids = [f"MAP-{table}-{n:02d}"
                for table in dictionary.output_table.drop_duplicates()
                for n in range(1, (dictionary.output_table == table).sum() + 1)]
assert list(mapping.mapping_id) == expected_ids, (
    f"mapping_id must follow the template scheme MAP-<output_table>-<nn>; "
    f"first mismatch at {next((i for i, (a, b) in enumerate(zip(mapping.mapping_id, expected_ids)) if a != b), None)}")

# --- 3. no blank cell reaches the submitted file -------------------------------
blank = mapping[(mapping == "").any(axis=1) | mapping.isna().any(axis=1)]
assert blank.empty, f"blank mapping cells:\n{blank}"

# --- 4. source_format uses only the published vocabulary -----------------------
allowed_formats = {"JSON", "XML", "both", "derived"}
bad_format = set(mapping.source_format) - allowed_formats
assert not bad_format, f"source_format values outside the published vocabulary: {bad_format}"

# --- 5. every path token resolves against the structural profile in T1-P2 ------
known_paths = set(structure_profile["structural_path"])
unresolved = [(row.mapping_id, column, token)
              for row in mapping.itertuples()
              for column in ("json_source_path", "xml_source_path")
              for cell in [getattr(row, column)] if cell != "N/A"
              for token in [t.strip() for t in cell.split("|")]
              if token not in known_paths]
assert not unresolved, f"path tokens not found in T1-P2: {unresolved[:5]}"

# --- 6. claims in the text must match what T1-P4 and T1-P6 actually measured ----
# String containment only - no regex, keeping the boundary of T1-P8 intact.
def expected_domain_phrase(table, field):
    values = CATEGORICAL_DOMAINS[(table, field)]
    if len(values) == 1:
        return f"Single observed value {values[0]}."
    return "Observed domain {" + ", ".join(values) + "}."

stale_domain = [(row.mapping_id, f"{row.output_table}.{row.target_field}")
                for row in mapping.itertuples()
                if (row.output_table, row.target_field) in CATEGORICAL_DOMAINS
                and expected_domain_phrase(row.output_table, row.target_field)
                not in row.transformation_or_derivation]
assert not stale_domain, f"domain claims disagree with the T1-P4 measurement: {stale_domain[:5]}"

measured = overlap_profile.set_index("target_table")
def expected_overlap_opening(table, key_name):
    m = measured.loc[table]
    if int(m.json_records) == 0 or int(m.xml_records) == 0:
        return f"Single-source attribute:"
    return (f"Reconcile on {key_name}: {int(m.within_json_repeat_pairs):,} repeated JSON rows and "
            f"{int(m.within_xml_repeat_pairs):,} repeated XML rows inside their own export, plus "
            f"{int(m.keys_in_both_sources):,} keys published by both exports, collapse to "
            f"{int(m.canonical_keys_expected):,} canonical rows.")

stale_overlap = [(row.mapping_id, row.output_table)
                 for row in mapping.itertuples()
                 if expected_overlap_opening(row.output_table, PRIMARY_KEYS[row.output_table])
                 not in row.overlap_or_conflict_rule]
assert not stale_overlap, f"overlap counts disagree with the T1-P6 measurement: {stale_overlap[:5]}"

print(f"  1. {len(mapping)}/{len(dictionary)} required target fields, dictionary order preserved")
print(f"  2. mapping_id {expected_ids[0]}..{expected_ids[-1]} (template scheme)")
print( "  3. no blank cell")
print(f"  4. source_format vocabulary: {sorted(set(mapping.source_format))}")
print(f"  5. all path tokens resolve against the {len(known_paths)} profiled structural paths")
print(f"  6. {len(CATEGORICAL_DOMAINS)} domain claims and {len(measured)} overlap rules match the measurements")
mapping.head(12)

loaded Group005_source_to_target_mapping.csv: 111 rows x 9 columns
  1. 111/111 required target fields, dictionary order preserved
  2. mapping_id MAP-orders-01..MAP-product_reviews-21 (template scheme)
  3. no blank cell
  4. source_format vocabulary: ['JSON', 'XML', 'both', 'derived']
  5. all path tokens resolve against the 201 profiled structural paths
  6. 41 domain claims and 6 overlap rules match the measurements


,mapping_id,output_table,target_field,source_format,json_source_path,xml_source_path,transformation_or_derivation,overlap_or_conflict_rule,notebook_evidence
0,MAP-orders-01,orders,order_id,both,$.orders[*].header.orderID,/OperationsExport/Orders/Order/Header/Order_ID,Trim only. Identifier case and leading zeros preserved (HORD######); used as the order...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P1 | T1-P3 | T1-P6
1,MAP-orders-02,orders,source_system_record_id,both,$.orders[*].header.sourceSystemRecordID,/OperationsExport/Orders/Order/Header/Source_System_Record_ID,Trim only; case and hyphenation preserved (SRC-005-H-######). Carried as a source line...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
2,MAP-orders-03,orders,customer_id,both,$.orders[*].header.customerID,/OperationsExport/Orders/Order/Header/Customer_ID,Trim only; leading zeros preserved (CUS#####). Foreign key to customers.customer_id.,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P3 | T1-P7
3,MAP-orders-04,orders,order_timestamp,both,$.orders[*].header.orderTimestamp,/OperationsExport/Orders/Order/Header/Order_Timestamp,JSON is already YYYY-MM-DD HH:MM:SS; XML is DD/MM/YYYY HH:MM:SS (day component reaches...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4
4,MAP-orders-05,orders,sales_channel,both,$.orders[*].header.salesChannel,/OperationsExport/Orders/Order/Header/Sales_Channel,"Trim only; structured category retained in published case. Observed domain {Mobile, St...",Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
5,MAP-orders-06,orders,payment_method,both,$.orders[*].header.paymentMethod,/OperationsExport/Orders/Order/Header/Payment_Method,"Trim only; internal spacing kept. Observed domain {Bank Transfer, Card, Gift Card, Pay...",Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
6,MAP-orders-07,orders,currency,both,$.orders[*].header.currency,/OperationsExport/Orders/Order/Header/Currency,Trim and upper-case ISO code; single observed value AUD. Cross-checked against the AUD...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4
7,MAP-orders-08,orders,nearest_warehouse,both,$.orders[*].header.nearestWarehouse,/OperationsExport/Orders/Order/Header/Nearest_Warehouse,"Trim only; warehouse name kept in published case. Observed domain {Bakers, Nickolson, ...",Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
8,MAP-orders-09,orders,order_status,both,$.orders[*].header.orderStatus,/OperationsExport/Orders/Order/Header/Order_Status,Trim only. Single observed value Completed.,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P2 | T1-P4 | T1-P6
9,MAP-orders-10,orders,order_price,derived,$.orders[*].shoppingCart[*].quantity | $.orders[*].shoppingCart[*].unitPrice | $.order...,/OperationsExport/Orders/Order/Shopping_Cart/Item/Quantity | /OperationsExport/Orders/...,Derived as step 2 of the published order arithmetic: the sum over the order's cart lin...,Reconcile on order_id: 68 repeated JSON rows and 68 repeated XML rows inside their own...,T1-P4 | T1-P6 | T1-P9


In [26]:
summary = (mapping.groupby(["output_table", "source_format"]).size()
           .unstack(fill_value=0)
           .reindex(columns=["JSON", "XML", "both", "derived"], fill_value=0))
summary["total"] = summary.sum(axis=1)
summary = summary.reindex(["orders", "order_items", "customers", "deliveries", "products", "product_reviews"])
summary.loc["TOTAL"] = summary.sum()
save_profile(summary.reset_index(), "mapping_source_format_summary")
summary

[saved] profiling/Group005_T1_mapping_source_format_summary.csv  (7 rows)


source_format,JSON,XML,both,derived,total
output_table,,,,,
orders,0,0,18,5,23
order_items,0,0,5,1,6
customers,20,0,0,0,20
deliveries,0,0,20,0,20
products,0,20,0,1,21
product_reviews,0,0,14,7,21
TOTAL,20,20,57,14,111


## 3. Text and regex functions

### 3.1 Cleaning and extraction implementation

---
#### `T1-P8` — Narrative field profile (inputs to the Task 3 regex contract)

The bounded narrative fields are profiled for the markers, markup, URLs, entities, emoji and
scripts they actually contain. Regular expressions appear here for the first time, applied to
values already extracted by the structured parsers — and written in the verbose `(?x)` style
of the Week 4 applied session. The cleaning functions themselves belong to Task 3.

In [27]:
MARKERS = ["[SYSTEM]", "[CATALOGUE]", "[VERIFIED_PURCHASE]", "[SOURCE:", "[RATING:",
           "#verified-buyer", "@store_support", "PROMO:", "Reference:", "SKU:"]

narrative_pools = {
    "orders.customer_note":           columns_of((json_headers, "customerNote"), (xml_headers, "Customer_Note")),
    "product_reviews.review_text":    columns_of((json_reviews, "reviewText"), (xml_reviews, "Review_Text")),
    "products.product_description":   columns_of((xml_products, "Product_Description")),
    "deliveries.delivery_note_clean": columns_of((json_deliveries, "deliveryNoteClean"), (xml_deliveries, "Delivery_Note_Clean")),
    "product_reviews.review_title":   columns_of((json_reviews, "reviewTitle"), (xml_reviews, "Review_Title")),
}

TAG_RE = re.compile(r"</?([a-zA-Z][a-zA-Z0-9]*)")
ENTITY_RE = re.compile(r"&[a-zA-Z]{2,8};|&#\d{2,5};")

# Latin-script letters whose Unicode name does not begin with "LATIN ". Defined
# locally, not imported from Group005_text_functions, so this Task 1 measurement
# stays independent of the Task 3 implementation it is later cross-checked against.
T1_LATIN_SCRIPT_RANGES = (
    (0x00AA, 0x00AA), (0x00BA, 0x00BA), (0x02B0, 0x02B8), (0x02E0, 0x02E4),
    (0x1D2C, 0x1D5C), (0x1D9B, 0x1DBE), (0x2071, 0x2071), (0x207F, 0x207F),
    (0x212A, 0x212B), (0x2132, 0x2132), (0x214E, 0x214E), (0x2183, 0x2183),
    (0x2C7D, 0x2C7D), (0xA770, 0xA770), (0xA7F2, 0xA7F4), (0xA7F8, 0xA7F9),
    (0xAB5C, 0xAB5F), (0xAB69, 0xAB69), (0xFF21, 0xFF3A), (0xFF41, 0xFF5A),
    (0x10780, 0x10785), (0x10787, 0x107B0), (0x107B2, 0x107BA),
)

def t1_is_latin_letter(char):
    """Unicode Script=Latin test; a name substring is not the script property."""
    if unicodedata.name(char, "").startswith("LATIN "):
        return True
    code_point = ord(char)
    return any(low <= code_point <= high for low, high in T1_LATIN_SCRIPT_RANGES)

def has_non_latin_letter(text):
    for char in text:
        if unicodedata.category(char).startswith("L") and not t1_is_latin_letter(char):
            return True
    return False

narrative_rows = []
for field, pool in narrative_pools.items():
    pool = pool.dropna().astype(str)
    joined = " ".join(pool)
    narrative_rows.append({
        "narrative_field": field,
        "values_profiled": len(pool),
        "markers_present": ", ".join(m for m in MARKERS if m in joined) or "none",
        "html_tags": ", ".join(sorted(set(TAG_RE.findall(joined)))) or "none",
        "html_entities_after_parse": ", ".join(sorted(set(ENTITY_RE.findall(joined)))) or "none",
        "with_url": int(pool.str.contains("http", regex=False).sum()),
        "with_emoji_So": int(pool.map(lambda t: any(unicodedata.category(c) == "So" for c in t)).sum()),
        "with_non_ascii": int(pool.map(lambda t: any(ord(c) > 127 for c in t)).sum()),
        "with_non_latin_letter": int(pool.map(has_non_latin_letter).sum()),
    })
narrative_profile = pd.DataFrame(narrative_rows)
save_profile(narrative_profile, "narrative_field_profile")
narrative_profile

[saved] profiling/Group005_T1_narrative_field_profile.csv  (5 rows)


,narrative_field,values_profiled,markers_present,html_tags,html_entities_after_parse,with_url,with_emoji_So,with_non_ascii,with_non_latin_letter
0,orders.customer_note,5636,"[SYSTEM], PROMO:",p,none,5636,0,0,0
1,product_reviews.review_text,7892,"[VERIFIED_PURCHASE], [SOURCE:, [RATING:, #verified-buyer, @store_support, Reference:, ...","article, aside, blockquote, br, div, p, section",&nbsp;,7892,7892,7892,315
2,products.product_description,1000,[CATALOGUE],section,none,1000,0,0,0
3,deliveries.delivery_note_clean,5636,none,none,none,0,0,0,0
4,product_reviews.review_title,7892,none,none,none,0,0,541,315


In [28]:
# Embedded business references, counted on the raw narrative with pandas .str + verbose regex.
ORDER_REF_RE = r'''(?x)
    \b            # word boundary, so a longer near-match is rejected
    (?P<prefix>[HC])ORD
    \d{6}         # exactly six digits
    \b
'''
SKU_RE = r'''(?x)
    \b SKU-
    (?P<token>[A-Za-z0-9]+)
    \b
'''
PROMO_RE = r'''(?x)
    \b (?P<band>B[1-5]) SAVE-
    \d{2}         # exactly two digits
    \b
'''

review_pool = narrative_pools["product_reviews.review_text"].dropna().astype(str)
note_pool = narrative_pools["orders.customer_note"].dropna().astype(str)

reference_profile = pd.DataFrame([
    {"reference": "order reference", "published_format": "HORD or CORD + exactly 6 digits",
     "source_field": "product_reviews.review_text",
     "values_carrying_it": int(review_pool.str.extract(ORDER_REF_RE)["prefix"].notna().sum()),
     "prefixes_observed": {key: int(value) for key, value in
                           review_pool.str.extract(ORDER_REF_RE)["prefix"].value_counts().items()},
     "target_field": "extracted_order_reference"},
    {"reference": "product SKU", "published_format": "SKU- + one or more ASCII letters/digits",
     "source_field": "product_reviews.review_text",
     "values_carrying_it": int(review_pool.str.extract(SKU_RE)["token"].notna().sum()),
     "prefixes_observed": {key: int(value) for key, value in
                           review_pool.str.extract(SKU_RE)["token"].str[:3].value_counts().items()},
     "target_field": "extracted_product_sku"},
    {"reference": "promotion code", "published_format": "B1SAVE- to B5SAVE- + exactly 2 digits",
     "source_field": "orders.customer_note",
     "values_carrying_it": int(note_pool.str.extract(PROMO_RE)["band"].notna().sum()),
     "prefixes_observed": {key: int(value) for key, value in
                           note_pool.str.extract(PROMO_RE)["band"].value_counts().items()},
     "target_field": "orders.promo_code"},
])
save_profile(reference_profile, "embedded_reference_profile")
reference_profile

[saved] profiling/Group005_T1_embedded_reference_profile.csv  (3 rows)


,reference,published_format,source_field,values_carrying_it,prefixes_observed,target_field
0,order reference,HORD or CORD + exactly 6 digits,product_reviews.review_text,7892,{'H': 7892},extracted_order_reference
1,product SKU,SKU- + one or more ASCII letters/digits,product_reviews.review_text,7892,"{'VEL': 3965, 'CAN': 3927}",extracted_product_sku
2,promotion code,B1SAVE- to B5SAVE- + exactly 2 digits,orders.customer_note,2103,{'B1': 2103},orders.promo_code


**Material narrative findings**

* `orders.customer_note` carries `[SYSTEM]`, `<p>` markup, an optional `PROMO: <code>` wrapper
  and a help URL — no emoji, no non-ASCII text.
* `products.product_description` carries `[CATALOGUE]`, `<section>` markup and a catalogue URL.
* `product_reviews.review_text` is the noisy field: six HTML-like tags, all seven
  bracketed/social markers, a URL, emoji in **every** value, and a complete
  `Reference: <order> SKU: <sku>` wrapper on every value. `&nbsp;` survives XML parsing and
  must be decoded with `html.unescape`, not stripped as literal text.
* Non-Latin scripts (Chinese, Devanagari, Arabic, Japanese, Cyrillic and others) appear in the
  review bodies. `review_body_clean` must preserve them; `review_body_latin_analysis` is a
  **separate** derived field. A non-ASCII character is not automatically non-Latin.
* `deliveries.delivery_note_clean` and `product_reviews.review_title` arrive already clean.
* Only the `HORD` order-reference prefix occurs in this package, but the `CORD` alternative is
  published in the contract and is therefore accepted by the extractor.

In [29]:
from Group005_text_functions import (
    build_latin_analysis,
    clean_narrative_text,
    contains_non_latin_script,
    extract_order_reference,
    extract_product_sku,
    extract_promo_code,
)

---
#### Check: regex stays inside its assessed boundary

The specification allows regex only for bounded narrative work, never for document structure.
This check re-reads the notebook's own source and reports which **Task 1** sections contain a
regular expression, so the structural-parsing claim is verifiable rather than asserted.

Its scope is deliberately narrow: it answers *"did structural parsing stay regex-free?"*, not
*"where is regex used at all?"*. Regex is also used, legitimately, in Task 3
(`Group005_text_functions.py`, for the bounded cleaning and reference patterns the specification
assigns to it) and in Task 4 (the `VAL-TEXT` checks, which match extracted references against
their published formats).

In [30]:
import io, json as _json

NOTEBOOK_PATH = f"{GROUP_ID}_solution.ipynb"
REGEX_MARKERS = ("re.compile", "re.sub(", "re.findall(", "re.match(", "re.search(", "regex=True")

if os.path.exists(NOTEBOOK_PATH):
    with io.open(NOTEBOOK_PATH, encoding="utf-8") as fh:
        cells = _json.load(fh)["cells"]
    section, rows = "(preamble)", []
    for cell in cells:
        source = "".join(cell["source"])
        if cell["cell_type"] == "markdown":
            for line in source.splitlines():
                if line.lstrip("# ").startswith("`T1-"):
                    section = line.strip("# ").split("`")[1]
        else:
            if "REGEX_MARKERS" in source:
                continue          # the checking cell names the markers, so skip itself
            hits = [m for m in REGEX_MARKERS if m in source]
            if hits:
                rows.append({"section": section, "regex_constructs": ", ".join(sorted(set(hits)))})
    regex_usage = pd.DataFrame(rows).drop_duplicates()
    print("Task 1 sections containing a regular expression:", sorted(regex_usage["section"].unique()))
    print("structural parsing sections (T1-P1, T1-P2) clean:",
          not regex_usage["section"].isin(["T1-P1", "T1-P2"]).any())
    print(regex_usage)
else:
    print("notebook file not on disk (running as the exported script) - check skipped")

Task 1 sections containing a regular expression: ['T1-P8']
structural parsing sections (T1-P1, T1-P2) clean: True
  section     regex_constructs
0   T1-P8           re.compile
2   T1-P8  re.compile, re.sub(


### 3.2 Public and student-designed tests

Three design decisions in `Group005_text_functions.py` decide most of the boundary cases, so
they are stated here before the evidence.

**Markers are removed from an explicit list, never by a general bracket pattern.** The
specification publishes exactly seven removable markers. A pattern such as `\[.*?\]` would also
delete `[NOTE]` or `[URGENT]`, which is customer wording and must survive. Case `TXT-18` tests
precisely this.

**Reference extraction rejects near-matches instead of truncating them.** A word boundary alone
is not enough for a SKU: in `SKU-ABC123-extra` the hyphen counts as a boundary, so `\b` would
return the valid-looking prefix `SKU-ABC123`. Instead of a word boundary, `_search_bounded`
inspects the character on each side of the match, and `_is_boundary` treats `-` and `_` as part
of the token rather than as separators, so the malformed value is rejected outright, as `TXT-12`
requires.

**Emoji are identified by Unicode block, not by category `So`.** Category `So` is much broader
than emoji: it also holds `©`, `®`, `™` and `°`, which are ordinary text symbols a customer may
have typed and which the specification never asks to remove. The published narratives use
`©` and `™` only inside `[SOURCE: ...]` markers, so both definitions agree on this package —
but they disagree on any private case that puts one in the body text.

#### Public cases

The unit supplies `templates/A1_public_text_test_cases.csv`. It is loaded with
`keep_default_na=False`; without that, pandas reads the expected value `NaN` as a missing cell
and every sentinel case compares incorrectly.

Every case is shown in full — input, expected, observed and status — so the evidence can be read
without rerunning anything.

In [31]:
PUBLIC_CASES_PATH = os.path.join(TEMPLATE_DIR, "A1_public_text_test_cases.csv")
public_cases = pd.read_csv(PUBLIC_CASES_PATH, keep_default_na=False)

TEXT_FUNCTIONS = {
    "clean_narrative_text": clean_narrative_text,
    "extract_order_reference": extract_order_reference,
    "extract_product_sku": extract_product_sku,
    "extract_promo_code": extract_promo_code,
    "build_latin_analysis": build_latin_analysis,
    "contains_non_latin_script": contains_non_latin_script,
}

public_results = pd.DataFrame([
    {
        "case_id": row.case_id,
        "function": row.function,
        "input_value": row.input_value,
        "expected": row.expected_output,
        "observed": str(TEXT_FUNCTIONS[row.function](row.input_value)),
        "purpose": row.purpose,
    }
    for row in public_cases.itertuples()
]).assign(status=lambda t: (t.expected == t.observed).map({True: "PASS", False: "FAIL"}))

print(f"T3-TEXT-01  public test cases: "
      f"{int((public_results.status == 'PASS').sum())}/{len(public_results)} PASS")
print(public_results.groupby("function").status.value_counts().unstack(fill_value=0).to_string())

with pd.option_context("display.max_colwidth", 76):
    display(public_results[["case_id", "function", "input_value", "expected", "observed", "status"]])

T3-TEXT-01  public test cases: 18/18 PASS
status                     PASS
function                       
build_latin_analysis          2
clean_narrative_text          8
contains_non_latin_script     1
extract_order_reference       3
extract_product_sku           2
extract_promo_code            2


,case_id,function,input_value,expected,observed,status
0,TXT-01,clean_narrative_text,[SYSTEM] <p>Leave at reception</p> PROMO: B3SAVE-24 https://orders.examp...,leave at reception,leave at reception,PASS
1,TXT-02,extract_promo_code,[SYSTEM] <p>Leave at reception</p> PROMO: B3SAVE-24,B3SAVE-24,B3SAVE-24,PASS
2,TXT-03,clean_narrative_text,[SOURCE: mobile-app] <p>Caf&eacute; setup was easy 😊</p> Reference: HORD...,café setup was easy,café setup was easy,PASS
3,TXT-04,extract_order_reference,Reference: HORD123456 | SKU: SKU-ABC123,HORD123456,HORD123456,PASS
4,TXT-05,extract_product_sku,Reference: HORD123456 | SKU: SKU-ABC123,SKU-ABC123,SKU-ABC123,PASS
5,TXT-06,extract_order_reference,codes XHORD1234567 and HORD12345 are invalid,NaN,NaN,PASS
6,TXT-07,build_latin_analysis,service était bon 包装很好,service était bon,service était bon,PASS
7,TXT-08,contains_non_latin_script,service était bon 包装很好,True,True,PASS
8,TXT-09,build_latin_analysis,包装很好,NaN,NaN,PASS
9,TXT-10,clean_narrative_text,[VERIFIED_PURCHASE] <div>Reliable for daily use</div>,reliable for daily use,reliable for daily use,PASS


**Observed result: 18/18 PASS. Status: PASS.** Three cases decide more than they look.

`TXT-18` supplies `[NOTE] Keep bracketed customer wording` and expects
`[note] keep bracketed customer wording`. The brackets survive because `[NOTE]` is not one of the
seven published markers; only the case changes.

`TXT-12` supplies `SKU-ABC123-extra` and expects the sentinel rather than `SKU-ABC123`. Returning
the prefix would be a silent data error, not partial credit.

`TXT-03` combines a parameterised `[SOURCE: ...]` marker, an HTML entity, an emoji, the full
`Reference: … | SKU: …` wrapper and a social token, and expects `café setup was easy`. The `é`
proves entity decoding runs before tag stripping and that accented Latin text is never treated
as foreign.

#### Student-designed cases

The public set leaves gaps the specification explicitly asks about: *"Document and test your text
functions with matched, unmatched, missing, multilingual and near-match examples."* It contains
no `None` input, no case for the `NaN` sentinel behaviour recorded in `ASM-13`, and only one case
for `contains_non_latin_script`.

The 61 cases below close those gaps, grouped by the five categories the specification names plus
the sentinel and symbol behaviour this package relies on.

In [32]:
OWN_CASES = [
    ("missing",      "None input to the cleaner",                clean_narrative_text, (None,), MISSING),
    ("missing",      "None input to the order extractor",        extract_order_reference, (None,), MISSING),
    ("missing",      "None input to the SKU extractor",          extract_product_sku, (None,), MISSING),
    ("missing",      "None input to the promo extractor",        extract_promo_code, (None,), MISSING),
    ("missing",      "whitespace-only input",                    clean_narrative_text, ("   ",), MISSING),
    ("missing",      "marker and emoji leave nothing readable",  clean_narrative_text, ("[SYSTEM] \U0001F60A",), MISSING),

    ("near-match",   "order reference with seven digits",        extract_order_reference, ("HORD1234567",), MISSING),
    ("near-match",   "order reference with five digits",         extract_order_reference, ("HORD12345",), MISSING),
    ("near-match",   "order reference with letters",             extract_order_reference, ("HORDABCDEF",), MISSING),
    ("near-match",   "order reference embedded in a word",       extract_order_reference, ("XHORD123456",), MISSING),
    ("near-match",   "promotion band outside B1-B5",             extract_promo_code, ("B6SAVE-12",), MISSING),
    ("near-match",   "promotion code with three digits",         extract_promo_code, ("B1SAVE-123",), MISSING),
    ("near-match",   "promotion code with one digit",            extract_promo_code, ("B1SAVE-1",), MISSING),
    ("near-match",   "SKU embedded in a longer token",           extract_product_sku, ("XSKU-ABC1",), MISSING),
    ("near-match",   "SKU prefix with no body",                  extract_product_sku, ("SKU-",), MISSING),

    ("matched",      "CORD prefix accepted, not only HORD",      extract_order_reference, ("ref CORD000001 here",), "CORD000001"),
    ("matched",      "lower-case reference normalised to upper", extract_order_reference, ("reference: hord123456",), "HORD123456"),
    ("matched",      "lower-case SKU normalised to upper",       extract_product_sku, ("SKU: sku-can00871",), "SKU-CAN00871"),
    ("matched",      "promotion code inside its wrapper",        extract_promo_code, ("PROMO: B5SAVE-09",), "B5SAVE-09"),

    ("unmatched",    "no reference present at all",              extract_order_reference, ("no reference at all",), MISSING),
    ("unmatched",    "note carrying no promotion code",          extract_promo_code, ("[SYSTEM] no special instruction",), MISSING),

    ("multilingual", "accented Latin is not non-Latin",          contains_non_latin_script, ("café niño über",), False),
    ("multilingual", "plain ASCII is not non-Latin",             contains_non_latin_script, ("hello world",), False),
    ("multilingual", "Cyrillic detected",                        contains_non_latin_script, ("привет",), True),
    ("multilingual", "Arabic detected",                          contains_non_latin_script, ("مرحبا",), True),
    ("multilingual", "diacritics survive the Latin analysis",    build_latin_analysis, ("café niño über",), "café niño über"),
    ("multilingual", "CJK letters dropped, Latin kept",          build_latin_analysis, ("good 很好 product",), "good product"),
    ("multilingual", "CJK punctuation dropped with its script",  build_latin_analysis, ("service était bon 包装很好。",), "service était bon"),
    ("multilingual", "Latin typography survives",                build_latin_analysis, ("«cite» — dash 很好",), "«cite» — dash"),
    ("multilingual", "cleaner preserves multilingual text",      clean_narrative_text, ("<p>我用candle quest 768</p>",), "我用candle quest 768"),

    ("sentinel",     "sentinel contains only Latin letters",     contains_non_latin_script, (MISSING,), False),
    ("symbol",       "trade mark sign is not an emoji",          clean_narrative_text, ("<p>Product™ good</p>",), "product™ good"),
    ("symbol",       "degree sign is not an emoji",              clean_narrative_text, ("<p>at 25° today</p>",), "at 25° today"),
    ("boundary",     "unpublished bracketed wording preserved",  clean_narrative_text, ("[URGENT] please deliver",), "[urgent] please deliver"),
    ("multilingual", "Vietnamese diacritics are Latin, not foreign", contains_non_latin_script, ("s\u1ea3n ph\u1ea9m r\u1ea5t t\u1ed1t",), False),
    ("multilingual", "Vietnamese survives the Latin analysis", build_latin_analysis, ("caf\u00e9 r\u1ea5t t\u1ed1t",), "caf\u00e9 r\u1ea5t t\u1ed1t"),
    ("multilingual", "Japanese alone leaves no Latin letter", build_latin_analysis, ("\u826f\u3044\u5546\u54c1",), MISSING),

    ("symbol",       "emoji variation sequence removed whole", clean_narrative_text, ("great \u00a9\ufe0f product",), "great product"),
    ("symbol",       "wavy dash with variation selector removed", clean_narrative_text, ("great \u3030\ufe0f product",), "great product"),
    ("symbol",       "bare wavy dash is text, not emoji", clean_narrative_text, ("great \u3030 dash",), "great \u3030 dash"),
    ("symbol",       "keycap emoji removed with its base digit", clean_narrative_text, ("rating 1\ufe0f\u20e3 good",), "rating good"),

    ("boundary",     "emoji inside the promo wrapper still matches", clean_narrative_text, ("PROMO: \U0001F60A B1SAVE-12",), MISSING),
    ("boundary",     "emoji inside the reference wrapper still matches", clean_narrative_text, ("Reference: \U0001F60A HORD123456 | SKU: SKU-ABC123",), MISSING),
    ("near-match",   "decomposed diacritic before a reference", extract_order_reference, ("e\u0301HORD123456",), MISSING),
    ("near-match",   "decomposed diacritic after a reference", extract_order_reference, ("HORD123456\u0301",), MISSING),
    ("near-match",   "CJK letter before a SKU", extract_product_sku, ("\u5305SKU-ABC123",), MISSING),
    ("near-match",   "Arabic-Indic digits are not ASCII digits", extract_order_reference, ("HORD\u0661\u0662\u0663\u0664\u0665\u0666",), MISSING),
    ("multilingual", "readable wording between the halves is not a separator", clean_narrative_text, ("Reference: HORD123456 \u6ce8\u610f SKU: SKU-ABC123",), "reference: hord123456 \u6ce8\u610f sku: sku-abc123"),
    ("boundary",     "an incomplete wrapper is left as customer text", clean_narrative_text, ("Reference: HORD123456",), "reference: hord123456"),
    ("boundary",     "a malformed SKU half leaves the whole wrapper intact", clean_narrative_text, ("Reference: HORD123456 | SKU: SKU-ABC123-extra",), "reference: hord123456 | sku: sku-abc123-extra"),
    ("boundary",     "an em dash is a valid wrapper separator", clean_narrative_text, ("Reference: HORD123456 \u2014 SKU: SKU-ABC123",), MISSING),
    ("near-match",   "CJK after a promo code blocks the wrapper", clean_narrative_text, ("PROMO: B1SAVE-12\u4e2d",), "promo: b1save-12\u4e2d"),
    ("near-match",   "combining mark after a promo code blocks it", clean_narrative_text, ("PROMO: B1SAVE-12\u0301",), "promo: b1save-12\u0301"),
    ("near-match",   "long s must not case-fold into the SKU prefix", extract_product_sku, ("\u017fKU-ABC123",), MISSING),
    ("near-match",   "long s must not case-fold inside B1SAVE", extract_promo_code, ("B1\u017fAVE-12",), MISSING),
    ("near-match",   "long s is not an ASCII SKU character", extract_product_sku, ("SKU-\u017f",), MISSING),
    ("near-match",   "dotless i is not an ASCII SKU character", extract_product_sku, ("SKU-\u0131",), MISSING),
    ("near-match",   "dotted capital I is not an ASCII SKU character", extract_product_sku, ("SKU-\u0130",), MISSING),
    ("near-match",   "the Kelvin sign is not an ASCII SKU character", extract_product_sku, ("SKU-\u212a",), MISSING),
    ("boundary",     "a marker spelled with a non-ASCII letter is not published", clean_narrative_text, ("[VER\u0130FIED_PURCHASE] Keep",), "[ver\u0069\u0307fied_purchase] keep"),
    ("boundary",     "a social token spelled with dotless i is not published", clean_narrative_text, ("#ver\u0131fied-buyer Keep",), "#ver\u0131fied-buyer keep"),
]

own_results = pd.DataFrame([
    {"case_id": f"OWN-{i:02d}", "category": category, "description": description,
     "input_value": repr(args[0]), "expected": repr(expected), "observed": repr(function(*args))}
    for i, (category, description, function, args, expected) in enumerate(OWN_CASES, start=1)
]).assign(status=lambda t: (t.expected == t.observed).map({True: "PASS", False: "FAIL"}))

print(f"T3-TEXT-02  student-designed cases: "
      f"{int((own_results.status == 'PASS').sum())}/{len(own_results)} PASS")
print(own_results.groupby("category").status.value_counts().unstack(fill_value=0).to_string())

with pd.option_context("display.max_colwidth", 54):
    display(own_results)

T3-TEXT-02  student-designed cases: 61/61 PASS
status        PASS
category          
boundary         8
matched          4
missing          6
multilingual    13
near-match      21
sentinel         1
symbol           6
unmatched        2


,case_id,category,description,input_value,expected,observed,status
0,OWN-01,missing,None input to the cleaner,None,'NaN','NaN',PASS
1,OWN-02,missing,None input to the order extractor,None,'NaN','NaN',PASS
2,OWN-03,missing,None input to the SKU extractor,None,'NaN','NaN',PASS
3,OWN-04,missing,None input to the promo extractor,None,'NaN','NaN',PASS
4,OWN-05,missing,whitespace-only input,' ','NaN','NaN',PASS
...,...,...,...,...,...,...,...
56,OWN-57,near-match,dotless i is not an ASCII SKU character,'SKU-ı','NaN','NaN',PASS
57,OWN-58,near-match,dotted capital I is not an ASCII SKU character,'SKU-İ','NaN','NaN',PASS
58,OWN-59,near-match,the Kelvin sign is not an ASCII SKU character,'SKU-K','NaN','NaN',PASS
59,OWN-60,boundary,a marker spelled with a non-ASCII letter is not pu...,'[VERİFIED_PURCHASE] Keep','[veri̇fied_purchase] keep','[veri̇fied_purchase] keep',PASS


**Observed result: 61/61 PASS. Status: PASS.**

The `near-match` group is deliberately the largest, at 21 cases. Rejection is where a
plausible-looking implementation quietly fails: every one of those inputs *looks* like a valid
reference. Nineteen are extraction cases that must return the sentinel; the other two are
cleaning cases, where the malformed text is not a reference and must therefore survive in
`review_body_clean` rather than be stripped out.

The `multilingual` group encodes the distinction the specification is most explicit about —
*"A non-ASCII character is not automatically non-Latin"*. `café niño über` is non-ASCII in places
yet entirely Latin, so `contains_non_latin_script` returns `False` and `build_latin_analysis`
returns it unchanged. Cyrillic and Arabic return `True`. The cleaner keeps
`我用candle quest 768` intact, because `review_body_clean` must preserve valid multilingual UTF-8
rather than erase it.

The `symbol` group guards the emoji definition. `Product™` and `25°` survive cleaning because
they are text symbols, not emoji — a distinction that Unicode category `So` alone does not make.

#### Derived review measures

The specification defines two measures the six published functions do not return, and fixes
their behaviour when the cleaned body is the sentinel:

> `review_length_chars` = number of Python characters in `review_body_clean`
> `review_word_count` = number of whitespace-separated tokens in `review_body_clean`
> *"When `review_body_clean` is the literal `NaN`, preserve the published sentinel behaviour
> rather than counting the three letters as an ordinary review."*

`ASM-13` records the reading applied here: the measures are `0` and `0`, not `3` and `1`, and
`contains_non_latin_script` is `False`. They are always emitted as a number and a boolean, never
as the string `NaN`, because the dictionary types those three fields as numeric, numeric and
boolean with `nullable=False`.

In [33]:
def review_length_chars(cleaned_body):
    '''Characters in review_body_clean; 0 when the body is the published sentinel.'''
    return 0 if cleaned_body == MISSING else len(cleaned_body)

def review_word_count(cleaned_body):
    '''Whitespace-separated tokens in review_body_clean; 0 for the sentinel.'''
    return 0 if cleaned_body == MISSING else len(cleaned_body.split())

measure_demo = pd.DataFrame([
    {"review_body_clean": body,
     "review_length_chars": review_length_chars(body),
     "review_word_count": review_word_count(body),
     "contains_non_latin_script": contains_non_latin_script(body),
     "note": note}
    for body, note in [
        ("the candle shift 970 is reliable", "ordinary English review"),
        ("我用candle quest 768", "multilingual review, letters preserved"),
        ("café était bon", "accented Latin only"),
        (MISSING, "sentinel: counted as 0/0/False, not 3/1"),
    ]
])
print("T3-TEXT-03  sentinel behaviour of the derived review measures")
display(measure_demo)

T3-TEXT-03  sentinel behaviour of the derived review measures


,review_body_clean,review_length_chars,review_word_count,contains_non_latin_script,note
0,the candle shift 970 is reliable,32,6,False,ordinary English review
1,我用candle quest 768,18,3,True,"multilingual review, letters preserved"
2,café était bon,14,3,False,accented Latin only
3,NaN,0,0,False,"sentinel: counted as 0/0/False, not 3/1"


**Observed result: the sentinel row reports `0`, `0` and `False`. Status: PASS.** The three
letters of `NaN` are not counted as a one-word review, and no numeric or boolean field ever
receives the string.

#### Cross-check against the Task 1 measurements

Passing a supplied test file proves the functions satisfy 18 published examples. It does not
prove they behave correctly on this package's 7,892 review bodies and 5,636 customer notes.

The profiling in `T1-P8` counted those narratives independently, before these functions were
written and through a different code path. Running the functions over the same raw values should
reproduce those counts exactly.

In [34]:
raw_reviews = columns_of((json_reviews, "reviewText"), (xml_reviews, "Review_Text")).dropna().astype(str)
raw_notes = columns_of((json_headers, "customerNote"), (xml_headers, "Customer_Note")).dropna().astype(str)

narrative_measured = pd.read_csv(os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_narrative_field_profile.csv"))
reference_measured = pd.read_csv(os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_embedded_reference_profile.csv"))
measured = lambda name: int(reference_measured.loc[reference_measured.reference == name,
                                                   "values_carrying_it"].iloc[0])

cross_check = pd.DataFrame([
    {"quantity": "review bodies containing a non-Latin letter",
     "task_3_functions": int(raw_reviews.map(clean_narrative_text).map(contains_non_latin_script).sum()),
     "measured_in_T1_P8": int(narrative_measured.loc[
         narrative_measured.narrative_field == "product_reviews.review_text",
         "with_non_latin_letter"].iloc[0])},
    {"quantity": "review bodies yielding an order reference",
     "task_3_functions": int((raw_reviews.map(extract_order_reference) != MISSING).sum()),
     "measured_in_T1_P8": measured("order reference")},
    {"quantity": "review bodies yielding a product SKU",
     "task_3_functions": int((raw_reviews.map(extract_product_sku) != MISSING).sum()),
     "measured_in_T1_P8": measured("product SKU")},
    {"quantity": "customer notes yielding a promotion code",
     "task_3_functions": int((raw_notes.map(extract_promo_code) != MISSING).sum()),
     "measured_in_T1_P8": measured("promotion code")},
]).assign(status=lambda t: (t.task_3_functions == t.measured_in_T1_P8).map({True: "PASS", False: "FAIL"}))

# The extracted SKU must also name the product the review is actually about.
review_products = columns_of((json_reviews, "productID"), (xml_reviews, "Product_ID")).astype(str)
catalogue_sku = dict(zip(xml_products["Product_ID"], xml_products["Product_Sku"].str.upper()))
sku_matches = int(sum(catalogue_sku.get(product) == sku for product, sku
                      in zip(review_products, raw_reviews.map(extract_product_sku))))

print("T3-TEXT-04  extraction reproduces the T1-P8 counts")
print(f"T3-TEXT-05  extracted SKU names the reviewed product: {sku_matches}/{len(raw_reviews)}")
cross_check

T3-TEXT-04  extraction reproduces the T1-P8 counts
T3-TEXT-05  extracted SKU names the reviewed product: 7892/7892


,quantity,task_3_functions,measured_in_T1_P8,status
0,review bodies containing a non-Latin letter,315,315,PASS
1,review bodies yielding an order reference,7892,7892,PASS
2,review bodies yielding a product SKU,7892,7892,PASS
3,customer notes yielding a promotion code,2103,2103,PASS


**Observed result: all four counts agree, and 7,892 of 7,892 extracted SKUs name the reviewed
product. Status: PASS.**

The last line is the strongest of the five. `extract_product_sku` reads only the review text and
has no access to the product catalogue, so a full match against `products.product_sku` for the
referenced `product_id` means the extraction is not merely well-formed but correct.

**Interpretation and limitation.** No narrative in this package cleans to an empty string, so the
sentinel path in `clean_narrative_text` is exercised only by the cases above, never by the data
itself. `ASM-13` records the same gap for the derived review measures. The rule is implemented
and tested against the published contract; it is simply not observed here, and a private test is
the only thing that will exercise it.

## 4. Build the six standardised relational tables

#### Task 2 — Six standardised relational tables

Task 2 turns the profiled source frames into exactly the six tables in the public data
dictionary. Reconciliation is value based: comparable fields are normalised first, every
non-missing value for a business key must agree, and only then are repeated/source-overlap
rows collapsed. No JSON-over-XML or XML-over-JSON precedence rule is used.

#### `T2-R1` — Reconcile canonical rows without source precedence

In [35]:
task2_conflict_rows = []

def _non_missing_values(series):
    '''Return distinct non-missing values while preserving their parsed Python types.'''
    values = []
    for value in series:
        if is_absent(value):
            continue
        if not any(value == observed for observed in values):
            values.append(value)
    return values

def reconcile_canonical(table):
    '''Coalesce normalised source frames to one canonical row per published key.

    One populated value may fill a missing counterpart. Two different populated values
    are recorded and rejected rather than silently resolved by source priority.
    '''
    key = PRIMARY_KEYS[table]
    frames = [NORMALISED[(table, source)] for source in ("JSON", "XML")
              if (table, source) in NORMALISED and not NORMALISED[(table, source)].empty]
    combined = pd.concat(frames, ignore_index=True)
    rows = []
    for key_value, group in combined.groupby(key, sort=True, dropna=False):
        row = {key: key_value}
        for column in combined.columns:
            if column == key:
                continue
            values = _non_missing_values(group[column])
            if len(values) > 1:
                task2_conflict_rows.append({
                    "target_table": table,
                    "primary_key": key,
                    "key_value": key_value,
                    "target_field": column,
                    "non_missing_values": " | ".join(map(str, values)),
                })
                # Keep conflicts unresolved rather than applying accidental source
                # precedence; the validation register records the resulting FAIL.
                row[column] = None
            else:
                row[column] = values[0] if values else None
        rows.append(row)
    return pd.DataFrame(rows, columns=combined.columns)


canonical_orders_source = reconcile_canonical("orders")
canonical_items_source = reconcile_canonical("order_items")
canonical_deliveries_source = reconcile_canonical("deliveries")
canonical_reviews_source = reconcile_canonical("product_reviews")


def reconcile_narrative_derivation(table, target_field, source_specs, transform):
    '''Derive a narrative field from parser-obtained raw values, then reconcile it.

    This preserves the published processing order: reference extraction happens
    on the raw JSON value/XML element content, before entity decoding or other
    narrative cleaning.  Differing derived results are recorded and left
    unresolved rather than selected by source order.
    '''
    key = PRIMARY_KEYS[table]
    pieces = []
    for frame, source_key, source_value in source_specs:
        pieces.append(pd.DataFrame({
            key: norm_id(frame[source_key]),
            target_field: frame[source_value].map(transform),
        }))
    combined = pd.concat(pieces, ignore_index=True)
    reconciled = {}
    for key_value, group in combined.groupby(key, sort=True, dropna=False):
        values = _non_missing_values(group[target_field])
        if len(values) > 1:
            task2_conflict_rows.append({
                "target_table": table,
                "primary_key": key,
                "key_value": key_value,
                "target_field": target_field,
                "non_missing_values": " | ".join(map(str, values)),
            })
            reconciled[key_value] = None
        else:
            reconciled[key_value] = values[0] if values else MISSING
    return pd.Series(reconciled, name=target_field)


order_note_clean_by_id = reconcile_narrative_derivation(
    "orders", "customer_note_clean",
    [(json_headers, "orderID", "customerNote"),
     (xml_headers, "Order_ID", "Customer_Note")],
    clean_narrative_text,
)
order_promo_by_id = reconcile_narrative_derivation(
    "orders", "promo_code",
    [(json_headers, "orderID", "customerNote"),
     (xml_headers, "Order_ID", "Customer_Note")],
    extract_promo_code,
)
review_clean_by_id = reconcile_narrative_derivation(
    "product_reviews", "review_body_clean",
    [(json_reviews, "reviewID", "reviewText"),
     (xml_reviews, "Review_ID", "Review_Text")],
    clean_narrative_text,
)
review_order_reference_by_id = reconcile_narrative_derivation(
    "product_reviews", "extracted_order_reference",
    [(json_reviews, "reviewID", "reviewText"),
     (xml_reviews, "Review_ID", "Review_Text")],
    extract_order_reference,
)
review_product_sku_by_id = reconcile_narrative_derivation(
    "product_reviews", "extracted_product_sku",
    [(json_reviews, "reviewID", "reviewText"),
     (xml_reviews, "Review_ID", "Review_Text")],
    extract_product_sku,
)

task2_conflicts = pd.DataFrame(task2_conflict_rows, columns=[
    "target_table", "primary_key", "key_value", "target_field", "non_missing_values"
])
task2_conflict_path = os.path.join(PROFILE_DIR, f"{GROUP_ID}_T2_reconciliation_conflicts.csv")
task2_conflicts.to_csv(task2_conflict_path, index=False)
print(f"[saved] {task2_conflict_path}  ({len(task2_conflicts)} rows)")
if task2_conflicts.empty:
    print("Reconciliation: no field-level conflict detected after normalisation.")
else:
    # Log conflicts and continue so outputs remain inspectable; VAL-CONFLICT-01
    # carries the failure into the validation register.
    print(f"Reconciliation: {len(task2_conflicts)} field-level conflict(s) recorded in "
          f"{os.path.basename(task2_conflict_path)}; see check VAL-CONFLICT-01.")
    print(task2_conflicts.head(20).to_string(index=False))



[saved] profiling/Group005_T2_reconciliation_conflicts.csv  (0 rows)
Reconciliation: no field-level conflict detected after normalisation.


In [36]:
def trim_series(series, case=None):
    '''Trim/collapse structured strings while retaining genuine missing values.'''
    def clean(value):
        if is_absent(value):
            return None
        result = " ".join(str(value).split())
        if case == "upper":
            return result.upper()
        if case == "lower":
            return result.lower()
        return result
    return series.map(clean)



#### `T2-B2` — Order items and published order arithmetic

In [37]:
order_items = canonical_items_source.copy()
order_items["order_item_id"] = trim_series(order_items["order_item_id"])
order_items["order_id"] = trim_series(order_items["order_id"])
order_items["product_id"] = trim_series(order_items["product_id"])
order_items["quantity"] = order_items["quantity"].astype(int)
order_items["unit_price"] = round2(order_items["unit_price"].astype(float))
order_items["line_revenue"] = round2(order_items["quantity"] * order_items["unit_price"])

order_price_by_id = round2(order_items.groupby("order_id", sort=True)["line_revenue"].sum())



### 4.1 `orders`

In [38]:
orders = canonical_orders_source.copy()
orders["order_id"] = trim_series(orders["order_id"])
orders["source_system_record_id"] = trim_series(orders["source_system_record_id"])
orders["customer_id"] = trim_series(orders["customer_id"])
orders["currency"] = trim_series(orders["currency"], "upper")
orders["coupon_code"] = trim_series(orders["coupon_code"], "upper").fillna(MISSING)
# Exact comparison requires whole percentage points to remain integers. Preserve
# a float only if the source contains a genuine fractional discount.
coupon_discount = orders["coupon_discount"].astype(float)
orders["coupon_discount"] = (
    coupon_discount.astype(int) if (coupon_discount % 1 == 0).all() else coupon_discount
)
orders["delivery_charges"] = round2(orders["delivery_charges"].astype(float))
orders["customer_lat"] = orders["customer_lat"].astype(float)
orders["customer_long"] = orders["customer_long"].astype(float)
orders["customer_note_clean"] = orders["order_id"].map(order_note_clean_by_id)
orders["promo_code"] = orders["order_id"].map(order_promo_by_id)
orders["order_price"] = orders["order_id"].map(order_price_by_id)
assert orders["order_price"].notna().all(), "an order has no canonical order-item lines"
orders["tax_amount"] = round2(orders["order_price"] / 11)
orders["order_total"] = round2(
    orders["order_price"] * (1 - orders["coupon_discount"] / 100)
    + orders["delivery_charges"]
)
orders = orders.drop(columns=["customer_note"])

### 4.2 `order_items`

`order_items` is built in `T2-B2` above rather than here, because the published arithmetic
runs in the opposite direction to the section order: `orders.order_price` is the sum of the
rounded `line_revenue` values, so the item grain has to be materialised before the order grain
can be completed. Building it inside this section would leave section 4.1 referring to a table
that does not exist yet.

The finished table is shown below at its published grain and field order.

In [39]:
# The dictionary field order is applied to every table together in section 7; here the
# item grain is simply shown as built, in the published field order for readability.
order_items_fields = list(dictionary.loc[dictionary.output_table == "order_items", "field_name"])
order_items_view = order_items[order_items_fields]

print(f"order_items: {len(order_items_view):,} rows x {order_items_view.shape[1]} columns")
print(f"  order_item_id complete and unique : "
      f"{order_items_view['order_item_id'].is_unique and order_items_view['order_item_id'].notna().all()}")
print(f"  line_revenue == round(quantity * unit_price, 2) : "
      f"{int((order_items_view['line_revenue'] == round2(order_items_view['quantity'] * order_items_view['unit_price'])).sum()):,}"
      f"/{len(order_items_view):,}")
display(order_items_view.head())

order_items: 15,739 rows x 6 columns
  order_item_id complete and unique : True
  line_revenue == round(quantity * unit_price, 2) : 15,739/15,739


,order_item_id,order_id,product_id,quantity,unit_price,line_revenue
0,HITM0000001,HORD000001,PRD0950,1,238.05,238.05
1,HITM0000002,HORD000001,PRD0613,1,1377.73,1377.73
2,HITM0000003,HORD000001,PRD0447,1,961.68,961.68
3,HITM0000004,HORD000002,PRD0857,1,1044.86,1044.86
4,HITM0000005,HORD000002,PRD0451,2,1300.44,2600.88


### 4.3 `customers`

#### `T2-B1` — Build the two single-source entity tables

In [40]:
customers = pd.DataFrame({
    "customer_id": trim_series(json_customers["customerID"]),
    "signup_date": pd.to_datetime(json_customers["signupDate"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d"),
    "loyalty_tier": trim_series(json_customers["loyaltyTier"]),
    "customer_segment": trim_series(json_customers["customerSegment"]),
    "age_band": trim_series(json_customers["ageBand"]),
    "preferred_channel": trim_series(json_customers["preferredChannel"]),
    "home_suburb": trim_series(json_customers["homeSuburb"]),
    "prior_12m_orders": json_customers["prior12MOrders"].astype(int),
    "lifetime_value_before_period": round2(json_customers["lifetimeValueBeforePeriod"].astype(float)),
    "marketing_consent": json_customers["marketingConsent"].astype(bool),
    "home_postcode": trim_series(json_customers["homePostcode"]),
    "home_state": trim_series(json_customers["homeState"], "upper"),
    "home_country": trim_series(json_customers["homeCountry"]),
    "preferred_language": trim_series(json_customers["preferredLanguage"], "lower"),
    "acquisition_source": trim_series(json_customers["acquisitionSource"]),
    "account_status": trim_series(json_customers["accountStatus"]),
    "preferred_device": trim_series(json_customers["preferredDevice"]),
    "email_domain": trim_series(json_customers["emailDomain"], "lower"),
    "household_size_band": trim_series(json_customers["householdSizeBand"]),
    "contact_frequency_preference": trim_series(json_customers["contactFrequencyPreference"]),
})



### 4.4 `deliveries`

#### `T2-B3` — Completed-order deliveries and canonical reviews

In [41]:
completed_order_ids = set(orders.loc[orders["order_status"] == "Completed", "order_id"])
deliveries = canonical_deliveries_source[
    canonical_deliveries_source["order_id"].isin(completed_order_ids)
].copy()
for column in ("delivery_id", "order_id", "carrier", "service_level", "delivery_status",
               "delay_reason", "delivery_window"):
    deliveries[column] = trim_series(deliveries[column])
for column in ("delay_days", "fulfilment_hours", "promised_days", "tracking_event_count"):
    deliveries[column] = deliveries[column].astype(int)
deliveries["delivery_cost"] = round2(deliveries["delivery_cost"].astype(float))
for column in ("shipping_distance_km", "estimated_carbon_kg"):
    deliveries[column] = deliveries[column].astype(float)
# This field is already clean and its published case is part of the categorical contract.
deliveries["delivery_note_clean"] = trim_series(deliveries["delivery_note_clean"]).fillna(MISSING)



### 4.5 `products`

In [42]:
products = pd.DataFrame({
    "product_id": trim_series(xml_products["Product_ID"]),
    "product_name": trim_series(xml_products["Product_Name"]),
    "category": trim_series(xml_products["Category"]),
    "brand": trim_series(xml_products["Brand"]),
    "unit_price": round2(money_series(xml_products["Unit_Price"])),
    "unit_cost": round2(money_series(xml_products["Unit_Cost"])),
    "launch_year": xml_products["Launch_Year"].astype(int),
    "warranty_months": xml_products["Warranty_Months"].astype(int),
    "weight_kg": xml_products["Weight_Kg"].astype(float),
    "product_sku": trim_series(xml_products["Product_Sku"], "upper"),
    "subcategory": trim_series(xml_products["Subcategory"]),
    "model_family": trim_series(xml_products["Model_Family"]),
    "colour": trim_series(xml_products["Colour"]),
    "supplier_id": trim_series(xml_products["Supplier_ID"]),
    "supplier_country": trim_series(xml_products["Supplier_Country"]),
    "launch_date": pd.to_datetime(xml_products["Launch_Date"], format="%d/%m/%Y").dt.strftime("%Y-%m-%d"),
    "tax_category": trim_series(xml_products["Tax_Category"], "upper"),
    "package_type": trim_series(xml_products["Package_Type"]),
    "recyclable_packaging": norm_bool(xml_products["Recyclable_Packaging"]),
    "active_flag": norm_bool(xml_products["Active_Flag"]),
    "product_description_clean": xml_products["Product_Description"].map(clean_narrative_text),
})

### 4.6 `product_reviews`

In [43]:
product_reviews = canonical_reviews_source.copy()
for column in ("review_id", "order_id", "order_item_id", "product_id", "customer_id"):
    product_reviews[column] = trim_series(product_reviews[column])
product_reviews["language_code"] = trim_series(product_reviews["language_code"], "lower")
product_reviews["rating"] = product_reviews["rating"].astype(int)
product_reviews["review_title"] = trim_series(product_reviews["review_title"]).fillna(MISSING)
product_reviews["helpful_votes"] = product_reviews["helpful_votes"].astype(int)
product_reviews["review_body_clean"] = product_reviews["review_id"].map(review_clean_by_id)
product_reviews["review_body_latin_analysis"] = product_reviews["review_body_clean"].map(build_latin_analysis)
product_reviews["contains_non_latin_script"] = product_reviews["review_body_clean"].map(contains_non_latin_script)
product_reviews["extracted_order_reference"] = product_reviews["review_id"].map(
    review_order_reference_by_id
)
product_reviews["extracted_product_sku"] = product_reviews["review_id"].map(
    review_product_sku_by_id
)
product_reviews["review_length_chars"] = product_reviews["review_body_clean"].map(
    lambda value: 0 if value == MISSING else len(value)
)
product_reviews["review_word_count"] = product_reviews["review_body_clean"].map(
    lambda value: 0 if value == MISSING else len(value.split())
)
product_reviews = product_reviews.drop(columns=["review_text"])

#### `T2-O1` — Enforce the public schema and write exactly six outputs

In [44]:
STANDARDISED = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "deliveries": deliveries,
    "products": products,
    "product_reviews": product_reviews,
}

TASK2_PRIMARY_KEYS = {
    "orders": "order_id",
    "order_items": "order_item_id",
    "customers": "customer_id",
    "deliveries": "delivery_id",
    "products": "product_id",
    "product_reviews": "review_id",
}

for table, frame in STANDARDISED.items():
    required_columns = list(dictionary.loc[dictionary.output_table == table, "field_name"])
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    extra_columns = sorted(set(frame.columns) - set(required_columns))
    assert not missing_columns and not extra_columns, (
        f"{table} schema mismatch: missing={missing_columns}, extra={extra_columns}"
    )
    STANDARDISED[table] = (frame[required_columns]
                           .sort_values(TASK2_PRIMARY_KEYS[table], kind="stable")
                           .reset_index(drop=True))

OUTPUT_PATHS = {
    table: os.path.join(OUTPUT_DIR, f"{GROUP_ID}_{table}_standardised.csv")
    for table in STANDARDISED
}

for table, frame in STANDARDISED.items():
    frame.to_csv(OUTPUT_PATHS[table], index=False)
    print(f"[saved] {OUTPUT_PATHS[table]}  ({len(frame):,} rows x {frame.shape[1]} columns)")

[saved] outputs/Group005_orders_standardised.csv  (5,000 rows x 23 columns)
[saved] outputs/Group005_order_items_standardised.csv  (15,739 rows x 6 columns)
[saved] outputs/Group005_customers_standardised.csv  (500 rows x 20 columns)
[saved] outputs/Group005_deliveries_standardised.csv  (5,000 rows x 20 columns)
[saved] outputs/Group005_products_standardised.csv  (1,000 rows x 21 columns)
[saved] outputs/Group005_product_reviews_standardised.csv  (7,000 rows x 21 columns)


## 5. Reconcile overlap and verify relationships

In [45]:
task2_reconciliation = pd.DataFrame([
    {
        "target_table": table,
        "source_rows": sum(len(NORMALISED[(table, source)])
                           for source in ("JSON", "XML") if (table, source) in NORMALISED),
        "canonical_rows": len(frame),
        "canonical_primary_keys": frame[PRIMARY_KEYS[table]].nunique(dropna=False),
    }
    for table, frame in [
        ("orders", canonical_orders_source),
        ("order_items", canonical_items_source),
        ("deliveries", canonical_deliveries_source),
        ("product_reviews", canonical_reviews_source),
    ]
])
task2_reconciliation

,target_table,source_rows,canonical_rows,canonical_primary_keys
0,orders,5636,5000,5000
1,order_items,17750,15739,15739
2,deliveries,5636,5000,5000
3,product_reviews,7892,7000,7000


## 6. Validation register

Keep each check executable and give it a stable `VAL-...` ID. Immediately after
each code check, record the observed result, `PASS`/`FAIL`, evidence and
resolution/interpretation. A genuine, explained failure is preferable to a
fabricated pass.

Required areas include schema/types, primary and foreign keys, row flow and
source coverage, overlap, arithmetic, temporal logic, text/reference behaviour
and multilingual handling.


### 6.1 Schema and type checks (`VAL-SCHEMA-...`)

Validation conditions:

- Output-file check passes only when the files physically present in `outputs/` are exactly the six required standardised CSV files.
- Schema check passes only when each submitted table has exactly the field names and field order specified by `public_data_dictionary.csv`; missing or extra fields fail the check.
- Required-field check passes only when every non-nullable field contains no pandas missing value, no empty string and no literal `NaN`.
- Type check passes only when numbers parse numerically, booleans are exactly `True` or `False`, dates round-trip as `YYYY-MM-DD`, timestamps round-trip as `YYYY-MM-DD HH:MM:SS`, and nullable-string fields follow the literal-`NaN` contract.
- Categorical-domain check passes only when every submitted categorical value is contained in the domain measured from the parsed source data; unexpected values fail the check.
- Numeric-range check passes only when every serialized numeric value is parseable and satisfies its field-specific semantic range, such as non-negative money, valid coordinates, positive weight and ratings from 1 to 5.
- Assumption checks in this section validate schema/domain/sentinel conditions for `ASM-04`, `ASM-05`, `ASM-06` and `ASM-16`.

In [46]:
task4_checks = []

def stable_validation_id(check_id):
    '''Require an explicit stable ID such as VAL-SCHEMA-01.'''
    normalized = str(check_id).upper()
    if re.fullmatch(r"VAL-[A-Z]+-[0-9]{2}", normalized) is None:
        raise ValueError(f"Validation ID must be explicit and stable: {check_id!r}")
    return normalized

def task4_check(check_id, description, passed, observed, evidence,
                pass_interpretation, fail_resolution):
    '''Append one complete, citable validation-register row.'''
    normalized_id = stable_validation_id(check_id)
    if any(row["check_id"] == normalized_id for row in task4_checks):
        raise ValueError(f"Duplicate validation ID: {normalized_id}")
    status = "PASS" if bool(passed) else "FAIL"
    task4_checks.append({
        "check_id": normalized_id,
        "description": description,
        "observed": str(observed),
        "status": status,
        "evidence": evidence,
        "resolution_or_interpretation": (
            pass_interpretation if status == "PASS" else fail_resolution
        ),
    })

def print_validation_results(section_name, prefixes):
    '''Print the observed result, status, evidence and interpretation for one section.'''
    results = pd.DataFrame(task4_checks)
    selected = results[results.check_id.str.startswith(tuple(prefixes))]
    passed = int(selected.status.eq("PASS").sum())
    print(f"\n{section_name}: {passed}/{len(selected)} checks PASS")
    print(selected[[
        "check_id", "description", "observed", "status", "evidence",
        "resolution_or_interpretation",
    ]].to_string(index=False))

### Validation setup and submitted-file loading

The submitted CSVs are re-read from disk, once typed and once as exact text, so the checks test the files a marker receives rather than the in-memory frames.

In [47]:
# Read both the typed and exact-text submitted representations.  The latter
# keeps identifier zeros, boolean spelling and the literal NaN sentinel visible.
written = {table: pd.read_csv(path, keep_default_na=False)
           for table, path in OUTPUT_PATHS.items()}
written_text = {table: pd.read_csv(path, dtype=str, keep_default_na=False)
                for table, path in OUTPUT_PATHS.items()}

### Output-file completeness validation

In [48]:
expected_output_files = {os.path.basename(path) for path in OUTPUT_PATHS.values()}
observed_output_files = {os.path.basename(path) for path in glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))}
task4_check(
    "VAL-FILE-01", "exactly six required output files",
    observed_output_files == expected_output_files, sorted(observed_output_files),
    "T2-O1; OUTPUT_PATHS compared with every CSV physically present in OUTPUT_DIR.",
    "All and only the six public data products are present.",
    "Remove extra CSVs or regenerate the missing/misnamed public data product.",
)

### Schema, required-field, and type validation

For every submitted table, these checks compare field order and names with the public dictionary, confirm non-nullable fields are populated, and validate serialized values against the declared data types, date formats, nullable flags, and literal-`NaN` rules. Primary-key checks are intentionally implemented in section 6.2.

In [49]:
for table_index, (table, frame) in enumerate(written_text.items(), start=1):
    contract = dictionary[dictionary.output_table == table]
    expected_columns = list(contract.field_name)
    schema_ok = list(frame.columns) == expected_columns
    task4_check(
        f"VAL-SCHEMA-{table_index:02d}", f"{table}: exact field names and order",
        schema_ok,
        f"expected_columns={len(expected_columns)}, observed_columns={len(frame.columns)}, "
        f"order_match={schema_ok}, missing={sorted(set(expected_columns) - set(frame.columns))}, "
        f"extra={sorted(set(frame.columns) - set(expected_columns))}",
        f"public_data_dictionary.csv positions compared with {os.path.basename(OUTPUT_PATHS[table])}.",
        "The submitted table has no missing, extra or reordered field.",
        "Reindex the table to the dictionary positions and remove helper columns before export.",
    )

    required_fields = list(contract.loc[~contract.nullable.astype(bool), "field_name"])
    missing_required = {
        column: int(
            frame[column].isna().sum()
            + frame[column].eq("").sum()
            + frame[column].eq(MISSING).sum()
        )
        for column in required_fields
        if frame[column].isna().any() or frame[column].eq("").any() or frame[column].eq(MISSING).any()
    }
    task4_check(
        f"VAL-MISSING-{table_index:02d}", f"{table}: required fields have no empty, pandas-missing or literal-NaN values",
        not missing_required, missing_required or "none",
        "Submitted CSV read with dtype=str and keep_default_na=False; required fields must contain neither empty cells nor the literal NaN sentinel.",
        "All non-nullable fields are represented with a value of the declared type; literal NaN is reserved for nullable string outputs.",
        "Derive or reconcile the missing required value; use literal NaN only for a nullable string field whose contract permits it.",
    )

    type_errors = []
    for row in contract.itertuples():
        series = frame[row.field_name]
        if row.data_type == "string":
            valid = series.notna().all() and series.ne("").all()
            if not bool(row.nullable):
                valid = valid and series.ne(MISSING).all()
        elif row.data_type == "number":
            valid = series.notna().all() and series.ne("").all() and series.ne(MISSING).all()
            valid = valid and pd.to_numeric(series, errors="coerce").notna().all()
        elif row.data_type == "boolean":
            valid = series.notna().all() and series.ne("").all() and series.ne(MISSING).all()
            valid = valid and set(series) <= {"True", "False"}
        elif row.data_type == "date":
            valid = series.notna().all() and series.ne("").all() and series.ne(MISSING).all()
            parsed = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
            valid = valid and parsed.notna().all() and parsed.dt.strftime("%Y-%m-%d").eq(series).all()
        elif row.data_type == "datetime":
            valid = series.notna().all() and series.ne("").all() and series.ne(MISSING).all()
            parsed = pd.to_datetime(series, format="%Y-%m-%d %H:%M:%S", errors="coerce")
            valid = valid and parsed.notna().all() and parsed.dt.strftime("%Y-%m-%d %H:%M:%S").eq(series).all()
        else:
            valid = False
        if not valid:
            type_errors.append(f"{row.field_name}:{row.data_type}")
    task4_check(
        f"VAL-TYPE-{table_index:02d}", f"{table}: serialized types, formats and missing representation",
        not type_errors,
        f"fields_checked={len(contract)}, type_errors={type_errors or 'none'}, nullable_contract=checked",
        "Submitted CSV lexical values checked against every public dictionary data_type, nullable flag and the literal-NaN contract.",
        "Numbers parse, booleans are exactly True/False, dates/timestamps round-trip, and only nullable strings may use literal NaN.",
        "Correct the listed serialization, conversion or missing-value representation before writing the submitted CSV.",
    )

print_validation_results("Schema checks", ("VAL-SCHEMA-",))
print_validation_results("Missing-value checks", ("VAL-MISSING-",))
print_validation_results("Type checks", ("VAL-TYPE-",))


Schema checks: 6/6 checks PASS
     check_id                                  description                                                                         observed status                                                                                      evidence                                  resolution_or_interpretation
VAL-SCHEMA-01          orders: exact field names and order expected_columns=23, observed_columns=23, order_match=True, missing=[], extra=[]   PASS          public_data_dictionary.csv positions compared with Group005_orders_standardised.csv. The submitted table has no missing, extra or reordered field.
VAL-SCHEMA-02     order_items: exact field names and order   expected_columns=6, observed_columns=6, order_match=True, missing=[], extra=[]   PASS     public_data_dictionary.csv positions compared with Group005_order_items_standardised.csv. The submitted table has no missing, extra or reordered field.
VAL-SCHEMA-03       customers: exact field names and order 

### Categorical-domain validation

Each categorical field is compared with the domain measured from the parser outputs, so no domain is hard-coded. The check detects any value falling outside that measured set — an invented category, a case change, a stripped separator. It does **not** establish that individual rows kept their original value: replacing one allowed category with another leaves the observed set unchanged and would still pass. Row-level fidelity is covered instead by the source reconciliation in `VAL-FLOW-…` and `VAL-CONFLICT-01`.

In [50]:
# Allowed categorical values are data-derived from the parser outputs, never hard-coded.
for domain_index, ((table, field), allowed_values) in enumerate(CATEGORICAL_DOMAINS.items(), start=1):
    observed_values = set(written_text[table][field])
    allowed = set(map(str, allowed_values))
    unexpected = sorted(observed_values - allowed)
    task4_check(
        f"VAL-DOMAIN-{domain_index:02d}", f"{table}.{field}: values stay in measured source domain",
        not unexpected,
        f"observed_count={len(observed_values)}, unexpected_count={len(unexpected)}, unexpected={unexpected}",
        f"T1-P4 measured source domain compared with exact-text submitted {table}.{field}.",
        "Every observed value lies inside the domain measured from the sources; no category was invented, case-folded or split during transformation.",
        "Trace unexpected values to an incorrect case/whitespace conversion or an unrecorded source category.",
    )
print_validation_results("Categorical-domain checks", ("VAL-DOMAIN-",))


Categorical-domain checks: 41/41 checks PASS
     check_id                                                                   description                                             observed status                                                                                                evidence                                                                                                                 resolution_or_interpretation
VAL-DOMAIN-01                   orders.sales_channel: values stay in measured source domain  observed_count=3, unexpected_count=0, unexpected=[]   PASS                   T1-P4 measured source domain compared with exact-text submitted orders.sales_channel. Every observed value lies inside the domain measured from the sources; no category was invented, case-folded or split during transformation.
VAL-DOMAIN-02                  orders.payment_method: values stay in measured source domain  observed_count=4, unexpected_count=0, unexpected=[]   PASS         

### Numeric-range validation

The serialized numeric fields are checked against their published semantic ranges, such as valid coordinates, non-negative monetary or operational measures, positive product weight, and review ratings from 1 to 5.

In [51]:
# Sensible ranges required by the HD validation descriptor.
range_contract = [
    ("orders", "order_price", lambda s: s.ge(0), "non-negative"),
    ("orders", "delivery_charges", lambda s: s.ge(0), "non-negative"),
    ("orders", "coupon_discount", lambda s: s.between(0, 100), "0 to 100 percentage points"),
    ("orders", "tax_amount", lambda s: s.ge(0), "non-negative"),
    ("orders", "order_total", lambda s: s.ge(0), "non-negative"),
    ("orders", "customer_lat", lambda s: s.between(-90, 90), "valid latitude"),
    ("orders", "customer_long", lambda s: s.between(-180, 180), "valid longitude"),
    ("order_items", "quantity", lambda s: s.ge(1), "at least one unit"),
    ("order_items", "unit_price", lambda s: s.ge(0), "non-negative"),
    ("order_items", "line_revenue", lambda s: s.ge(0), "non-negative"),
    ("customers", "prior_12m_orders", lambda s: s.ge(0), "non-negative"),
    ("customers", "lifetime_value_before_period", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "delay_days", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "fulfilment_hours", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "delivery_cost", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "promised_days", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "tracking_event_count", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "shipping_distance_km", lambda s: s.ge(0), "non-negative"),
    ("deliveries", "estimated_carbon_kg", lambda s: s.ge(0), "non-negative"),
    ("products", "unit_price", lambda s: s.ge(0), "non-negative"),
    ("products", "unit_cost", lambda s: s.ge(0), "non-negative"),
    ("products", "launch_year", lambda s: s.between(1900, 2100), "plausible catalogue year"),
    ("products", "warranty_months", lambda s: s.ge(0), "non-negative"),
    ("products", "weight_kg", lambda s: s.gt(0), "positive"),
    ("product_reviews", "rating", lambda s: s.between(1, 5), "1 to 5"),
    ("product_reviews", "helpful_votes", lambda s: s.ge(0), "non-negative"),
    ("product_reviews", "review_length_chars", lambda s: s.ge(0), "non-negative"),
    ("product_reviews", "review_word_count", lambda s: s.ge(0), "non-negative"),
]
for range_index, (table, field, predicate, rule) in enumerate(range_contract, start=1):
    values = pd.to_numeric(written_text[table][field], errors="coerce")
    valid = values.notna() & predicate(values)
    task4_check(
        f"VAL-RANGE-{range_index:02d}", f"{table}.{field}: sensible range ({rule})",
        valid.all(), f"min={values.min()}, max={values.max()}, invalid={(~valid).sum()}",
        f"All serialized {field} values parsed numerically and checked against the semantic range.",
        "The field is numerically usable and contains no implausible value under its public meaning.",
        "Trace invalid values to parsing, percentage/currency conversion or source-quality evidence and document treatment.",
    )
print_validation_results("Numeric-range checks", ("VAL-RANGE-",))


Numeric-range checks: 28/28 checks PASS
    check_id                                                           description                                  observed status                                                                                                      evidence                                                                resolution_or_interpretation
VAL-RANGE-01                     orders.order_price: sensible range (non-negative)        min=31.29, max=16345.38, invalid=0   PASS                  All serialized order_price values parsed numerically and checked against the semantic range. The field is numerically usable and contains no implausible value under its public meaning.
VAL-RANGE-02                orders.delivery_charges: sensible range (non-negative)            min=7.02, max=31.38, invalid=0   PASS             All serialized delivery_charges values parsed numerically and checked against the semantic range. The field is numerically usable and contains no i

### Task 1 assumptions covered in 6.1

This section keeps assumptions that directly control schema serialization, categorical domains and missing-value representation: `ASM-04` (coupon absence is reconciled as one missing fact), `ASM-05` (nullable prescribed strings use literal `NaN`), `ASM-06` (`delay_reason='none'` is a category), and `ASM-16` (whole percentage-point discounts retain integer serialization). The remaining assumptions are validated in the section whose business meaning they control: relationships in 6.2, arithmetic in 6.4, temporal ordering in 6.5, and text/multilingual processing in 6.6.

In [52]:
# ASM-01: XML source dates are day-first and serialized as ISO dates in outputs.
xml_date_values = pd.concat([
    xml_deliveries["Dispatch_Date"],
    xml_deliveries["Promised_Date"],
    xml_deliveries["Delivered_Date"],
    xml_products["Launch_Date"],
], ignore_index=True).dropna().astype(str)
xml_date_first_component = pd.to_numeric(xml_date_values.str.slice(0, 2), errors="coerce")
xml_dates_parse_day_first = pd.to_datetime(
    xml_date_values, format="%d/%m/%Y", errors="coerce"
).notna().all()
task4_check(
    "VAL-ASM-01", "XML dates use the measured day-first convention before ISO export",
    xml_dates_parse_day_first and xml_date_first_component.between(1, 31).all(),
    f"xml_values={len(xml_date_values)}, first_component_max={xml_date_first_component.max()}, "
    f"day_first_parse={xml_dates_parse_day_first}",
    "T1-P4 XML date fields are parsed with %d/%m/%Y and compared with the submitted ISO-date fields checked by VAL-TYPE-*.",
    "The XML day-first convention is supported and the published outputs use unambiguous YYYY-MM-DD dates.",
    "Recheck the XML date convention and conversion before accepting temporal validation results.",
)

# ASM-02 and ASM-16: percentage points are preserved and whole observed discounts remain integers.
source_discounts = pd.concat([
    pd.DataFrame({"order_id": json_headers["orderID"], "coupon_discount": percent_series(json_headers["couponDiscount"])}),
    pd.DataFrame({"order_id": xml_headers["Order_ID"], "coupon_discount": percent_series(xml_headers["Coupon_Discount"])}),
], ignore_index=True).drop_duplicates("order_id").set_index("order_id")["coupon_discount"]
written_discounts = written["orders"].set_index("order_id")["coupon_discount"]
discount_differences = written_discounts.sub(source_discounts).abs()
discount_values_match = discount_differences.le(NUMERIC_TOLERANCE).all()
discount_text = written_text["orders"]["coupon_discount"]
discount_numeric = pd.to_numeric(discount_text, errors="coerce")
discount_integer_serialization = discount_text.str.fullmatch(r"[0-9]+(?:\.0)?").all() and discount_numeric.mod(1).eq(0).all()
task4_check(
    "VAL-ASM-02", "coupon_discount is percentage points, not a fraction",
    discount_values_match,
    f"orders_checked={len(written_discounts)}, source_output_max_abs_diff={discount_differences.max():.6f}",
    "JSON numeric and XML percent source values are normalized to percentage points and reconciled to submitted orders by order_id.",
    "The submitted discount values preserve the source percentage-point meaning used by the order-total formula.",
    "Reconcile coupon_discount units and recompute order_total only after the published percentage-point conversion.",
)
task4_check(
    "VAL-ASM-16", "whole observed coupon discounts are serialized without fractional values",
    discount_integer_serialization,
    f"orders_checked={len(discount_text)}, non_integer_values={int((~discount_numeric.mod(1).eq(0)).sum())}",
    "The public dictionary requires exact normalization; submitted discount text is checked for integral numeric values.",
    "Observed whole percentage points remain integer-valued in the submitted CSV rather than becoming fractional stand-ins.",
    "Preserve fractional values only when the source contains a genuine fraction; otherwise serialize whole percentage points as integers.",
)

# ASM-06: the literal category 'none' remains meaningful for on-time, zero-delay deliveries.
delay_none = written["deliveries"]["delay_reason"].eq("none")
delay_none_semantics = (
    written["deliveries"].loc[delay_none, "delay_days"].ge(0).all()
    and written["deliveries"].loc[delay_none, "on_time_in_full"].eq(True).all()
)
task4_check(
    "VAL-ASM-06", "delay_reason='none' remains a category rather than a missing value",
    delay_none.any() and delay_none_semantics,
    f"none_rows={int(delay_none.sum())}, none_with_on_time_true={int(written['deliveries'].loc[delay_none, 'on_time_in_full'].eq(True).sum())}",
    "The measured categorical value is checked in the submitted deliveries table together with its source-supported operational meaning.",
    "The 'none' category is preserved and is not converted to the literal NaN sentinel.",
    "Retain 'none' as a category and investigate any row whose delay semantics disagree with the source evidence.",
)

# ASM-09: language_code must agree with the structured source attribute, not text analysis.
source_languages = pd.concat([
    pd.DataFrame({"review_id": json_reviews["reviewID"], "language_code": json_reviews["languageCode"].astype(str).str.lower()}),
    pd.DataFrame({"review_id": xml_reviews["Review_ID"], "language_code": xml_reviews["Language_Code"].astype(str).str.lower()}),
], ignore_index=True).drop_duplicates("review_id").set_index("review_id")["language_code"]
written_languages = written["product_reviews"].set_index("review_id")["language_code"].astype(str).str.lower()
task4_check(
    "VAL-ASM-09", "language_code agrees with the structured review attribute",
    written_languages.eq(source_languages).all(),
    f"reviews_checked={len(written_languages)}, mismatches={int((~written_languages.eq(source_languages)).sum())}",
    "Submitted language_code values are compared with structured JSON/XML language attributes; no text-derived script label is used.",
    "Language provenance remains structured and independent from contains_non_latin_script or Latin analysis.",
    "Restore language_code from the structured review attribute and do not infer it from narrative text.",
)

# ASM-10: reviews after the order reporting period are retained when their event order is valid.
source_review_timestamps = pd.concat([
    pd.to_datetime(json_reviews["reviewTimestamp"], format="%Y-%m-%d %H:%M:%S"),
    pd.to_datetime(xml_reviews["Review_Timestamp"], format="%d/%m/%Y %H:%M:%S"),
], ignore_index=True)
written_review_timestamps = pd.to_datetime(written["product_reviews"]["review_timestamp"], format="%Y-%m-%d %H:%M:%S")
late_reviews_retained = written_review_timestamps.max() == source_review_timestamps.max()
task4_check(
    "VAL-ASM-10", "valid reviews after 2018-12-31 are retained",
    late_reviews_retained and written_review_timestamps.max() > pd.Timestamp("2018-12-31"),
    f"source_max={source_review_timestamps.max()}, output_max={written_review_timestamps.max()}, "
    f"late_output_rows={int(written_review_timestamps.gt(pd.Timestamp('2018-12-31')).sum())}",
    "The maximum structured review timestamp is compared before and after transformation; temporal checks separately verify review event ordering.",
    "Late but valid reviews remain in the canonical output rather than being truncated to the order reporting period.",
    "Retain reviews whose delivery/order sequence is valid and investigate only genuine timestamp anomalies.",
)

# ASM-11: historical order-item prices come from order-item sources, not catalogue prices.
source_item_prices = pd.concat([
    pd.DataFrame({"order_item_id": NORMALISED[("order_items", "JSON")]["order_item_id"], "unit_price": NORMALISED[("order_items", "JSON")]["unit_price"]}),
    pd.DataFrame({"order_item_id": NORMALISED[("order_items", "XML")]["order_item_id"], "unit_price": NORMALISED[("order_items", "XML")]["unit_price"]}),
], ignore_index=True).drop_duplicates("order_item_id").set_index("order_item_id")["unit_price"]
written_item_prices = written["order_items"].set_index("order_item_id")["unit_price"]
item_price_differences = written_item_prices.sub(source_item_prices).abs()
task4_check(
    "VAL-ASM-11", "historical order-item prices are preserved independently of catalogue prices",
    item_price_differences.le(MONEY_TOLERANCE).all(),
    f"items_checked={len(written_item_prices)}, max_abs_diff={item_price_differences.max():.6f}",
    "Submitted order_items.unit_price is reconciled against source order-item prices; products.unit_price is not used in this comparison.",
    "Historical line pricing is preserved and catalogue current-state prices do not overwrite order-event values.",
    "Rebuild order_items.unit_price from the historical item source and keep catalogue prices in products only.",
)

# ASM-12: inactive catalogue products remain available to satisfy product references.
inactive_source_ids = set(xml_products.loc[xml_products["Active_Flag"].eq("N"), "Product_ID"])
inactive_output_ids = set(written["products"].loc[~written["products"]["active_flag"], "product_id"])
task4_check(
    "VAL-ASM-12", "inactive catalogue products are retained",
    inactive_source_ids <= inactive_output_ids,
    f"inactive_source={len(inactive_source_ids)}, inactive_output={len(inactive_output_ids)}, missing_inactive={len(inactive_source_ids - inactive_output_ids)}",
    "Inactive ProductCatalogue IDs are compared with submitted products.active_flag and product_id values.",
    "Inactive products remain in the catalogue and can continue to resolve order-item and review foreign keys.",
    "Retain inactive products; do not filter the catalogue by active_flag before publishing the products table.",
)

In [53]:
print_validation_results("Assumption checks", ("VAL-ASM-",))


Assumption checks: 8/8 checks PASS
  check_id                                                                  description                                                                             observed status                                                                                                                                      evidence                                                                                           resolution_or_interpretation
VAL-ASM-01            XML dates use the measured day-first convention before ISO export                        xml_values=9454, first_component_max=31, day_first_parse=True   PASS                         T1-P4 XML date fields are parsed with %d/%m/%Y and compared with the submitted ISO-date fields checked by VAL-TYPE-*.                  The XML day-first convention is supported and the published outputs use unambiguous YYYY-MM-DD dates.
VAL-ASM-02                         coupon_discount is percentage points, not a fra

In [54]:
print_validation_results(
    "6.1 Schema and type checks",
    ("VAL-FILE-", "VAL-SCHEMA-", "VAL-MISSING-", "VAL-TYPE-", "VAL-DOMAIN-", "VAL-RANGE-", "VAL-ASM-"),
)


6.1 Schema and type checks: 96/96 checks PASS
      check_id                                                                          description                                                                                                                                                                                                                                        observed status                                                                                                                                      evidence                                                                                                                 resolution_or_interpretation
   VAL-FILE-01                                                    exactly six required output files ['Group005_customers_standardised.csv', 'Group005_deliveries_standardised.csv', 'Group005_order_items_standardised.csv', 'Group005_orders_standardised.csv', 'Group005_product_reviews_standardised.csv', 'Group005_products_st

### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)

Validation conditions:

- Primary-key check passes only when the primary-key column contains no pandas missing value, no empty string, no literal `NaN`, and every key is unique.
- Foreign-key check passes only when every child key is present and every non-missing child value matches a key in the referenced parent table; missing or unmatched values fail the check.
- Delivery-grain check passes only when the set of submitted delivery order IDs is exactly the set of completed order IDs and each delivery order ID is unique.
- Review-to-order-item relationship check passes only when each review's referenced order item exists and its `order_id` and `product_id` equal the corresponding order-item values.
- Review-to-order relationship check passes only when each review's `customer_id` equals the `customer_id` of its referenced order.
- These checks validate the relationship assumptions associated with `ASM-08` and `ASM-12`.

**Assumptions validated in this section:** `ASM-08` delivery grain is restricted to completed orders; `ASM-12` inactive products remain available for foreign-key resolution. The primary/foreign-key checks below validate these relationship and retention assumptions against submitted tables.

In [55]:
fk_contract = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("deliveries", "order_id", "orders", "order_id"),
    ("product_reviews", "order_id", "orders", "order_id"),
    ("product_reviews", "order_item_id", "order_items", "order_item_id"),
    ("product_reviews", "product_id", "products", "product_id"),
    ("product_reviews", "customer_id", "customers", "customer_id"),
]
for table_index, (table, frame) in enumerate(written_text.items(), start=1):
    key = TASK2_PRIMARY_KEYS[table]
    pk_missing = frame[key].isna() | frame[key].eq("") | frame[key].eq(MISSING)
    pk_ok = not pk_missing.any() and frame[key].is_unique
    pk_observed = (
        f"missing_values={int(pk_missing.sum())}, distinct_keys={frame[key].nunique()}, "
        f"duplicate_keys={int(frame[key].duplicated().sum())}"
    )
    task4_check(
        f"VAL-PK-{table_index:02d}", f"{table}: primary key complete and unique",
        pk_ok, pk_observed,
        f"Exact-text {key} values in the submitted CSV; grain from public_data_dictionary.csv.",
        "No primary-key value is missing and every primary key is unique at the required grain.",
        "Trace missing, literal-NaN or repeated keys to within-source duplication or incorrect flattening, then reconcile by the published key.",
    )

for fk_index, (child_table, child_field, parent_table, parent_field) in enumerate(fk_contract, start=1):
    child_values = written_text[child_table][child_field]
    parent_values = set(written_text[parent_table][parent_field])
    missing_count = int(child_values.isna().sum() + child_values.eq("").sum() + child_values.eq(MISSING).sum())
    unmatched_mask = child_values.ne("") & child_values.ne(MISSING) & ~child_values.isin(parent_values)
    unmatched_values = set(child_values[unmatched_mask])
    task4_check(
        f"VAL-FK-{fk_index:02d}",
        f"{child_table}.{child_field} -> {parent_table}.{parent_field}",
        missing_count == 0 and not unmatched_values,
        f"missing_values={missing_count}, unmatched_values={len(unmatched_values)}, "
        f"unmatched_rows={int(unmatched_mask.sum())}",
        f"Exact submitted {child_field} values anti-joined to {parent_table}.{parent_field}.",
        "Every required child key is present and resolves to one published parent entity.",
        "Trace missing, literal-NaN or unmatched keys to a missing parent, incorrect identifier normalisation or a broken flattening relationship.",
    )

In [56]:
print_validation_results("Primary-key checks", ("VAL-PK-",))
print_validation_results("Foreign-key checks", ("VAL-FK-",))


Primary-key checks: 6/6 checks PASS
 check_id                                      description                                                observed status                                                                                     evidence                                                           resolution_or_interpretation
VAL-PK-01          orders: primary key complete and unique  missing_values=0, distinct_keys=5000, duplicate_keys=0   PASS      Exact-text order_id values in the submitted CSV; grain from public_data_dictionary.csv. No primary-key value is missing and every primary key is unique at the required grain.
VAL-PK-02     order_items: primary key complete and unique missing_values=0, distinct_keys=15739, duplicate_keys=0   PASS Exact-text order_item_id values in the submitted CSV; grain from public_data_dictionary.csv. No primary-key value is missing and every primary key is unique at the required grain.
VAL-PK-03       customers: primary key complete and uniqu

### Delivery-grain validation

In [57]:
completed_written = set(written["orders"].loc[
    written["orders"].order_status == "Completed", "order_id"
])
delivery_order_ids = set(written["deliveries"].order_id)
task4_check(
    "VAL-GRAIN-01", "one delivery per completed order",
    delivery_order_ids == completed_written and written["deliveries"].order_id.is_unique,
    f"completed_orders={len(completed_written)}, delivery_orders={len(delivery_order_ids)}",
    "Submitted orders filtered on the published Completed category and compared with unique deliveries.order_id.",
    "The delivery table has exactly the required completed-order grain.",
    "Filter canonical deliveries by completed order IDs and reconcile duplicate delivery IDs.",
)

In [58]:
print_validation_results("Delivery-grain checks", ("VAL-GRAIN-",))


Delivery-grain checks: 1/1 checks PASS
    check_id                      description                                    observed status                                                                                                    evidence                                       resolution_or_interpretation
VAL-GRAIN-01 one delivery per completed order completed_orders=5000, delivery_orders=5000   PASS Submitted orders filtered on the published Completed category and compared with unique deliveries.order_id. The delivery table has exactly the required completed-order grain.


### Review-to-order-item relationship validation

In [59]:
review_item_merge_error = None
try:
    review_item = written["product_reviews"].merge(
        written["order_items"][["order_item_id", "order_id", "product_id"]],
        on="order_item_id", how="left", suffixes=("", "_item"),
        validate="many_to_one", indicator=True,
    )
except (pd.errors.MergeError, KeyError) as exc:
    review_item = pd.DataFrame()
    review_item_merge_error = f"{type(exc).__name__}: {exc}"
review_item_unmatched = (
    int(review_item["_merge"].ne("both").sum()) if review_item_merge_error is None
    else len(written["product_reviews"])
)
review_links_ok = (
    review_item_merge_error is None
    and len(review_item) == len(written["product_reviews"])
    and review_item_unmatched == 0
    and review_item["order_id"].eq(review_item["order_id_item"]).all()
    and review_item["product_id"].eq(review_item["product_id_item"]).all()
)
task4_check(
    "VAL-REL-01", "review order/product agree with referenced order item",
    review_links_ok,
    f"rows_checked={len(review_item)}, unmatched={review_item_unmatched}, merge_error={review_item_merge_error or 'none'}",
    "Left many-to-one merge on order_item_id, complete row coverage, then order_id and product_id equality.",
    "The review-to-item one-to-many relationship preserves the item's order and product.",
    "Correct the review foreign keys from the structured source evidence; do not infer them from narrative text.",
)

In [60]:
print_validation_results("Review-item relationship checks", ("VAL-REL-01",))


Review-item relationship checks: 1/1 checks PASS
  check_id                                           description                                         observed status                                                                                               evidence                                                        resolution_or_interpretation
VAL-REL-01 review order/product agree with referenced order item rows_checked=7000, unmatched=0, merge_error=none   PASS Left many-to-one merge on order_item_id, complete row coverage, then order_id and product_id equality. The review-to-item one-to-many relationship preserves the item's order and product.


### Review-to-order customer validation

In [61]:
review_order_merge_error = None
try:
    review_order = written["product_reviews"].merge(
        written["orders"][["order_id", "customer_id"]],
        on="order_id", how="left", suffixes=("", "_order"),
        validate="many_to_one", indicator=True,
    )
except (pd.errors.MergeError, KeyError) as exc:
    review_order = pd.DataFrame()
    review_order_merge_error = f"{type(exc).__name__}: {exc}"
review_order_unmatched = (
    int(review_order["_merge"].ne("both").sum()) if review_order_merge_error is None
    else len(written["product_reviews"])
)
review_customer_ok = (
    review_order_merge_error is None
    and len(review_order) == len(written["product_reviews"])
    and review_order_unmatched == 0
    and review_order["customer_id"].eq(review_order["customer_id_order"]).all()
)
task4_check(
    "VAL-REL-02", "review customer agrees with referenced order",
    review_customer_ok,
    f"rows_checked={len(review_order)}, unmatched={review_order_unmatched}, merge_error={review_order_merge_error or 'none'}",
    "Left many-to-one merge on order_id, complete row coverage, then customer_id equality.",
    "Every review belongs to the same customer as its structured order.",
    "Trace mismatches to review/order reconciliation rather than overwriting either key.",
)

In [62]:
print_validation_results("Review-order relationship checks", ("VAL-REL-02",))


Review-order relationship checks: 1/1 checks PASS
  check_id                                  description                                         observed status                                                                              evidence                                       resolution_or_interpretation
VAL-REL-02 review customer agrees with referenced order rows_checked=7000, unmatched=0, merge_error=none   PASS Left many-to-one merge on order_id, complete row coverage, then customer_id equality. Every review belongs to the same customer as its structured order.


In [63]:
print_validation_results(
    "6.2 Primary- and foreign-key checks",
    ("VAL-PK-", "VAL-FK-", "VAL-GRAIN-", "VAL-REL-"),
)


6.2 Primary- and foreign-key checks: 17/17 checks PASS
    check_id                                                description                                                observed status                                                                                                    evidence                                                           resolution_or_interpretation
   VAL-PK-01                    orders: primary key complete and unique  missing_values=0, distinct_keys=5000, duplicate_keys=0   PASS                     Exact-text order_id values in the submitted CSV; grain from public_data_dictionary.csv. No primary-key value is missing and every primary key is unique at the required grain.
   VAL-PK-02               order_items: primary key complete and unique missing_values=0, distinct_keys=15739, duplicate_keys=0   PASS                Exact-text order_item_id values in the submitted CSV; grain from public_data_dictionary.csv. No primary-key value is missing and every

### 6.3 Source coverage and reconciliation checks (`VAL-FLOW-...`)

Validation conditions:

- Source-flow check passes only when the submitted primary-key set exactly equals the canonical key set derived from the parsed sources and the submitted row count equals that key-set size.
- The observed source row counts, distinct-key counts, shared-key counts and submitted output count are reported as evidence; no fixed expected row count is used as the sole pass condition.
- Conflict check passes only when the reconciliation conflict profile contains no unresolved field-level disagreement for a shared key.
- A conflict is reported rather than overwritten by JSON/XML source precedence, supporting `ASM-07`.

In [64]:
expected_key_sets = {table: canonical_keys(table) for table in TASK2_PRIMARY_KEYS}
expected_key_sets["deliveries"] = set(
    canonical_deliveries_source.loc[
        canonical_deliveries_source.order_id.isin(completed_order_ids), "delivery_id"
    ]
)
for flow_index, (table, frame) in enumerate(written_text.items(), start=1):
    observed_keys = set(frame[TASK2_PRIMARY_KEYS[table]])
    expected_keys = expected_key_sets[table]
    flow = overlap_profile.loc[overlap_profile.target_table == table].iloc[0]
    observed = (
        f"JSON rows/keys={flow.json_records}/{flow.json_distinct_keys}; "
        f"XML rows/keys={flow.xml_records}/{flow.xml_distinct_keys}; "
        f"shared={flow.keys_in_both_sources}; output={len(frame)}; "
        f"keys_match={observed_keys == expected_keys}; row_count_match={len(frame) == len(expected_keys)}"
    )
    task4_check(
        f"VAL-FLOW-{flow_index:02d}", f"{table}: source/overlap row flow and canonical key coverage",
        observed_keys == expected_keys and len(frame) == len(expected_keys), observed,
        "T1-P3/T1-P6 measured within-source repeats and cross-source shared keys; T2-R1 canonical output keys.",
        "All source keys survive exactly once after data-driven reconciliation at the target grain.",
        "Inspect the duplicate/overlap profile and trace missing or extra keys through the canonical-key union.",
    )

In [65]:
print_validation_results("Source-flow checks", ("VAL-FLOW-",))


Source-flow checks: 6/6 checks PASS
   check_id                                                         description                                                                                                            observed status                                                                                              evidence                                                               resolution_or_interpretation
VAL-FLOW-01          orders: source/overlap row flow and canonical key coverage   JSON rows/keys=2818/2750; XML rows/keys=2818/2750; shared=500; output=5000; keys_match=True; row_count_match=True   PASS T1-P3/T1-P6 measured within-source repeats and cross-source shared keys; T2-R1 canonical output keys. All source keys survive exactly once after data-driven reconciliation at the target grain.
VAL-FLOW-02     order_items: source/overlap row flow and canonical key coverage JSON rows/keys=8947/8723; XML rows/keys=8803/8618; shared=1602; output=15739; keys_match=Tr

### Cross-source conflict validation

In [66]:
task4_check(
    "VAL-CONFLICT-01", "no unresolved field-level conflict for a shared key",
    task2_conflicts.empty,
    f"conflicts={len(task2_conflicts)}, profile={os.path.basename(task2_conflict_path)}",
    "T2-R1 compares normalised/derived non-missing values by stable primary key and writes every disagreement to the profile.",
    "No arbitrary source precedence was needed; all shared values agree after published normalisation.",
    "Leave disagreeing fields unresolved, inspect both source values and document a justified resolution before submission.",
)

In [67]:
print_validation_results("Conflict checks", ("VAL-CONFLICT-",))


Conflict checks: 1/1 checks PASS
       check_id                                         description                                                      observed status                                                                                                                 evidence                                                                      resolution_or_interpretation
VAL-CONFLICT-01 no unresolved field-level conflict for a shared key conflicts=0, profile=Group005_T2_reconciliation_conflicts.csv   PASS T2-R1 compares normalised/derived non-missing values by stable primary key and writes every disagreement to the profile. No arbitrary source precedence was needed; all shared values agree after published normalisation.


In [68]:
print_validation_results(
    "6.3 Source coverage and reconciliation checks",
    ("VAL-FLOW-", "VAL-CONFLICT-"),
)


6.3 Source coverage and reconciliation checks: 7/7 checks PASS
       check_id                                                         description                                                                                                            observed status                                                                                                                 evidence                                                                      resolution_or_interpretation
    VAL-FLOW-01          orders: source/overlap row flow and canonical key coverage   JSON rows/keys=2818/2750; XML rows/keys=2818/2750; shared=500; output=5000; keys_match=True; row_count_match=True   PASS                    T1-P3/T1-P6 measured within-source repeats and cross-source shared keys; T2-R1 canonical output keys.        All source keys survive exactly once after data-driven reconciliation at the target grain.
    VAL-FLOW-02     order_items: source/overlap row flow and canonical key coverage 

### 6.4 Arithmetic checks (`VAL-ARITH-...`)

Validation conditions:

- Line-revenue check passes only when every submitted `line_revenue` is within `MONEY_TOLERANCE` of `round(quantity * unit_price, 2)`.
- Order-price check passes only when every submitted `order_price` is within `MONEY_TOLERANCE` of the independently recomputed sum of rounded canonical line revenues.
- Included-GST check passes only when every submitted `tax_amount` is within `MONEY_TOLERANCE` of `round(order_price / 11, 2)`.
- Order-total check passes only when every submitted `order_total` is within `MONEY_TOLERANCE` of `round(order_price * (1 - coupon_discount / 100) + delivery_charges, 2)`.
- Tax-separation check passes only when every submitted `order_total` differs from the deliberately incorrect tax-added formula by more than `MONEY_TOLERANCE`.
- These checks validate the monetary assumptions associated with `ASM-02`, `ASM-03`, `ASM-11` and `ASM-14`.

**Assumptions validated in this section:** `ASM-02` coupon discounts are percentage points, `ASM-03` tax is included GST, `ASM-11` historical item prices remain separate from catalogue prices, and `ASM-14` uses the published Python cent-rounding rule. The arithmetic checks below independently recompute the submitted monetary fields.

In [69]:
line_expected = round2(written["order_items"]["quantity"] * written["order_items"]["unit_price"])
line_diff = (line_expected - written["order_items"]["line_revenue"]).abs()
task4_check(
    "VAL-ARITH-01", "line_revenue = round(quantity * unit_price, 2)",
    line_diff.le(MONEY_TOLERANCE).all(), f"max_abs_diff={line_diff.max():.6f}",
    "All submitted order-item rows; Python cent rounding and absolute tolerance 0.01.",
    "Every line follows published arithmetic step 1.",
    "Recalculate line_revenue from quantity and unit_price before aggregating orders.",
)

In [70]:
print_validation_results("Arithmetic line-revenue checks", ("VAL-ARITH-01",))


Arithmetic line-revenue checks: 1/1 checks PASS
    check_id                                    description              observed status                                                                         evidence                    resolution_or_interpretation
VAL-ARITH-01 line_revenue = round(quantity * unit_price, 2) max_abs_diff=0.000000   PASS All submitted order-item rows; Python cent rounding and absolute tolerance 0.01. Every line follows published arithmetic step 1.


### Order-item line-revenue validation

In [71]:
price_expected = round2(line_expected.groupby(written["order_items"]["order_id"]).sum())
order_price_diff = (written["orders"].set_index("order_id")["order_price"] - price_expected).abs()
task4_check(
    "VAL-ARITH-02", "order_price = sum of independently recomputed rounded line revenue",
    order_price_diff.le(MONEY_TOLERANCE).all(),
    f"orders_checked={len(order_price_diff)}, max_abs_diff={order_price_diff.max():.6f}, "
    f"outside_tolerance={int((order_price_diff > MONEY_TOLERANCE).sum())}",
    "Quantity and unit_price are recomputed at order-item grain, rounded to cents, then grouped by order_id; tolerance 0.01.",
    "Every order follows published arithmetic step 2 without validating against the submitted line_revenue column.",
    "Reconcile order-item duplicates before summing the rounded line revenues.",
)

In [72]:
print_validation_results("Arithmetic order-price checks", ("VAL-ARITH-02",))


Arithmetic order-price checks: 1/1 checks PASS
    check_id                                                        description                                                        observed status                                                                                                                evidence                                                                                  resolution_or_interpretation
VAL-ARITH-02 order_price = sum of independently recomputed rounded line revenue orders_checked=5000, max_abs_diff=0.000000, outside_tolerance=0   PASS Quantity and unit_price are recomputed at order-item grain, rounded to cents, then grouped by order_id; tolerance 0.01. Every order follows published arithmetic step 2 without validating against the submitted line_revenue column.


### Order-price aggregation validation

Summed over de-duplicated items: `T1-P4` measured that summing raw lines matches only 2,682 of 2,750 orders.

In [73]:
tax_expected = round2(written["orders"]["order_price"] / 11)
tax_diff = (tax_expected - written["orders"]["tax_amount"]).abs()
task4_check(
    "VAL-ARITH-03", "tax_amount is included GST order_price / 11 before discount",
    tax_diff.le(MONEY_TOLERANCE).all(), f"max_abs_diff={tax_diff.max():.6f}",
    "All canonical orders; included-GST formula rounded to cents with tolerance 0.01.",
    "Tax is reported separately and has not been recomputed after discount.",
    "Recalculate tax_amount from undiscounted order_price and do not add it to order_total.",
)

In [74]:
print_validation_results("Arithmetic GST checks", ("VAL-ARITH-03",))


Arithmetic GST checks: 1/1 checks PASS
    check_id                                                 description              observed status                                                                         evidence                                           resolution_or_interpretation
VAL-ARITH-03 tax_amount is included GST order_price / 11 before discount max_abs_diff=0.000000   PASS All canonical orders; included-GST formula rounded to cents with tolerance 0.01. Tax is reported separately and has not been recomputed after discount.


### Included-GST validation

In [75]:
total_expected = round2(
    written["orders"]["order_price"] * (1 - written["orders"]["coupon_discount"] / 100)
    + written["orders"]["delivery_charges"]
)
total_diff = (total_expected - written["orders"]["order_total"]).abs()
task4_check(
    "VAL-ARITH-04", "order_total follows discount then delivery sequence",
    total_diff.le(MONEY_TOLERANCE).all(), f"max_abs_diff={total_diff.max():.6f}",
    "order_price * (1 - coupon_discount/100) + delivery_charges, then cent rounding; tolerance 0.01.",
    "Published arithmetic steps 4-6 are satisfied and tax was not added again.",
    "Apply the discount to order_price, add delivery_charges, then round once to cents.",
)

In [76]:
print_validation_results("Arithmetic order-total checks", ("VAL-ARITH-04",))


Arithmetic order-total checks: 1/1 checks PASS
    check_id                                         description              observed status                                                                                        evidence                                              resolution_or_interpretation
VAL-ARITH-04 order_total follows discount then delivery sequence max_abs_diff=0.000000   PASS order_price * (1 - coupon_discount/100) + delivery_charges, then cent rounding; tolerance 0.01. Published arithmetic steps 4-6 are satisfied and tax was not added again.


### Discounted-order-total validation

In [77]:
tax_added_total = round2(total_expected + tax_expected)
tax_not_added_again = (written["orders"]["order_total"] - tax_added_total).abs()
task4_check(
    "VAL-ARITH-05", "tax_amount is reported separately and not added again to order_total",
    (tax_not_added_again > MONEY_TOLERANCE).all(),
    f"orders_checked={len(tax_not_added_again)}, orders_matching_tax_added_formula="
    f"{int((tax_not_added_again <= MONEY_TOLERANCE).sum())}, min_difference={tax_not_added_again.min():.6f}",
    "Submitted order_total is compared with the deliberately incorrect formula that adds included GST a second time.",
    "The published total excludes a second GST addition; tax_amount remains a separate included-GST field.",
    "Recalculate order_total from discounted order_price plus delivery_charges; do not add tax_amount again.",
)

In [78]:
print_validation_results("Arithmetic tax-separation checks", ("VAL-ARITH-05",))


Arithmetic tax-separation checks: 1/1 checks PASS
    check_id                                                          description                                                                          observed status                                                                                                        evidence                                                                          resolution_or_interpretation
VAL-ARITH-05 tax_amount is reported separately and not added again to order_total orders_checked=5000, orders_matching_tax_added_formula=0, min_difference=2.840000   PASS Submitted order_total is compared with the deliberately incorrect formula that adds included GST a second time. The published total excludes a second GST addition; tax_amount remains a separate included-GST field.


In [79]:
print_validation_results(
    "6.4 Arithmetic checks",
    ("VAL-ARITH-",),
)


6.4 Arithmetic checks: 5/5 checks PASS
    check_id                                                          description                                                                          observed status                                                                                                                evidence                                                                                  resolution_or_interpretation
VAL-ARITH-01                       line_revenue = round(quantity * unit_price, 2)                                                             max_abs_diff=0.000000   PASS                                        All submitted order-item rows; Python cent rounding and absolute tolerance 0.01.                                                               Every line follows published arithmetic step 1.
VAL-ARITH-02   order_price = sum of independently recomputed rounded line revenue                   orders_checked=5000, max_abs_diff=0.000000, outside_tolera

### Temporal-order validation

### 6.5 Temporal checks (`VAL-TIME-...`)

Validation conditions:

- Order-to-dispatch check passes only when no submitted order has an order date later than its dispatch date.
- Order-to-promised check passes only when no submitted order has an order date later than its promised date.
- Dispatch-to-promised check passes only when no submitted delivery has a dispatch date later than its promised date.
- Dispatch-to-delivered check passes only when no submitted delivery has a dispatch date later than its delivered date.
- Delivery-to-review check passes only when no review timestamp occurs before the associated delivered date.
- Order-to-review check passes only when no review timestamp occurs before the associated order timestamp.
- These checks validate the temporal assumptions associated with `ASM-01` and `ASM-10`.

**Assumptions validated in this section:** `ASM-01` XML dates use day-first parsing and `ASM-10` valid reviews after the order reporting period are retained. The temporal checks below validate the resulting event ordering and retention behavior.

In [80]:
# Temporal checks record merge/cardinality failures instead of terminating the notebook.
order_delivery_merge_error = None
try:
    order_delivery_time = written_text["deliveries"][[
        "order_id", "dispatch_date", "promised_date", "delivered_date",
    ]].merge(
        written_text["orders"][["order_id", "order_timestamp"]],
        on="order_id", how="left", validate="one_to_one", indicator=True,
    )
except (pd.errors.MergeError, KeyError) as exc:
    order_delivery_time = pd.DataFrame()
    order_delivery_merge_error = f"{type(exc).__name__}: {exc}"

temporal_definitions = [
    ("VAL-TIME-01", "order date <= dispatch date", "order_timestamp", "dispatch_date", True),
    ("VAL-TIME-02", "order date <= promised date", "order_timestamp", "promised_date", True),
    ("VAL-TIME-03", "dispatch date <= promised date", "dispatch_date", "promised_date", False),
    ("VAL-TIME-04", "dispatch date <= delivered date", "dispatch_date", "delivered_date", False),
]
if order_delivery_merge_error is not None:
    for check_id, description, _, _, _ in temporal_definitions:
        task4_check(
            check_id, description, False, f"merge_error={order_delivery_merge_error}",
            "Submitted deliveries left-joined to orders by order_id with one-to-one cardinality validation.",
            "The operational event sequence is temporally valid.",
            "Repair the missing columns or relationship cardinality, then inspect the affected dates.",
        )
else:
    order_delivery_unmatched = int(order_delivery_time["_merge"].ne("both").sum())
    for column in ("order_timestamp", "dispatch_date", "promised_date", "delivered_date"):
        order_delivery_time[column] = pd.to_datetime(order_delivery_time[column], errors="coerce")
    for check_id, description, earlier_column, later_column, normalize_earlier in temporal_definitions:
        earlier = order_delivery_time[earlier_column]
        if normalize_earlier:
            earlier = earlier.dt.normalize()
        later = order_delivery_time[later_column]
        invalid = earlier.isna() | later.isna()
        violations = (~invalid) & earlier.gt(later)
        task4_check(
            check_id, description,
            order_delivery_unmatched == 0 and not invalid.any() and not violations.any(),
            f"rows={len(violations)}, unmatched={order_delivery_unmatched}, invalid={int(invalid.sum())}, violations={int(violations.sum())}",
            "Submitted deliveries left-joined to orders by order_id, with cardinality, coverage and parsed timestamps checked.",
            "The operational event sequence is temporally valid.",
            "Inspect relationship coverage, source date parsing and the affected event sequence.",
        )

def validate_review_time(check_id, description, right_table, right_field, comparison):
    merge_error = None
    try:
        joined = written_text["product_reviews"][["review_id", "order_id", "review_timestamp"]].merge(
            written_text[right_table][["order_id", right_field]],
            on="order_id", how="left", validate="many_to_one", indicator=True,
        )
    except (pd.errors.MergeError, KeyError) as exc:
        joined = pd.DataFrame()
        merge_error = f"{type(exc).__name__}: {exc}"
    if merge_error is not None:
        task4_check(
            check_id, description, False, f"merge_error={merge_error}",
            f"Submitted reviews left-joined to {right_table} by order_id with many-to-one cardinality validation.",
            "The review event sequence is temporally valid.",
            "Repair the missing columns or relationship cardinality, then inspect the affected timestamps.",
        )
        return
    unmatched = int(joined["_merge"].ne("both").sum())
    review_timestamp = pd.to_datetime(joined["review_timestamp"], errors="coerce")
    related_timestamp = pd.to_datetime(joined[right_field], errors="coerce")
    invalid = review_timestamp.isna() | related_timestamp.isna()
    violations = (~invalid) & comparison(review_timestamp, related_timestamp)
    task4_check(
        check_id, description, unmatched == 0 and not invalid.any() and not violations.any(),
        f"rows={len(joined)}, unmatched={unmatched}, invalid={int(invalid.sum())}, violations={int(violations.sum())}",
        f"Submitted reviews left-joined to {right_table} by order_id, with cardinality, coverage and parsed timestamps checked.",
        "The review event sequence is temporally valid.",
        "Inspect relationship coverage, source timestamp parsing and the affected event sequence.",
    )

validate_review_time(
    "VAL-TIME-05", "delivered date <= review timestamp", "deliveries", "delivered_date",
    lambda review, delivered: review.dt.normalize().lt(delivered),
)
validate_review_time(
    "VAL-TIME-06", "order timestamp <= review timestamp", "orders", "order_timestamp",
    lambda review, ordered: review.lt(ordered),
)

In [81]:
print_validation_results(
    "6.5 Temporal checks",
    ("VAL-TIME-",),
)


6.5 Temporal checks: 6/6 checks PASS
   check_id                         description                                        observed status                                                                                                           evidence                        resolution_or_interpretation
VAL-TIME-01         order date <= dispatch date rows=5000, unmatched=0, invalid=0, violations=0   PASS  Submitted deliveries left-joined to orders by order_id, with cardinality, coverage and parsed timestamps checked. The operational event sequence is temporally valid.
VAL-TIME-02         order date <= promised date rows=5000, unmatched=0, invalid=0, violations=0   PASS  Submitted deliveries left-joined to orders by order_id, with cardinality, coverage and parsed timestamps checked. The operational event sequence is temporally valid.
VAL-TIME-03      dispatch date <= promised date rows=5000, unmatched=0, invalid=0, violations=0   PASS  Submitted deliveries left-joined to orders by or

### 6.6 Text and multilingual checks (`VAL-TEXT-...`)

Validation conditions:

- Review SKU check passes only when the extracted review SKU equals the SKU of the referenced product.
- Review order-reference check passes only when the extracted review order reference equals the structured `order_id`.
- Promotion check passes only when the submitted promotion code equals the code extracted from the raw structured customer note.
- Cleaned-narrative checks pass only when no prohibited marker, tag, URL, emoji/symbol residue or invalid case remains in the submitted cleaned field.
- Review-measure check passes only when character and word counts are recomputed from `review_body_clean`, with literal `NaN` treated as zero rather than as ordinary text.
- Multilingual check passes only when the submitted non-Latin indicator agrees with Unicode-based recomputation and the Latin-analysis output contains no non-Latin letters.
- Reference-format checks pass only when populated extracted references and promotion codes match their published ASCII formats and literal `NaN` is retained when absent.
- These checks validate the text assumptions associated with `ASM-09`, `ASM-13` and `ASM-15`.

**Assumptions validated in this section:** `ASM-09` language provenance comes from structured review attributes, `ASM-13` literal-`NaN` review measures use zero/zero/False behavior, and `ASM-15` separates multilingual preservation from Latin-only analysis. The text checks below validate these contracts.

### Narrative extraction and multilingual validation

Extraction runs on the **raw** parser value, before cleaning: a reference removed by the cleaner could not otherwise be recovered.

In [82]:
# Validation-local Unicode rules provide an independent cross-check of Task 3 output.
VALIDATION_EMOJI_RANGES = (
    (0x1F000, 0x1FAFF), (0x2600, 0x27BF), (0x2B00, 0x2BFF),
    (0x231A, 0x231B), (0x23E9, 0x23FA), (0x25FD, 0x25FE),
    (0xFE00, 0xFE0F), (0x200D, 0x200D),
)

def validation_is_emoji_character(char):
    codepoint = ord(char)
    return codepoint == 0x20E3 or any(low <= codepoint <= high for low, high in VALIDATION_EMOJI_RANGES)

# Unicode Script=Latin letters whose name does not begin with "LATIN ". Restated here
# rather than imported, so this check stays independent of the submitted module while
# still using the script property instead of a name substring. Verified against
# Unicode Scripts.txt: zero disagreements over every code point.
VALIDATION_LATIN_SCRIPT_RANGES = (
    (0x00AA, 0x00AA), (0x00BA, 0x00BA), (0x02B0, 0x02B8), (0x02E0, 0x02E4),
    (0x1D2C, 0x1D5C), (0x1D9B, 0x1DBE), (0x2071, 0x2071), (0x207F, 0x207F),
    (0x212A, 0x212B), (0x2132, 0x2132), (0x214E, 0x214E), (0x2183, 0x2183),
    (0x2C7D, 0x2C7D), (0xA770, 0xA770), (0xA7F2, 0xA7F4), (0xA7F8, 0xA7F9),
    (0xAB5C, 0xAB5F), (0xAB69, 0xAB69), (0xFF21, 0xFF3A), (0xFF41, 0xFF5A),
    (0x10780, 0x10785), (0x10787, 0x107B0), (0x107B2, 0x107BA),
)


def validation_is_latin_letter(char):
    """True when char is a letter whose Unicode Script is Latin."""
    if not unicodedata.category(char).startswith("L"):
        return False
    if unicodedata.name(char, "").startswith("LATIN "):
        return True
    code_point = ord(char)
    return any(low <= code_point <= high for low, high in VALIDATION_LATIN_SCRIPT_RANGES)


def validation_contains_non_latin_script(value):
    if value == MISSING:
        return False
    return any(
        unicodedata.category(char).startswith("L")
        and not validation_is_latin_letter(char)
        for char in unicodedata.normalize("NFC", str(value))
    )

sku_by_product = written["products"].set_index("product_id")["product_sku"]
expected_review_sku = written["product_reviews"]["product_id"].map(sku_by_product)
task4_check(
    "VAL-TEXT-01", "extracted review SKU agrees with referenced product",
    written["product_reviews"]["extracted_product_sku"].eq(expected_review_sku).all(),
    f"rows_checked={len(expected_review_sku)}",
    "Raw parser-obtained review text extracted before cleaning, then compared through product_id to products.product_sku.",
    "Every extracted SKU agrees with the structured product relationship.",
    "Inspect raw wrapper boundaries and the product foreign key; never repair the relationship from a near-match.",
)
task4_check(
    "VAL-TEXT-02", "extracted review order agrees with structured order_id",
    written["product_reviews"]["extracted_order_reference"].eq(
        written["product_reviews"]["order_id"]
    ).all(), f"rows_checked={len(written['product_reviews'])}",
    "Raw parser-obtained review text extracted before cleaning and compared with structured order_id.",
    "Every extracted order reference agrees with the structured review relationship.",
    "Inspect extraction boundaries and the structured source key; do not accept malformed near-matches.",
)
VALIDATION_PROMO_PATTERN = re.compile(r"B[1-5]SAVE-[0-9]{2}", re.IGNORECASE | re.ASCII)

def validation_extract_promo(value):
    if value is None or (isinstance(value, float) and value != value):
        return MISSING
    text = str(value)
    for match in VALIDATION_PROMO_PATTERN.finditer(text):
        start, end = match.span()
        before = text[start - 1] if start else None
        after = text[end] if end < len(text) else None
        before_ok = before is None or (before not in "_-" and unicodedata.category(before)[0] not in "LNM")
        after_ok = after is None or (after not in "_-" and unicodedata.category(after)[0] not in "LNM")
        if before_ok and after_ok:
            return match.group(0).upper()
    return MISSING

validation_promo_candidates = pd.concat([
    pd.DataFrame({
        "order_id": json_headers["orderID"],
        "promo_code": json_headers["customerNote"].map(validation_extract_promo),
    }),
    pd.DataFrame({
        "order_id": xml_headers["Order_ID"],
        "promo_code": xml_headers["Customer_Note"].map(validation_extract_promo),
    }),
], ignore_index=True).drop_duplicates()
validation_promo_values = validation_promo_candidates.groupby("order_id")["promo_code"].apply(
    lambda series: sorted(set(series) - {MISSING})
)
validation_promo_conflicts = validation_promo_values.map(len).gt(1)
validation_promo_by_id = validation_promo_values.map(
    lambda values: values[0] if len(values) == 1 else MISSING
)
expected_promo = written_text["orders"]["order_id"].map(validation_promo_by_id)
task4_check(
    "VAL-TEXT-03", "promotion extraction agrees with raw structured customer_note",
    not validation_promo_conflicts.any() and written_text["orders"]["promo_code"].eq(expected_promo).all(),
    f"rows_checked={len(expected_promo)}, literal_NaN={(expected_promo == MISSING).sum()}, independent_conflicts={int(validation_promo_conflicts.sum())}",
    "A validation-local bounded pattern extracts PROMO references from parser-obtained raw notes and independently reconciles them by order_id.",
    "Promotion codes and literal-NaN absence follow the published extraction order.",
    "Correct the raw-value extraction boundary or resolve a recorded cross-source derivation conflict.",
)

# Cleaned narrative residue, measures and multilingual preservation.
residue_tokens = (
    "<", ">", "http://", "https://", "www.", "[system]", "[catalogue]",
    "[verified_purchase]", "[source:", "[rating:", "#verified-buyer", "@store_support",
)
for text_index, (table, field) in enumerate((
    ("orders", "customer_note_clean"),
    ("products", "product_description_clean"),
    ("product_reviews", "review_body_clean"),
), start=4):
    values = written_text[table][field]
    residue = values.map(lambda value: any(token in value.lower() for token in residue_tokens))
    emoji_or_symbol = values.map(lambda value: any(
        validation_is_emoji_character(char)
        for char in value
    ))
    lower_case = values.eq(MISSING) | values.eq(values.str.lower())
    valid = not residue.any() and not emoji_or_symbol.any() and lower_case.all()
    task4_check(
        f"VAL-TEXT-{text_index:02d}", f"{table}.{field}: cleaned narrative contract",
        valid,
        f"residue={residue.sum()}, emoji_or_symbol={emoji_or_symbol.sum()}, not_lower={len(values)-lower_case.sum()}",
        "Submitted narrative scanned for tags, listed markers, URLs, emoji/symbol residue and lower-case output.",
        "The designated cleaned narrative follows the published removal and case contract.",
        "Apply the published cleaning order to the parser-obtained raw value and recheck residue.",
    )

review_body = written_text["product_reviews"]["review_body_clean"]
expected_length = review_body.map(lambda value: 0 if value == MISSING else len(value))
expected_words = review_body.map(lambda value: 0 if value == MISSING else len(value.split()))
measure_ok = (
    written["product_reviews"]["review_length_chars"].eq(expected_length).all()
    and written["product_reviews"]["review_word_count"].eq(expected_words).all()
)
task4_check(
    "VAL-TEXT-07", "review character/word measures follow sentinel behaviour",
    measure_ok, f"rows={len(review_body)}",
    "Python len and whitespace-token counts recomputed from submitted review_body_clean; literal NaN maps to zero.",
    "Review measures describe human-readable cleaned text rather than the three sentinel characters.",
    "Recompute both measures from review_body_clean with an explicit literal-NaN branch.",
)

# The multilingual and Latin-analysis contract is asserted once, by VAL-TEXT-11 below.



In [83]:
print_validation_results(
    "Narrative extraction and cleaning checks",
    ("VAL-TEXT-01", "VAL-TEXT-02", "VAL-TEXT-03", "VAL-TEXT-04", "VAL-TEXT-05", "VAL-TEXT-06", "VAL-TEXT-07"),
)


Narrative extraction and cleaning checks: 7/7 checks PASS
   check_id                                                    description                                                     observed status                                                                                                                                   evidence                                                                    resolution_or_interpretation
VAL-TEXT-01            extracted review SKU agrees with referenced product                                            rows_checked=7000   PASS                       Raw parser-obtained review text extracted before cleaning, then compared through product_id to products.product_sku.                            Every extracted SKU agrees with the structured product relationship.
VAL-TEXT-02         extracted review order agrees with structured order_id                                            rows_checked=7000   PASS                                           

In [84]:
reference_contracts = {
    "extracted_order_reference": re.compile(r"(?:HORD|CORD)[0-9]{6}"),
    "extracted_product_sku": re.compile(r"SKU-[A-Z0-9]+"),
}
for text_index, (field, pattern) in enumerate(reference_contracts.items(), start=8):
    values = written_text["product_reviews"][field]
    populated = values.ne(MISSING) & values.ne("")
    malformed = populated & ~values.str.fullmatch(pattern)
    lower_case = populated & values.ne(values.str.upper())
    task4_check(
        f"VAL-TEXT-{text_index:02d}", f"product_reviews.{field}: published format and uppercase",
        not malformed.any() and not lower_case.any(),
        f"literal_NaN={(values == MISSING).sum()}, malformed={int(malformed.sum())}, lower_case={int(lower_case.sum())}",
        f"Exact submitted {field} values checked against the published ASCII format; literal NaN is allowed when absent.",
        "Every populated extracted reference is uppercase and follows its exact published format.",
        "Re-run the bounded extractor on the raw narrative and preserve literal NaN for an absent reference.",
    )
promo_values = written_text["orders"]["promo_code"]
promo_populated = promo_values.ne(MISSING) & promo_values.ne("")
promo_malformed = promo_populated & ~promo_values.str.fullmatch(r"B[1-5]SAVE-[0-9]{2}")
task4_check(
    "VAL-TEXT-10", "orders.promo_code: published format and uppercase",
    not promo_malformed.any(),
    f"literal_NaN={(promo_values == MISSING).sum()}, malformed={int(promo_malformed.sum())}",
    "Exact submitted promo_code values checked against B1SAVE- through B5SAVE- plus two ASCII digits.",
    "Every populated promotion code is uppercase and follows the published format.",
    "Re-run promotion extraction on the raw customer note and retain literal NaN when absent.",
)
review_body = written_text["product_reviews"]["review_body_clean"]
latin_body = written_text["product_reviews"]["review_body_latin_analysis"]
expected_non_latin = review_body.map(validation_contains_non_latin_script)
observed_non_latin = written_text["product_reviews"]["contains_non_latin_script"].eq("True")
latin_contains_non_latin = latin_body.map(validation_contains_non_latin_script)
def validation_expected_latin_analysis(cleaned):
    """Re-derive the complete published Latin-analysis text.

    Written from the specification rather than by calling build_latin_analysis, so a
    defect in the submitted function cannot hide behind the check meant to detect it.
    Contract: keep Latin letters, keep combining marks only after a retained Latin
    letter, keep Unicode numerics and non-wide punctuation, drop every other letter,
    wide/fullwidth punctuation and symbol, then collapse; literal NaN when no Latin
    letter survives.
    """
    if cleaned == MISSING:
        return MISSING
    kept, latin_seen, mark_belongs_to_latin = [], False, False
    for char in unicodedata.normalize("NFC", str(cleaned)):
        category = unicodedata.category(char)
        if category.startswith("L"):
            if validation_is_latin_letter(char):
                kept.append(char); latin_seen = True; mark_belongs_to_latin = True
            else:
                kept.append(" "); mark_belongs_to_latin = False
        elif category.startswith("M"):
            kept.append(char if mark_belongs_to_latin else " ")
        elif category.startswith("N"):
            kept.append(char); mark_belongs_to_latin = False
        elif char.isspace():
            kept.append(" "); mark_belongs_to_latin = False
        elif category.startswith("P"):
            kept.append(" " if unicodedata.east_asian_width(char) in ("W", "F") else char)
            mark_belongs_to_latin = False
        else:
            kept.append(" "); mark_belongs_to_latin = False
    result = re.sub(r"\s+", " ", "".join(kept)).strip()
    return result if latin_seen and result else MISSING


def has_latin_letter(value):
    return value != MISSING and any(
        validation_is_latin_letter(char)
        for char in value
    )
expected_latin_missing = ~review_body.map(has_latin_letter)
# Keep the row-level Series so the observed result can report a real mismatch count.
latin_sentinel_match = latin_body.eq(MISSING).eq(expected_latin_missing)
non_ascii_latin = review_body.map(lambda value: any(
    ord(char) > 127 and validation_is_latin_letter(char) for char in value
) if value != MISSING else False)
latin_only_marked_non_latin = non_ascii_latin & ~expected_non_latin & observed_non_latin
# Full-text equality is the condition that actually binds. The three tests below it
# are each satisfiable by a degenerate column (for example every value set to "a"),
# because no review in this export lacks a Latin letter, so the sentinel branch never
# fires. Comparing the complete expected string closes that gap.
expected_latin_text = review_body.map(validation_expected_latin_analysis)
latin_text_mismatches = latin_body.ne(expected_latin_text)
multilingual_ok = (
    not latin_text_mismatches.any()
    and expected_non_latin.eq(observed_non_latin).all()
    and not latin_contains_non_latin.any()
    and latin_sentinel_match.all()
    and not latin_only_marked_non_latin.any()
)
task4_check(
    "VAL-TEXT-11", "review multilingual and Latin-analysis contract",
    multilingual_ok,
    f"latin_text_mismatches={int(latin_text_mismatches.sum())} of {len(latin_body)}, clean_non_latin={expected_non_latin.sum()}, indicator_true={observed_non_latin.sum()}, latin_output_with_non_latin={latin_contains_non_latin.sum()}, latin_sentinel_mismatches={int((~latin_sentinel_match).sum())}, latin_only_marked_non_latin={int(latin_only_marked_non_latin.sum())}",
    "Complete expected Latin-analysis text re-derived from the submitted review_body_clean by an independent implementation of the published contract, then compared string-for-string; Unicode letter names/categories are recomputed on both review outputs.",
    "Multilingual letters remain in review_body_clean, the indicator agrees, Latin analysis excludes non-Latin letters, and sentinel behaviour is correct.",
    "Rebuild Latin analysis from review_body_clean and classify scripts with Unicode metadata, preserving the multilingual clean field.",
)

# The checks above all read review_body_clean as their input, so a column that had its
# multilingual content stripped would agree with every statistic derived from it.
# review_clean_by_id is derived from the raw parser-obtained JSON/XML review text, so
# comparing it against the submitted column tests content, not internal consistency.
expected_clean_from_source = written_text["product_reviews"]["review_id"].map(review_clean_by_id)
clean_from_source_mismatches = written_text["product_reviews"]["review_body_clean"].ne(expected_clean_from_source)
source_non_latin = expected_clean_from_source.map(validation_contains_non_latin_script)
lost_non_latin = int((source_non_latin & ~expected_non_latin).sum())
task4_check(
    "VAL-TEXT-12", "submitted review_body_clean reproduces the cleaned raw source text",
    not clean_from_source_mismatches.any() and lost_non_latin == 0,
    f"mismatches={int(clean_from_source_mismatches.sum())} of {len(expected_clean_from_source)}, "
    f"non_latin_in_source={int(source_non_latin.sum())}, non_latin_retained={int(expected_non_latin.sum())}, lost={lost_non_latin}",
    "Raw JSON/XML review text reconciled by review_id and cleaned through the published order, then compared with the submitted column.",
    "The submitted cleaned review text is exactly what the published cleaning order produces from the raw source, and no multilingual content was dropped.",
    "Re-derive review_body_clean from the parser-obtained raw value; never edit the exported CSV directly.",
)

In [85]:
print_validation_results(
    "Text format and multilingual contract checks",
    ("VAL-TEXT-08", "VAL-TEXT-09", "VAL-TEXT-10", "VAL-TEXT-11", "VAL-TEXT-12"),
)


Text format and multilingual contract checks: 5/5 checks PASS
   check_id                                                               description                                                                                                                                                            observed status                                                                                                                                                                                                                                                  evidence                                                                                                                          resolution_or_interpretation
VAL-TEXT-08 product_reviews.extracted_order_reference: published format and uppercase                                                                                                                            literal_NaN=0, malformed=0, lower_case=0   PASS                        

In [86]:
print_validation_results(
    "6.6 Text and multilingual checks",
    ("VAL-TEXT-",),
)


6.6 Text and multilingual checks: 12/12 checks PASS
   check_id                                                               description                                                                                                                                                            observed status                                                                                                                                                                                                                                                  evidence                                                                                                                          resolution_or_interpretation
VAL-TEXT-01                       extracted review SKU agrees with referenced product                                                                                                                                                   rows_checked=7000   PASS                                  

### 6.7 Literal `NaN` reminder

Validation conditions:

- Sentinel check passes only when nullable `coupon_code` and `promo_code` values are never blank and every absent value is the literal `NaN`.
- Populated promotion codes must match the published promotion format; malformed populated values fail the check.
- This validation enforces `ASM-05` for the submitted nullable string fields.

**Assumption validated in this section:** `ASM-05` prescribed nullable string outputs use the literal `NaN` sentinel rather than an empty cell or pandas missing value.

### Literal-`NaN` sentinel validation

In [87]:
# Prescribed nullable strings must remain visible rather than becoming blank CSV cells.
for sentinel_index, field in enumerate(("coupon_code", "promo_code"), start=1):
    values = written_text["orders"][field]
    malformed = values.map(
        lambda value: value != MISSING and re.fullmatch(r"B[1-5]SAVE-[0-9]{2}", value) is None
    )
    valid = values.ne("").all() and not malformed.any()
    task4_check(
        f"VAL-SENTINEL-{sentinel_index:02d}", f"orders.{field}: literal NaN or valid published code",
        valid, f"literal_NaN={(values == MISSING).sum()}, blank={(values == '').sum()}, malformed={malformed.sum()}",
        f"Exact-text {field} values checked with keep_default_na=False and an independent full-format rule.",
        "Every absent code is the three-character sentinel NaN; every populated code follows the public format.",
        "Replace blank/pandas missing values with literal NaN and correct malformed populated codes.",
    )



In [88]:
print_validation_results(
    "6.7 Literal NaN reminder",
    ("VAL-SENTINEL-",),
)


6.7 Literal NaN reminder: 2/2 checks PASS
       check_id                                             description                               observed status                                                                                              evidence                                                                           resolution_or_interpretation
VAL-SENTINEL-01 orders.coupon_code: literal NaN or valid published code literal_NaN=3133, blank=0, malformed=0   PASS Exact-text coupon_code values checked with keep_default_na=False and an independent full-format rule. Every absent code is the three-character sentinel NaN; every populated code follows the public format.
VAL-SENTINEL-02  orders.promo_code: literal NaN or valid published code literal_NaN=3133, blank=0, malformed=0   PASS  Exact-text promo_code values checked with keep_default_na=False and an independent full-format rule. Every absent code is the three-character sentinel NaN; every populated code follows the pub

### Validation-register summary

The final code cell displays the complete in-memory validation register and keeps any failures visible with their proposed resolution. No separate validation-register CSV is generated.

In [89]:
task4_validation = pd.DataFrame(task4_checks)
print(task4_validation.to_string(index=False))
failed_task4_checks = task4_validation[task4_validation.status != "PASS"]
passed_task4_checks = len(task4_validation) - len(failed_task4_checks)
print(f"Task 4 validation: {passed_task4_checks}/{len(task4_validation)} checks PASS")
# Keep failures visible with a proposed resolution; never hide unresolved values
# through source precedence.
if not failed_task4_checks.empty:
    print(f"\n{len(failed_task4_checks)} Task 4 check(s) FAIL - investigate before submission:")
    print(failed_task4_checks.to_string(index=False))

       check_id                                                                          description                                                                                                                                                                                                                                        observed status                                                                                                                                                                                                                                                  evidence                                                                                                                          resolution_or_interpretation
    VAL-FILE-01                                                    exactly six required output files ['Group005_customers_standardised.csv', 'Group005_deliveries_standardised.csv', 'Group005_order_items_standardised.csv', 'Group005_orders_standardised.

## 7. Export the six CSV files

The six CSV files are written in the existing `T2-O1` cell before validation so the validation register checks the serialized deliverables rather than only in-memory DataFrames.

---
#### Deliverables written by this notebook

Enumerated from disk rather than listed by hand, so the manifest cannot drift.

In [90]:
deliverables = sorted(glob.glob(os.path.join(PROFILE_DIR, f"{GROUP_ID}_T1_*.csv"))) + [MAPPING_PATH]
artefact_manifest = pd.DataFrame([
    {"artefact": path,
     "rows": len(pd.read_csv(path, keep_default_na=False)),
     "bytes": os.path.getsize(path)}
    for path in deliverables
])
print(f"{len(artefact_manifest)} artefacts written "
      f"({len(deliverables) - 1} profiling CSVs + the source-to-target mapping)")
artefact_manifest

14 artefacts written (13 profiling CSVs + the source-to-target mapping)


,artefact,rows,bytes
0,profiling/Group005_T1_assumptions_register.csv,16,5586
1,profiling/Group005_T1_categorical_domain_profile.csv,41,2545
2,profiling/Group005_T1_duplicate_and_overlap_profile.csv,6,613
3,profiling/Group005_T1_embedded_reference_profile.csv,3,455
4,profiling/Group005_T1_field_level_conflicts.csv,0,26
5,profiling/Group005_T1_format_conventions.csv,8,2201
6,profiling/Group005_T1_mapping_source_format_summary.csv,7,198
7,profiling/Group005_T1_narrative_field_profile.csv,5,599
8,profiling/Group005_T1_order_arithmetic_checks.csv,2,447
9,profiling/Group005_T1_raw_frame_profile.csv,11,818


---
#### Task 1 summary

| Question | Answer, with evidence |
|---|---|
| Are both sources parsed structurally? | Yes — `json.load` + `pandas.json_normalize`, and `BeautifulSoup(..., "lxml-xml")`, following the Week 2/3 applied sessions. No regex reconstructs document structure (`T1-P1`, `T1-P2`). |
| Is everything in pandas? | Yes — eleven raw DataFrames are built immediately after parsing, and every profiling question from `T1-P3` on is answered with pandas (`T1-P2`). |
| What are the major objects and repeated elements? | 3 JSON collections, 4 XML collections; `shoppingCart[*]` / `Shopping_Cart/Item` is the only one-to-many child of an order (`T1-P2`). |
| What is each collection's grain? | One record per order, order item, customer, delivery, product and review — but **no shared transactional key is unique within its own export**; only the single-source `customers` and `products` collections are already unique (`T1-P3`). |
| Which target fields come from where? | Customers are JSON-only, products XML-only, orders / order items / deliveries / reviews are in both (`T1-P2`, `T1-P5`, `T1-M1`). |
| How do the formats differ? | ISO vs day-first dates, native vs `Y`/`N` booleans, float vs `"AUD 1,234.56"`, integer vs `"15%"`, empty string vs empty element (`T1-P4`). |
| Does the published arithmetic hold? | All six steps reproduce the published values in both exports — but only with Python's rounding and only on de-duplicated order lines (`T1-P4`, `ASM-14`). |
| Is there duplication and overlap? | Both kinds, in every shared table; **zero field-level conflicts after normalisation**, so no source precedence rule is needed (`T1-P6`). |
| Do the relationships hold? | All eight required foreign keys resolve against the union of the two sources with zero orphans (`T1-P7`). |
| What must be assumed before transforming? | Every assumption is registered with its supporting evidence and the consequence if it is wrong (`T1-P9`, `ASM-01` onwards). |
| Is the mapping complete? | 111/111 required target fields, all six blank columns filled, asserted against the dictionary and against the profiled paths (`T1-M1`). |

**Task 3 evidence:** 18/18 supplied public cases and 61/61 student-designed cases covering
matched, unmatched, missing, multilingual and near-match inputs are demonstrated in section
3.2, together with four counts reproduced independently from the section 1 profiling.

**Not in scope here (Tasks 5 and 6):** the EDA notebook and the findings/ML sections of the
report.

## 8. Final reproducibility record

#### Task 2 summary

* Exactly six CSV files are written with dictionary field order and no helper columns.
* Shared tables are reconciled after normalisation with zero silent conflict choices.
* Order-item, order, included-GST and discounted-total arithmetic is re-derived in sequence.
* Every primary/foreign key and the order/item/review one-to-many relationships remain valid.
* Literal `NaN` is retained only for prescribed string results; numeric/boolean fields never
  receive that sentinel.